In [167]:
from zoneinfo import ZoneInfo
import datetime
import json
from typing import List

thailand_tz = ZoneInfo("Asia/Bangkok")


class PostLoanOrder:
    def __init__(self, order_dict):
        self.order_create_time = order_dict["订单创建时间"]
        self.platform = order_dict["平台"]
        self.app_name = order_dict["包名"]
        self.borrow_id = order_dict["订单号"]
        self.contract_id = order_dict["合同id"]
        self.bank = order_dict["银行"]
        self.bank_card_num = order_dict["银行卡号"]
        self.phone = order_dict["手机号"]
        self.id_number = order_dict["身份证号"]
        self.user_id = order_dict["用户id"]
        self.contract_status = order_dict["合同状态"]
        self.order_type = order_dict["订单类型"]
        self.borrow_amt = float(order_dict["合同额"])
        self.in_hand_amt = float(order_dict["到手额"])
        self.order_flag = order_dict["订单标识"]
        self.is_force = order_dict["是否强制放款"]
        self.product_id = order_dict["产品id"]
        self.root_borrow_id = order_dict["根订单号"]
        self.repay_amt = (
            float(order_dict["总还款"]) if order_dict["总还款"] is not None else None
        )
        self.repay_cnt = order_dict["还款次数"]
        self.extend_cnt = order_dict["展期次数"]
        self.repay_time = order_dict["结清时间"]
        self.loan_time = order_dict["放款时间"]
        self.expect_repay_time = order_dict["应还日期"]
        self.order_id = order_dict["订单id"]
        self.borrow_count = int(order_dict["借款次数"])
        self.overdue_days = (
            int(order_dict["逾期天数"]) if order_dict["逾期天数"] is not None else None
        )

    @classmethod
    def load_from_dict(cls, order_dict):
        instance = cls.__new__(cls)
        for key in order_dict:
            setattr(instance, key, order_dict[key])
        return instance

    @property
    def product_type(self):
        if self.product_id == "000001":
            return "合规产品"
        elif self.product_id == "000002":
            return "前置740"
        else:
            return "未知产品"

    @property
    def is_regret(self):
        if self.order_flag in ("人工悔贷", "自动悔贷"):
            return 1
        else:
            return 0

    @property
    def is_onway(self):
        if self.contract_status == "还款中":
            return 1
        else:
            return 0

    @property
    def is_extend(self):
        if self.contract_status == "展期结清":
            return 1
        else:
            return 0

    @property
    def is_extended_order(self):
        if self.borrow_id != self.root_borrow_id:
            return 1
        else:
            return 0

    @property
    def is_root_order(self):
        if self.root_borrow_id == self.borrow_id:
            return 1
        else:
            return 0

    @property
    def is_early_repay(self):
        if self.repay_time is not None and self.overdue_days < 0:
            return 1
        else:
            return 0

    @property
    def is_late_repay(self):
        if self.repay_time is not None and self.overdue_days > 0:
            return 1
        else:
            return 0

    @property
    def loan_dt(self):
        return datetime.datetime.fromtimestamp(self.loan_time, tz=thailand_tz)

    @property
    def is_repay(self):
        if self.repay_time is not None:
            return 1
        else:
            return 0

    @property
    def expect_repay_dt(self):
        return datetime.datetime.fromtimestamp(self.expect_repay_time, tz=thailand_tz)

    @property
    def repay_dt(self):
        return datetime.datetime.fromtimestamp(self.repay_time, tz=thailand_tz)

    @property
    def create_dt(self):
        return datetime.datetime.fromtimestamp(self.order_create_time, tz=thailand_tz)

    @property
    def is_ontime_repay(self):
        if self.repay_time is not None and self.overdue_days <= 0:
            return 1
        else:
            return 0

    @property
    def is_xm(self):
        if self.platform is None:
            return 1
        else:
            return 0

    @classmethod
    def from_lst_to_jsn(cls, post_loan_order_lst):
        return [order.__dict__ for order in post_loan_order_lst]


In [168]:
class VerifyBehavior:
    def __init__(self, data_dict):
        self.onway_cnt = 0
        self.platform = data_dict["平台"]
        self.app_name = data_dict["包名"]
        self.user_id = str(data_dict["用户id"])
        self.phone = data_dict["手机号"]
        self.idCard_no = data_dict["身份证号"]
        self.bank_no = data_dict["银行卡号"]
        self.register_time = int(data_dict["注册时间"])
        self.channel_name = data_dict["渠道"]
        self.sn = data_dict["流水号"]
        self.verify_time = int(data_dict["机审时间"])
        self.risk_level = data_dict["风险等级"]
        self.risk_type = data_dict["风险类型"]
        self.human_verify_status = data_dict["人审状态"]
        self.request_data = data_dict["请求"]
        self.response_data = data_dict["返回"]
        self.confirm_cnt = data_dict["领取数"]
        self.is_dk = data_dict["is_dk"]

    @classmethod
    def load_from_dict(cls, order_dict):
        instance = cls.__new__(cls)
        for key in order_dict:
            setattr(instance, key, order_dict[key])
        return instance

    # 返回变量字典
    @property
    def result_var(self):
        try:
            resp = json.loads(self.response_data or "{}")
            if self.is_dk == 0:
                # result.resultVariable
                ctu_result = resp.get("result", {})
            else:
                # ctuEntireResult.resultVariable
                ctu_result = resp.get("ctuEntireResult", {})

            return ctu_result.get("resultVariable", {})
        except Exception as e:
            return {}

    # 当前进件时的在贷数量
    def get_onway_cnt(self, post_loan_order_lst: List[PostLoanOrder]):
        count = 0
        for post_loan_order in post_loan_order_lst:
            if self.user_id != post_loan_order.user_id:
                continue
            if post_loan_order.loan_time < self.verify_time and (
                post_loan_order.repay_time is None
                or post_loan_order.repay_time > self.verify_time
            ):
                count += 1
        self.onway_cnt = count
        return count

    # 请求类型 0 资质风控 1预约风控
    @property
    def call_risk_type(self):
        try:
            req = json.loads(self.request_data or "{}")
            return req.get("call_risk_type", 0)
        except Exception as e:
            pass
        return 0

    # 是否首贷
    @property
    def is_first_borrow(self):
        try:
            req = json.loads(self.request_data or "{}")
            return int(req.get("borrow_count", 0)) == 0
        except Exception as e:
            pass
        return False

    # 最大可借应用数-顶象返回值中取
    @property
    def max_select_apply(self):
        if self.is_dk == 0:
            return 1
        else:
            return int(self.result_var.get("appliacation_limit", 1))

    # 最大实际可借应用数 只允许 通手机号 或者 同用户id
    def get_max_actual_apply_cnt(self, post_loan_order_lst):
        if not hasattr(self, "onway_cnt"):
            self.get_onway_cnt(post_loan_order_lst)

        if self.return_apply_cnt < self.onway_cnt:
            return 0
        else:
            visual_apply_cnt = self.return_apply_cnt - self.onway_cnt
            return min(self.max_select_apply, visual_apply_cnt)

    # 返回总应用数 顶象返回额度不为空的应用数
    @property
    def return_apply_cnt(self):
        if self.is_dk == 0:
            return 1
        else:
            if "appliacation_limit" not in self.result_var:
                return 1
            else:
                return sum(
                    1
                    for key in self.result_var
                    if key.startswith("credit_amt_")
                    and self.result_var[key] is not None
                    and self.result_var[key] != ""
                )

    # 转成字典
    @classmethod
    def from_lst_to_jsn(cls, verify_behavior_lst):
        return [data.__dict__ for data in verify_behavior_lst]

In [169]:
veirfy_data_lst = [
    {
        "app_name": "easyloan_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 2,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1757045044,
        "request_data": '{"Settlement_frequency_2":"0","Settlement_frequency_1":"0","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"49.237.177.19","borrowing_loan_num":"0","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1757045044","diffid_face_num":"0","ios_manufacturer":"","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"Mozilla","app_id":"118","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"tree","ios_system_version":"","r81_60p70p81_b":"0","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"0","emergency_contact_name":"สมับติ","borrow_rank":"0","geohash_add_time":"1757045228000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"0","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148 Safari Line/15.13.0","email":"","user_due_amount_sum":"0","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0","call_risk_type":"0","last_financial_product_id_1":"-999","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"-999","last_financial_product_id_3":"-999","last_financial_product_id_4":"-999","h5_languages":"[\\"th-TH\\"]","app_name":"happy cashing_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"0","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jd7swq0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"0","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"0","h5_browserName":"Safari","last_7d_self_same_face_id_cnt":"0","personal_city":"792","client_ip":"","h5_platform":"iPhone","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"None","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0,\\"app_9999d_fi_效率_cnt\\":0,\\"bj_ios_model_prob\\":0.271092952988736,\\"bj_ios_model_score\\":619.7816890864531,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"is_available\\":1,\\"job_monthly_income\\":0,\\"pure_new_tag\\":0}","user_contract_amount_sum":"0","first_borrow_whitelist_in":"1","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"tmk_th_com_03","h5_screen":"{\\"h5_available_height\\":932,\\"h5_available_left\\":0,\\"h5_available_top\\":0,\\"h5_available_width\\":430,\\"h5_colorDepth\\":24,\\"h5_devicePixelRatio\\":3,\\"h5_height\\":932,\\"h5_innerHeight\\":745,\\"h5_innerWidth\\":430,\\"h5_keepAwake\\":true,\\"h5_maxTouchPoints\\":5,\\"h5_orientation_angle\\":0,\\"h5_orientation_type\\":\\"portrait-primary\\",\\"h5_pixelDepth\\":24,\\"h5_width\\":430}","loan_id":"0","ios_charging_status":"","if_id_number_modified":"0","whitelist_type":"1","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"0","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"SVD20250905110714ynhlt71W","age":"30","last_credit_amt":"0","h5_vendor":"Apple Computer, Inc.","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1757045234207","ph_diffzalo_num":"3","h5_hardwareConcurrency":"4","last_settlement_order_difference":"-1","last_7d_all_same_face_phone_cnt":"0","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"5","lng":"100.79365312691327","another_contact_name":"ทัศนาทิพย์","ip":"","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"c9520a45f9bb4d59b3865919b6538392","other_contact_phone":"","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"0","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"30","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"0","borrow_count_9":"0","ios_rooted":"","user_max_appliacation_limit":"0","if_dk_event":"1","face_similarity":"90.8","borrow_count_1":"0","const_id":"","liveness_livenessScore":" \'100\'","last_contract_amount_10":"0","diffphone_face_num":"0","borrow_create_time":"1733677501001","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"0","last_contract_amount_1":"0","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"Gecko","first_credit_amount_9":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"0","last_7d_self_same_face_apply_cnt":"0","first_credit_amount_3":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"0","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"0","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"0","platform_check_baiduai_black":"0","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888168740953315","if_live_detection":"1","automatic_face_pass":"1","same_calendarid_dif_phone":"0","ios_app_version":"","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"1","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"0","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"","h5_language":"th-TH","h5_productSub":"20030107","h5_appName":"Netscape","personal_education":"","ios_app_package":"","ios_battery_percentage":"","borrowing_loan_id":"","emergency_is_others":"0"}',
        "response_data": '{"uuid":"0e42ee5d-b44c-4db7-85a0-53059d58209e","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197498446007,"leftValue":"异常校验"},{"id":382197500543014,"leftValue":"异常校验"},{"id":411574860709947,"leftValue":"异常校验"},{"id":382197502640796,"leftValue":"异常校验"},{"id":382197500543452,"leftValue":"泰国_IOS_首贷_新包老客_活体检测分_Meta_20240621"},{"id":382197500543485,"leftValue":"泰国_IOS_首贷_新包老客_人脸相似度_Meta_20240621"},{"id":382197500543495,"leftValue":"泰国_IOS_首贷_新包老客_其他平台当前逾期数量_Meta_20240621"},{"id":382197500543508,"leftValue":"泰国_IOS_首贷_新包老客_当前逾期人脸_Meta_20240620"},{"id":382197500543521,"leftValue":"泰国_IOS_首贷_新包老客_年龄准入_Meta_20240620"},{"id":382197500543538,"leftValue":"泰国_IOS_首贷_新包老客_外部黑名单_Meta_20240621"},{"id":382197500543551,"leftValue":"泰国_IOS_首贷_新包老客_云服务黑名单_Meta_20240621"},{"id":389414262866011,"leftValue":"泰国_IOS_首贷_新包老客_人脸黑名单_Meta_20241111"},{"id":393581725417533,"leftValue":"ocr无法识别年龄"},{"id":436544793935884,"leftValue":"电销包新包老客拒绝"},{"id":382197498446159,"leftValue":"泰国_IOS_首贷_新包老客_非正常安装程序_Meta_20240620"},{"id":382197498446172,"leftValue":"泰国_IOS_首贷_新包老客_越狱_Meta_20240620"},{"id":405023739543570,"leftValue":"电销欺诈"},{"id":432743791919143,"leftValue":"中介窝点风险"},{"id":433108941733894,"leftValue":"近7天同geohash不同手机号数量"},{"id":434209938145287,"leftValue":"泰国_IOS_首贷_纯新户_APP相似度"},{"id":434209938145299,"leftValue":"泰国_IOS_首贷_纯新户_通讯录相似度"},{"id":434209938145311,"leftValue":"泰国_IOS_首贷_纯新户_人脸聚集风险"},{"id":434209938145323,"leftValue":"泰国_IOS_首贷_纯新户_同银行卡不同姓名"},{"id":434209938145335,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同姓名"},{"id":382197496348689,"leftValue":"泰国_IOS_首贷_新包老客_在途订单_Meta_20240620"},{"id":418788323295235,"leftValue":"泰国_IOS_首贷_新包老客_在途金额"},{"id":432349613326340,"leftValue":"泰国_IOS_首贷_新包老客_在途订单"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25090511Vd0uFUYR","extraInfo":{"_observe_mode":[false],"_cost_time":94,"ruleIdCodeNamePair":"[{\\"code\\":\\"rule1\\",\\"id\\":405047798071317,\\"name\\":\\"白名单_投放返回\\"}]","riskId":"0e42ee5d-b44c-4db7-85a0-53059d58209e,1757045450648,bab0e20638be9d1173d490934d140aae"},"hitPolicies":[{"id":382197500543324,"name":"泰国_IOS贷超_前处理_公共变量_赋值"}],"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":"","th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":"","his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":9999,"th_app_last_1d_fin_update_cnt_b2407_meta_20240715":"","th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":"","th_app_last_7d_food_update_cnt_b2407_meta_20240715":"","bj_ios_model_score":619.7816890864531,"is_error":"0","credit_amt_1":"3500","th_beh_reject_pctb2407_meta_20240717":"","credit_amt_2":"2000","credit_amt_3":"","Expected_loan_time_1":"1757045234207","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":0,"Expected_loan_time_2":"1757045234207","Expected_loan_time_3":"","th_app_last_21d_social_update_cnt_meta_20240715":"","lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":0.0,"coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"","th_app_last_1d_biz_update_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_meta_20240619":0,"financial_product_id_1":"000002","th_model_bscore_meta_20240719":"","financial_product_id_3":"","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":0,"th_app_last_21d_highrisk_update_cnt_meta_20240715":"","th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":0.0,"th_app_loan_earlist_install_days_b2407_meta_20240715":"","risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":"","th_app_last_3m_highrisk_install_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_a2405_meta_20240619":0,"job_monthly_income":"None","credit_amt":"3000","is_advertising":"1","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":"","th_app_loan_latest_install_days_a2405_meta_20240619":9999,"th_app_last_7d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_per_repay_b2407_meta_20240717":"","appliacation_limit":"2","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":-1,"th_model_bversion_meta_20240719":"","th_app_last_3d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_pct_meta_20240717":"","th_model_score_2_meta_20240619":576.22,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":0.0,"th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"","overdue_detail":"0","model_score":575.5600000000001,"th_app_last_30d_pe_update_cnt_meta_20240715":""}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "SVD20250905110714ynhlt71W",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
        "verify_time": 1757045244,
    },
    {
        "app_name": "easyloan_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 2,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1757045044,
        "request_data": '{"Settlement_frequency_2":"1","Settlement_frequency_1":"0","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"27.55.83.113","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1757045044","diffid_face_num":"0","ios_manufacturer":"","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"Mozilla","app_id":"118","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"tree","ios_system_version":"","r81_60p70p81_b":"0.50","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"1","geohash_add_time":"1758095422000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"1","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148 Safari Line/15.13.0","email":"","user_due_amount_sum":"3500.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.7045486111","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"-999","last_financial_product_id_4":"-999","h5_languages":"[\\"th-TH\\"]","app_name":"happy cashing_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"12.3258101852","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jd7x700","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"2000","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"3500","h5_browserName":"Safari","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"","h5_platform":"iPhone","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"3500.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0,\\"app_9999d_fi_效率_cnt\\":0,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":2000,\\"pl_loan_5o_user_avg_wait_hours\\":0,\\"pl_loan_7o_phone_max_amt\\":3500,\\"pl_loan_9999d_phone_max_amt\\":3500,\\"pl_loan_9999d_user_max_amt\\":3500,\\"pl_repay_1o_phone_ontime_latest_hours\\":148.50363,\\"pl_repay_2o_phone_ontime_latest_hours\\":148.50363,\\"pl_repay_30d_phone_ontime_latest_hours\\":148.50363,\\"pl_repay_30d_user_ontime_latest_hours\\":148.50363,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.36364,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_ontime_earliest_hours\\":148.50363,\\"pl_repay_60d_phone_ontime_latest_hours\\":148.50363,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":148.50363,\\"pl_repay_7o_phone_early_amt_all_pct\\":0,\\"pl_repay_90d_phone_early_cnt\\":0,\\"pl_repay_90d_user_early_cnt\\":0,\\"pl_repay_9999d_phone_cnt\\":2,\\"pl_repay_9999d_phone_early_cnt\\":0,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.4133698915358983,\\"th_ios_bcard_2508_score\\":573.7796128358646}","first_borrow_whitelist_in":"1","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":932,\\"h5_available_left\\":0,\\"h5_available_top\\":0,\\"h5_available_width\\":430,\\"h5_colorDepth\\":24,\\"h5_devicePixelRatio\\":3,\\"h5_height\\":932,\\"h5_innerHeight\\":745,\\"h5_innerWidth\\":430,\\"h5_keepAwake\\":true,\\"h5_maxTouchPoints\\":5,\\"h5_orientation_angle\\":0,\\"h5_orientation_type\\":\\"portrait-primary\\",\\"h5_pixelDepth\\":24,\\"h5_width\\":430}","loan_id":"0","ios_charging_status":"","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"SVD20250917065630W9gFO8Pm","age":"30","last_credit_amt":"3500","h5_vendor":"Apple Computer, Inc.","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1758110191070","ph_diffzalo_num":"3","h5_hardwareConcurrency":"4","last_settlement_order_difference":"0.70","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"5","lng":"100.7935104747132","another_contact_name":"ทัศนาทิพย์","ip":"","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"c9520a45f9bb4d59b3865919b6538392","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"1","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"","user_max_appliacation_limit":"2","if_dk_event":"1","face_similarity":"90.8","borrow_count_1":"0","const_id":"","liveness_livenessScore":" \'100\'","last_contract_amount_10":"0","diffphone_face_num":"0","borrow_create_time":"1733674021008","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"2000.00","last_contract_amount_1":"3500.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"Gecko","first_credit_amount_9":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"2000.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3500.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"1","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888604799810743","if_live_detection":"1","automatic_face_pass":"1","same_calendarid_dif_phone":"0","ios_app_version":"","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"1","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"","h5_language":"th-TH","h5_productSub":"20030107","h5_appName":"Netscape","personal_education":"","ios_app_package":"","ios_battery_percentage":"","borrowing_loan_id":"1","emergency_is_others":"0"}',
        "response_data": '{"uuid":"d3102726-7c2a-40e8-a631-230b48154a37","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25091706672kwX2t","extraInfo":{"_cost_time":163,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"d3102726-7c2a-40e8-a631-230b48154a37,1758110127003,d8e158bb99d0c873837c2728b5ecd81b"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"5","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":4200.0,"th_beh_reject_pctb2407_meta_20240717":"0.8889","credit_amt_2":2700.0,"credit_amt_3":"500","Expected_loan_time_1":"1758110191070","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1758110191070","Expected_loan_time_3":"1758110191070","th_app_last_21d_social_update_cnt_meta_20240715":0,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.5000","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.5000","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":0,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":0,"th_app_last_3m_highrisk_install_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","credit_amt":"3500","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.5000","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":0}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "SVD20250917065630W9gFO8Pm",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
        "verify_time": 1758110195,
    },
    {
        "app_name": "easyloan_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1757045044,
        "request_data": '{"Settlement_frequency_2":"1","Settlement_frequency_1":"1","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"27.55.95.250","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1757045044","diffid_face_num":"0","ios_manufacturer":"","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"Mozilla","app_id":"118","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"tree","ios_system_version":"","r81_60p70p81_b":"0.50","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"2","geohash_add_time":"1758134718000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"2","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148 Safari Line/15.13.0","email":"","user_due_amount_sum":"3200.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.1531134259","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[\\"th-TH\\"]","app_name":"happy cashing_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"0.2839699074","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"-6","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jemmk70","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"2700","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"500","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"3500","h5_browserName":"Safari","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"","h5_platform":"iPhone","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"3200.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0,\\"app_9999d_fi_效率_cnt\\":0,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":2700,\\"pl_loan_5o_user_avg_wait_hours\\":149.25194,\\"pl_loan_7o_phone_max_amt\\":3500,\\"pl_loan_9999d_phone_max_amt\\":3500,\\"pl_loan_9999d_user_max_amt\\":3500,\\"pl_repay_30d_phone_ontime_latest_hours\\":155.31938,\\"pl_repay_30d_user_ontime_latest_hours\\":155.31938,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.63218,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":156.77494,\\"pl_repay_60d_phone_ontime_earliest_hours\\":156.77494,\\"pl_repay_60d_phone_ontime_latest_hours\\":155.31938,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":156.77494,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.63729,\\"pl_repay_90d_phone_early_cnt\\":1,\\"pl_repay_90d_user_early_cnt\\":1,\\"pl_repay_9999d_phone_cnt\\":5,\\"pl_repay_9999d_phone_early_cnt\\":1,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.3579008698204822,\\"th_ios_bcard_2508_score\\":580.5439765839996}","first_borrow_whitelist_in":"1","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":932,\\"h5_available_left\\":0,\\"h5_available_top\\":0,\\"h5_available_width\\":430,\\"h5_colorDepth\\":24,\\"h5_devicePixelRatio\\":3,\\"h5_height\\":932,\\"h5_innerHeight\\":745,\\"h5_innerWidth\\":430,\\"h5_keepAwake\\":true,\\"h5_maxTouchPoints\\":5,\\"h5_orientation_angle\\":0,\\"h5_orientation_type\\":\\"portrait-primary\\",\\"h5_pixelDepth\\":24,\\"h5_width\\":430}","loan_id":"0","ios_charging_status":"","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"SVD20250918014528DtP1MGJ3","age":"30","last_credit_amt":"500","h5_vendor":"Apple Computer, Inc.","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1758134728456","ph_diffzalo_num":"3","h5_hardwareConcurrency":"4","last_settlement_order_difference":"0.15","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"5","lng":"100.79597496553777","another_contact_name":"ทัศนาทิพย์","ip":"","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"c9520a45f9bb4d59b3865919b6538392","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"1","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"","user_max_appliacation_limit":"2","if_dk_event":"1","face_similarity":"90.8","borrow_count_1":"1","const_id":"","liveness_livenessScore":" \'100\'","last_contract_amount_10":"0","diffphone_face_num":"0","borrow_create_time":"1733674080001","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"2700.00","last_contract_amount_1":"3500.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"Gecko","first_credit_amount_9":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"2000.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3500.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"500.00","borrow_count":"2","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.893772281564456","if_live_detection":"1","automatic_face_pass":"1","same_calendarid_dif_phone":"0","ios_app_version":"","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"1","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"","h5_language":"th-TH","h5_productSub":"20030107","h5_appName":"Netscape","personal_education":"","ios_app_package":"","ios_battery_percentage":"","borrowing_loan_id":"2,3","emergency_is_others":"0"}',
        "response_data": '{"uuid":"ae301004-2eb3-4a4e-b9e5-61a5ce4b9565","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz250918018QEHXt2d","extraInfo":{"_cost_time":192,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"ae301004-2eb3-4a4e-b9e5-61a5ce4b9565,1758134663582,b4d438ccf710b439de7341c27dcadc4c"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"8","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":"3500","th_beh_reject_pctb2407_meta_20240717":"0.8000","credit_amt_2":"2700","credit_amt_3":"500","Expected_loan_time_1":"1758134728456","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1758134728456","Expected_loan_time_3":"1758134728456","th_app_last_21d_social_update_cnt_meta_20240715":0,"lastoverdue_day":"-6","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.5000","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.5000","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":0,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":0,"th_app_last_3m_highrisk_install_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","credit_amt":"3500","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.5000","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":0}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "SVD20250918014528DtP1MGJ3",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
        "verify_time": 1758134732,
    },
    {
        "app_name": "easyloan_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 2,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1757045044,
        "request_data": '{"Settlement_frequency_2":"2","Settlement_frequency_1":"1","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"1","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"49.237.43.30","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1757045044","diffid_face_num":"0","ios_manufacturer":"","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"Mozilla","app_id":"118","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"tree","ios_system_version":"","r81_60p70p81_b":"0.80","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"3","geohash_add_time":"1758622794000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148 Safari Line/15.13.0","email":"","user_due_amount_sum":"3500.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.3116666667","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[\\"th-TH\\"]","app_name":"happy cashing_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"5.7127314815","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"-6","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jde82s0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"2700","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"500","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"3500","h5_browserName":"Safari","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"","h5_platform":"iPhone","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"3500.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0,\\"app_9999d_fi_效率_cnt\\":0,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":3500,\\"pl_loan_5o_user_avg_wait_hours\\":152.60556,\\"pl_loan_7o_phone_max_amt\\":3500,\\"pl_loan_9999d_phone_max_amt\\":3500,\\"pl_loan_9999d_user_max_amt\\":3500,\\"pl_repay_2o_phone_ontime_latest_hours\\":7.48041,\\"pl_repay_30d_phone_ontime_latest_hours\\":7.48041,\\"pl_repay_30d_user_ontime_latest_hours\\":7.48041,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.5977,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":293.88097,\\"pl_repay_60d_phone_ontime_earliest_hours\\":293.88097,\\"pl_repay_60d_phone_ontime_latest_hours\\":7.48041,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":293.88097,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.4524,\\"pl_repay_90d_phone_early_cnt\\":1,\\"pl_repay_90d_user_early_cnt\\":1,\\"pl_repay_9999d_phone_cnt\\":7,\\"pl_repay_9999d_phone_early_cnt\\":1,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.3921364760013703,\\"th_ios_bcard_2508_score\\":576.3270887790387}","first_borrow_whitelist_in":"1","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":932,\\"h5_available_left\\":0,\\"h5_available_top\\":0,\\"h5_available_width\\":430,\\"h5_colorDepth\\":24,\\"h5_devicePixelRatio\\":3,\\"h5_height\\":932,\\"h5_innerHeight\\":745,\\"h5_innerWidth\\":430,\\"h5_keepAwake\\":true,\\"h5_maxTouchPoints\\":5,\\"h5_orientation_angle\\":0,\\"h5_orientation_type\\":\\"portrait-primary\\",\\"h5_pixelDepth\\":24,\\"h5_width\\":430}","loan_id":"0","ios_charging_status":"","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"SVD20250923065149T1k4l62q","age":"30","last_credit_amt":"3500","h5_vendor":"Apple Computer, Inc.","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1758628309530","ph_diffzalo_num":"3","h5_hardwareConcurrency":"4","last_settlement_order_difference":"0.31","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"5","lng":"100.79341184396765","another_contact_name":"ทัศนาทิพย์","ip":"","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"c9520a45f9bb4d59b3865919b6538392","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"2","borrow_count_3":"1","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"90.8","borrow_count_1":"1","const_id":"","liveness_livenessScore":" \'100\'","last_contract_amount_10":"0","diffphone_face_num":"0","borrow_create_time":"1733670181008","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"2700.00","last_contract_amount_1":"3500.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"Gecko","first_credit_amount_9":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"2000.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3500.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"500.00","borrow_count":"4","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888798944610784","if_live_detection":"1","automatic_face_pass":"1","same_calendarid_dif_phone":"0","ios_app_version":"","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"1","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"","h5_language":"th-TH","h5_productSub":"20030107","h5_appName":"Netscape","personal_education":"","ios_app_package":"","ios_battery_percentage":"","borrowing_loan_id":"1","emergency_is_others":"0"}',
        "response_data": '{"uuid":"7edeb55b-5c2a-43e5-92d8-c95f8a208042","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz250923063sR2v7Wj","extraInfo":{"_cost_time":147,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"7edeb55b-5c2a-43e5-92d8-c95f8a208042,1758628529747,da2cb7ea994029d3f55d8c534b3745da"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"11","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":"3500","th_beh_reject_pctb2407_meta_20240717":"0.7273","credit_amt_2":"2700","credit_amt_3":"500","Expected_loan_time_1":"1758628309530","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1758628309530","Expected_loan_time_3":"1758628309530","th_app_last_21d_social_update_cnt_meta_20240715":0,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6667","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.8000","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":0,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":0,"th_app_last_3m_highrisk_install_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","credit_amt":"3500","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.8000","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":0}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "SVD20250923065149T1k4l62q",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
        "verify_time": 1758628313,
    },
    {
        "app_name": "easyloan_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1757045044,
        "request_data": '{"Settlement_frequency_2":"2","Settlement_frequency_1":"2","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"1","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"27.55.78.68","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1757045044","diffid_face_num":"0","ios_manufacturer":"","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"Mozilla","app_id":"118","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"tree","ios_system_version":"","r81_60p70p81_b":"0.71","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"4","geohash_add_time":"1758704449000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148 Safari Line/15.13.0","email":"","user_due_amount_sum":"3200.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.2609143519","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[\\"th-TH\\"]","app_name":"happy cashing_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"0.8813425926","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jde3qh0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"2700","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"500","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"3500","h5_browserName":"Safari","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"","h5_platform":"iPhone","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"3200.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0,\\"app_9999d_fi_效率_cnt\\":0,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":2700,\\"pl_loan_5o_user_avg_wait_hours\\":8.08972,\\"pl_loan_7o_phone_max_amt\\":3500,\\"pl_loan_9999d_phone_max_amt\\":3500,\\"pl_loan_9999d_user_max_amt\\":3500,\\"pl_repay_30d_phone_ontime_latest_hours\\":6.26211,\\"pl_repay_30d_user_ontime_latest_hours\\":6.26211,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.65957,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":315.03377,\\"pl_repay_60d_phone_ontime_earliest_hours\\":315.03377,\\"pl_repay_60d_phone_ontime_latest_hours\\":6.26211,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":315.03377,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.34343,\\"pl_repay_90d_phone_early_cnt\\":1,\\"pl_repay_90d_user_early_cnt\\":1,\\"pl_repay_9999d_phone_cnt\\":8,\\"pl_repay_9999d_phone_early_cnt\\":1,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.3177515519693301,\\"th_ios_bcard_2508_score\\":585.7272174166354}","first_borrow_whitelist_in":"1","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":932,\\"h5_available_left\\":0,\\"h5_available_top\\":0,\\"h5_available_width\\":430,\\"h5_colorDepth\\":24,\\"h5_devicePixelRatio\\":3,\\"h5_height\\":932,\\"h5_innerHeight\\":745,\\"h5_innerWidth\\":430,\\"h5_keepAwake\\":true,\\"h5_maxTouchPoints\\":5,\\"h5_orientation_angle\\":0,\\"h5_orientation_type\\":\\"portrait-primary\\",\\"h5_pixelDepth\\":24,\\"h5_width\\":430}","loan_id":"0","ios_charging_status":"","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"SVD20250924040059j4dJ31WY","age":"30","last_credit_amt":"2700","h5_vendor":"Apple Computer, Inc.","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1758704460164","ph_diffzalo_num":"3","h5_hardwareConcurrency":"4","last_settlement_order_difference":"0.26","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"5","lng":"100.79329515892873","another_contact_name":"ทัศนาทิพย์","ip":"","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"c9520a45f9bb4d59b3865919b6538392","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"2","borrow_count_3":"1","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"90.8","borrow_count_1":"2","const_id":"","liveness_livenessScore":" \'100\'","last_contract_amount_10":"0","diffphone_face_num":"0","borrow_create_time":"1733670241006","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"2700.00","last_contract_amount_1":"3500.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"Gecko","first_credit_amount_9":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"2000.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3500.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"500.00","borrow_count":"5","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.88897281275967","if_live_detection":"1","automatic_face_pass":"1","same_calendarid_dif_phone":"0","ios_app_version":"","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"1","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"","h5_language":"th-TH","h5_productSub":"20030107","h5_appName":"Netscape","personal_education":"","ios_app_package":"","ios_battery_percentage":"","borrowing_loan_id":"2,3","emergency_is_others":"0"}',
        "response_data": '{"uuid":"3a0daecf-6fe3-439f-8bc0-cde588bd6a03","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25092404vgrl79TZ","extraInfo":{"_cost_time":168,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"3a0daecf-6fe3-439f-8bc0-cde588bd6a03,1758704680198,e8ef00b84c7260c6aa21a77e2a376697"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"14","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":4800.0,"th_beh_reject_pctb2407_meta_20240717":"0.6667","credit_amt_2":4000.0,"credit_amt_3":1800.0,"Expected_loan_time_1":"1758704460164","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1758704460164","Expected_loan_time_3":"1758704460164","th_app_last_21d_social_update_cnt_meta_20240715":0,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6000","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.7143","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":0,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":0,"th_app_last_3m_highrisk_install_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","credit_amt":"3500","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.7143","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":0}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "SVD20250924040059j4dJ31WY",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
        "verify_time": 1758704463,
    },
    {
        "app_name": "easyloan_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 0,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1757045044,
        "request_data": '{"Settlement_frequency_2":"2","Settlement_frequency_1":"2","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"1","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"27.55.78.68","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1757045044","diffid_face_num":"0","ios_manufacturer":"","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"Mozilla","app_id":"118","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"tree","ios_system_version":"","r81_60p70p81_b":"0.75","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"4","geohash_add_time":"1758709265000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148 Safari Line/15.13.0","email":"","user_due_amount_sum":"3200.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.0106481481","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[\\"th-TH\\"]","app_name":"happy cashing_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"0.9371180556","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"-6","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jdytfg0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"2700","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"500","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"3500","h5_browserName":"Safari","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"","h5_platform":"iPhone","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"3200.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0,\\"app_9999d_fi_效率_cnt\\":0,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":4800,\\"pl_loan_5o_user_avg_wait_hours\\":7.77074,\\"pl_loan_7o_phone_max_amt\\":4800,\\"pl_loan_9999d_phone_max_amt\\":4800,\\"pl_loan_9999d_user_max_amt\\":4800,\\"pl_repay_1o_phone_early_latest_hours\\":0.25583,\\"pl_repay_1o_phone_ontime_latest_hours\\":0.25583,\\"pl_repay_2o_phone_early_earliest_hours\\":0.25583,\\"pl_repay_2o_phone_ontime_latest_hours\\":0.25583,\\"pl_repay_30d_phone_ontime_latest_hours\\":0.25583,\\"pl_repay_30d_user_ontime_latest_hours\\":0.25583,\\"pl_repay_3o_phone_early_cnt\\":1,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.72174,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":316.37249,\\"pl_repay_60d_phone_ontime_earliest_hours\\":316.37249,\\"pl_repay_60d_phone_ontime_latest_hours\\":0.25583,\\"pl_repay_60d_user_latest_overdue_days\\":-6,\\"pl_repay_60d_user_ontime_earliest_hours\\":316.37249,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.16542,\\"pl_repay_90d_phone_early_cnt\\":2,\\"pl_repay_90d_user_early_cnt\\":2,\\"pl_repay_9999d_phone_cnt\\":9,\\"pl_repay_9999d_phone_early_cnt\\":2,\\"pl_repay_9999d_user_latest_overdue_days\\":-6,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.33017585576734954,\\"th_ios_bcard_2508_score\\":584.0902117635943}","first_borrow_whitelist_in":"1","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":932,\\"h5_available_left\\":0,\\"h5_available_top\\":0,\\"h5_available_width\\":430,\\"h5_colorDepth\\":24,\\"h5_devicePixelRatio\\":3,\\"h5_height\\":932,\\"h5_innerHeight\\":745,\\"h5_innerWidth\\":430,\\"h5_keepAwake\\":true,\\"h5_maxTouchPoints\\":5,\\"h5_orientation_angle\\":0,\\"h5_orientation_type\\":\\"portrait-primary\\",\\"h5_pixelDepth\\":24,\\"h5_width\\":430}","loan_id":"0","ios_charging_status":"","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"SVD20250924052117a8p7164h","age":"30","last_credit_amt":"2700","h5_vendor":"Apple Computer, Inc.","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1758709279555","ph_diffzalo_num":"3","h5_hardwareConcurrency":"4","last_settlement_order_difference":"0.01","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"5","lng":"100.79762307608006","another_contact_name":"ทัศนาทิพย์","ip":"","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"c9520a45f9bb4d59b3865919b6538392","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"2","borrow_count_3":"1","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"20","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"90.8","borrow_count_1":"2","const_id":"","liveness_livenessScore":" \'100\'","last_contract_amount_10":"0","diffphone_face_num":"0","borrow_create_time":"1733670241006","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"2700.00","last_contract_amount_1":"4800.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"Gecko","first_credit_amount_9":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"2000.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3500.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"500.00","borrow_count":"5","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.891111321004075","if_live_detection":"1","automatic_face_pass":"1","same_calendarid_dif_phone":"0","ios_app_version":"","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"1","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"","h5_language":"th-TH","h5_productSub":"20030107","h5_appName":"Netscape","personal_education":"","ios_app_package":"","ios_battery_percentage":"","borrowing_loan_id":"2,3","emergency_is_others":"0"}',
        "response_data": '{"uuid":"a9d61b38-a156-417a-9bab-5b52fd38a1a1","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz250924051T9HjmK3","extraInfo":{"_cost_time":174,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"a9d61b38-a156-417a-9bab-5b52fd38a1a1,1758709015212,fb00f1f56e31445db222c7d649cec629"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"17","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":4800.0,"th_beh_reject_pctb2407_meta_20240717":"0.6154","credit_amt_2":4000.0,"credit_amt_3":1800.0,"Expected_loan_time_1":"1758709279555","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1758709279555","Expected_loan_time_3":"1758709279555","th_app_last_21d_social_update_cnt_meta_20240715":0,"lastoverdue_day":"-6","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6667","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.7500","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":0,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":0,"th_app_last_3m_highrisk_install_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","credit_amt":"3500","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.7500","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":0}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "SVD20250924052117a8p7164h",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
        "verify_time": 1758709283,
    },
    {
        "app_name": "easyloan_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 2,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1757045044,
        "request_data": '{"Settlement_frequency_2":"2","Settlement_frequency_1":"2","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"2","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"1.0.210.98","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1757045044","diffid_face_num":"0","ios_manufacturer":"","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"Mozilla","app_id":"118","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"tree","ios_system_version":"","r81_60p70p81_b":"0.88","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"5","geohash_add_time":"1758850940000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148 Safari Line/15.13.0","email":"","user_due_amount_sum":"2700.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"1.2209837963","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[\\"th-TH\\"]","app_name":"happy cashing_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"3.7980092593","last_overdue_days_3":"-3","diffbank_id_num":"0","last_overdue_days_1":"-6","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w4x6cycpn80","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"2700","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"500","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"3500","h5_browserName":"Safari","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"","h5_platform":"iPhone","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"2700.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0,\\"app_9999d_fi_效率_cnt\\":0,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":4800,\\"pl_loan_5o_user_avg_wait_hours\\":7.77074,\\"pl_loan_7o_phone_max_amt\\":4800,\\"pl_loan_9999d_phone_max_amt\\":4800,\\"pl_loan_9999d_user_max_amt\\":4800,\\"pl_repay_1o_phone_early_latest_hours\\":68.917,\\"pl_repay_1o_phone_ontime_latest_hours\\":68.917,\\"pl_repay_2o_phone_early_earliest_hours\\":68.917,\\"pl_repay_2o_phone_ontime_latest_hours\\":68.917,\\"pl_repay_30d_phone_ontime_latest_hours\\":68.917,\\"pl_repay_30d_user_ontime_latest_hours\\":68.917,\\"pl_repay_3o_phone_early_cnt\\":2,\\"pl_repay_3o_phone_s-1_cnt\\":1,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.76522,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.25,\\"pl_repay_60d_phone_early_earliest_hours\\":385.03367,\\"pl_repay_60d_phone_ontime_earliest_hours\\":385.03367,\\"pl_repay_60d_phone_ontime_latest_hours\\":68.917,\\"pl_repay_60d_user_latest_overdue_days\\":-6,\\"pl_repay_60d_user_ontime_earliest_hours\\":385.03367,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.18872,\\"pl_repay_90d_phone_early_cnt\\":3,\\"pl_repay_90d_user_early_cnt\\":3,\\"pl_repay_9999d_phone_cnt\\":10,\\"pl_repay_9999d_phone_early_cnt\\":3,\\"pl_repay_9999d_user_latest_overdue_days\\":-6,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.2839347886237775,\\"th_ios_bcard_2508_score\\":590.3698837909137}","first_borrow_whitelist_in":"1","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":932,\\"h5_available_left\\":0,\\"h5_available_top\\":0,\\"h5_available_width\\":430,\\"h5_colorDepth\\":24,\\"h5_devicePixelRatio\\":3,\\"h5_height\\":932,\\"h5_innerHeight\\":745,\\"h5_innerWidth\\":430,\\"h5_keepAwake\\":true,\\"h5_maxTouchPoints\\":5,\\"h5_orientation_angle\\":0,\\"h5_orientation_type\\":\\"portrait-primary\\",\\"h5_pixelDepth\\":24,\\"h5_width\\":430}","loan_id":"0","ios_charging_status":"","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"SVD20250927020058GqHND71f","age":"30","last_credit_amt":"2700","h5_vendor":"Apple Computer, Inc.","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1758956458685","ph_diffzalo_num":"3","h5_hardwareConcurrency":"4","last_settlement_order_difference":"1.22","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"5","lng":"100.27387522706425","another_contact_name":"ทัศนาทิพย์","ip":"","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"c9520a45f9bb4d59b3865919b6538392","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"2","borrow_count_3":"2","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"20","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"90.8","borrow_count_1":"2","const_id":"","liveness_livenessScore":" \'100\'","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1733670421004","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"2700.00","last_contract_amount_1":"4800.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"Gecko","first_credit_amount_9":"0","churn_type":"1,2","current_app_overdue_order_cnt":"0","first_credit_amount_2":"2000.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3500.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"500.00","borrow_count":"6","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"14.584183476728965","if_live_detection":"1","automatic_face_pass":"1","same_calendarid_dif_phone":"0","ios_app_version":"","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"1","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"","h5_language":"th-TH","h5_productSub":"20030107","h5_appName":"Netscape","personal_education":"","ios_app_package":"","ios_battery_percentage":"","borrowing_loan_id":"2","emergency_is_others":"0"}',
        "response_data": '{"uuid":"e2d53e88-b381-46aa-99a9-6461d0513eae","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz250927021oSQ4FZ9","extraInfo":{"_cost_time":191,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"e2d53e88-b381-46aa-99a9-6461d0513eae,1758956392574,4ceaa66f15be7d8c43db7305b1561acf"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"20","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":5000.0,"th_beh_reject_pctb2407_meta_20240717":"0.5714","credit_amt_2":4200.0,"credit_amt_3":2000.0,"Expected_loan_time_1":"1758956458685","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1758956458685","Expected_loan_time_3":"1758956458685","th_app_last_21d_social_update_cnt_meta_20240715":0,"lastoverdue_day":"-3","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.8333","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.8750","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":0,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":0,"th_app_last_3m_highrisk_install_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","credit_amt":"3500","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.8750","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":0}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "SVD20250927020058GqHND71f",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
        "verify_time": 1758956463,
    },
    {
        "app_name": "easyloan_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 2,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1757045044,
        "request_data": '{"Settlement_frequency_2":"3","Settlement_frequency_1":"2","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"3","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"49.237.35.169","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1757045044","diffid_face_num":"0","ios_manufacturer":"","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"Mozilla","app_id":"118","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"tree","ios_system_version":"","r81_60p70p81_b":"0.82","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"6","geohash_add_time":"1758956514000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148 Safari Line/15.13.0","email":"","user_due_amount_sum":"8200.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"2.0586458333","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[\\"th-TH\\"]","app_name":"happy cashing_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"3.1006134259","last_overdue_days_3":"-4","diffbank_id_num":"0","last_overdue_days_1":"-6","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jde8160","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"2700","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"2000","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"5000","h5_browserName":"Safari","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"","h5_platform":"iPhone","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"8200.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0,\\"app_9999d_fi_效率_cnt\\":0,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":3200,\\"pl_loan_5o_user_avg_wait_hours\\":69.12889,\\"pl_loan_7o_phone_max_amt\\":5000,\\"pl_loan_9999d_phone_max_amt\\":5000,\\"pl_loan_9999d_user_max_amt\\":5000,\\"pl_repay_30d_phone_ontime_latest_hours\\":49.40768,\\"pl_repay_30d_user_ontime_latest_hours\\":49.40768,\\"pl_repay_3o_phone_early_cnt\\":1,\\"pl_repay_3o_phone_s-1_cnt\\":1,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.45333,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.33333,\\"pl_repay_60d_phone_early_earliest_hours\\":469.92157,\\"pl_repay_60d_phone_ontime_earliest_hours\\":469.92157,\\"pl_repay_60d_phone_ontime_latest_hours\\":49.40768,\\"pl_repay_60d_user_latest_overdue_days\\":-4,\\"pl_repay_60d_user_ontime_earliest_hours\\":469.92157,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.23795,\\"pl_repay_90d_phone_early_cnt\\":4,\\"pl_repay_90d_user_early_cnt\\":4,\\"pl_repay_9999d_phone_cnt\\":12,\\"pl_repay_9999d_phone_early_cnt\\":4,\\"pl_repay_9999d_user_latest_overdue_days\\":-4,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.3058059556646397,\\"th_ios_bcard_2508_score\\":587.333706378399}","first_borrow_whitelist_in":"1","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":932,\\"h5_available_left\\":0,\\"h5_available_top\\":0,\\"h5_available_width\\":430,\\"h5_colorDepth\\":24,\\"h5_devicePixelRatio\\":3,\\"h5_height\\":932,\\"h5_innerHeight\\":745,\\"h5_innerWidth\\":430,\\"h5_keepAwake\\":true,\\"h5_maxTouchPoints\\":5,\\"h5_orientation_angle\\":0,\\"h5_orientation_type\\":\\"portrait-primary\\",\\"h5_pixelDepth\\":24,\\"h5_width\\":430}","loan_id":"0","ios_charging_status":"","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"SVD202510010254155132qUwX","age":"30","last_credit_amt":"5000","h5_vendor":"Apple Computer, Inc.","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1759262055298","ph_diffzalo_num":"3","h5_hardwareConcurrency":"4","last_settlement_order_difference":"2.06","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"5","lng":"100.79343569760573","another_contact_name":"ทัศนาทิพย์","ip":"","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"c9520a45f9bb4d59b3865919b6538392","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"3","borrow_count_3":"3","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"11","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"90.8","borrow_count_1":"2","const_id":"","liveness_livenessScore":" \'100\'","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730307660002","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"2700.00","last_contract_amount_1":"5000.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"Gecko","first_credit_amount_9":"0","churn_type":"1,2","current_app_overdue_order_cnt":"0","first_credit_amount_2":"2000.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3500.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"2000.00","borrow_count":"8","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888745398743353","if_live_detection":"1","automatic_face_pass":"1","same_calendarid_dif_phone":"0","ios_app_version":"","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"1","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"","h5_language":"th-TH","h5_productSub":"20030107","h5_appName":"Netscape","personal_education":"","ios_app_package":"","ios_battery_percentage":"","borrowing_loan_id":"1","emergency_is_others":"0"}',
        "response_data": '{"uuid":"6e3fac67-477b-4c8c-b8a6-2634ba80d14c","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz251001025Inf8YiP","extraInfo":{"_cost_time":196,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"6e3fac67-477b-4c8c-b8a6-2634ba80d14c,1759261988195,1412d876aa14aa5ccc56251ca99cdea8"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"20","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":6000.0,"th_beh_reject_pctb2407_meta_20240717":"0.5000","credit_amt_2":3700.0,"credit_amt_3":3000.0,"Expected_loan_time_1":"1759262055298","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1759262055298","Expected_loan_time_3":"1759262055298","th_app_last_21d_social_update_cnt_meta_20240715":0,"lastoverdue_day":"-4","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.7778","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.8182","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":1.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":0,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":0,"th_app_last_3m_highrisk_install_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","credit_amt":"5000","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.8182","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":0}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "SVD202510010254155132qUwX",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
        "verify_time": 1759262060,
    },
    {
        "app_name": "easyloan_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 0,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1757045044,
        "request_data": '{"Settlement_frequency_2":"3","Settlement_frequency_1":"3","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"3","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"27.55.92.181","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1757045044","diffid_face_num":"0","ios_manufacturer":"","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"Mozilla","app_id":"118","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"tree","ios_system_version":"","r81_60p70p81_b":"0.77","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"7","geohash_add_time":"1759262340000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148 Safari Line/15.13.0","email":"","user_due_amount_sum":"9900.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.0364467593","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[\\"th-TH\\"]","app_name":"happy cashing_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"5.9222685185","last_overdue_days_3":"-4","diffbank_id_num":"0","last_overdue_days_1":"-6","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w4zrm3gbwv0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"3700","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"3000","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"5000","h5_browserName":"Safari","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"","h5_platform":"iPhone","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"9900.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0,\\"app_9999d_fi_效率_cnt\\":0,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":3000,\\"pl_loan_5o_user_avg_wait_hours\\":61.05375,\\"pl_loan_7o_phone_max_amt\\":5000,\\"pl_loan_9999d_phone_max_amt\\":5000,\\"pl_loan_9999d_user_max_amt\\":5000,\\"pl_repay_30d_phone_ontime_latest_hours\\":7.94095,\\"pl_repay_30d_user_ontime_latest_hours\\":7.94095,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.33557,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.5,\\"pl_repay_60d_phone_early_earliest_hours\\":537.64123,\\"pl_repay_60d_phone_ontime_earliest_hours\\":537.64123,\\"pl_repay_60d_phone_ontime_latest_hours\\":7.94095,\\"pl_repay_60d_user_latest_overdue_days\\":-6,\\"pl_repay_60d_user_ontime_earliest_hours\\":537.64123,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.35784,\\"pl_repay_90d_phone_early_cnt\\":5,\\"pl_repay_90d_user_early_cnt\\":5,\\"pl_repay_9999d_phone_cnt\\":14,\\"pl_repay_9999d_phone_early_cnt\\":5,\\"pl_repay_9999d_user_latest_overdue_days\\":-6,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.29567757655647514,\\"th_ios_bcard_2508_score\\":588.7234802399258}","first_borrow_whitelist_in":"1","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":932,\\"h5_available_left\\":0,\\"h5_available_top\\":0,\\"h5_available_width\\":430,\\"h5_colorDepth\\":24,\\"h5_devicePixelRatio\\":3,\\"h5_height\\":932,\\"h5_innerHeight\\":745,\\"h5_innerWidth\\":430,\\"h5_keepAwake\\":true,\\"h5_maxTouchPoints\\":5,\\"h5_orientation_angle\\":0,\\"h5_orientation_type\\":\\"portrait-primary\\",\\"h5_pixelDepth\\":24,\\"h5_width\\":430}","loan_id":"0","ios_charging_status":"","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"SVD20251003103725sctxsD4q","age":"30","last_credit_amt":"3700","h5_vendor":"Apple Computer, Inc.","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1759505845947","ph_diffzalo_num":"3","h5_hardwareConcurrency":"4","last_settlement_order_difference":"0.04","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"5","lng":"100.43146917939956","another_contact_name":"ทัศนาทิพย์","ip":"","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"c9520a45f9bb4d59b3865919b6538392","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"3","borrow_count_3":"3","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"90.8","borrow_count_1":"3","const_id":"","liveness_livenessScore":" \'100\'","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730307782002","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"3700.00","last_contract_amount_1":"5000.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"Gecko","first_credit_amount_9":"0","churn_type":"1,2","current_app_overdue_order_cnt":"0","first_credit_amount_2":"2000.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3500.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"3000.00","borrow_count":"9","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.752890981038192","if_live_detection":"1","automatic_face_pass":"1","same_calendarid_dif_phone":"0","ios_app_version":"","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"1","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"","h5_language":"th-TH","h5_productSub":"20030107","h5_appName":"Netscape","personal_education":"","ios_app_package":"","ios_battery_percentage":"","borrowing_loan_id":"2,3","emergency_is_others":"0"}',
        "response_data": '{"uuid":"2c75c0cc-ef09-4a9e-9b03-44e4f4bb26f0","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25100310v69V7g0o","extraInfo":{"_cost_time":175,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"2c75c0cc-ef09-4a9e-9b03-44e4f4bb26f0,1759505577084,086e4357ea61431cfaf8a20644d95523"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"24","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":6000.0,"th_beh_reject_pctb2407_meta_20240717":"0.4706","credit_amt_2":4700.0,"credit_amt_3":4000.0,"Expected_loan_time_1":"1759505845947","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1759505845947","Expected_loan_time_3":"1759505845947","th_app_last_21d_social_update_cnt_meta_20240715":0,"lastoverdue_day":"-6","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6250","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.7692","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":1.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":0,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":0,"th_app_last_3m_highrisk_install_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","credit_amt":"5000","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.7692","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":0}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "SVD20251003103725sctxsD4q",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
        "verify_time": 1759505850,
    },
    {
        "app_name": "easyloan_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1757045044,
        "request_data": '{"Settlement_frequency_2":"3","Settlement_frequency_1":"3","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"3","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"49.237.10.135","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1757045044","diffid_face_num":"0","ios_manufacturer":"","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"Mozilla","app_id":"118","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"tree","ios_system_version":"","r81_60p70p81_b":"0.77","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"7","geohash_add_time":"1759515833000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148 Safari Line/15.13.0","email":"","user_due_amount_sum":"9900.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.1521296296","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[\\"th-TH\\"]","app_name":"happy cashing_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"6.0379513889","last_overdue_days_3":"-4","diffbank_id_num":"0","last_overdue_days_1":"-6","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w4xnnctddn0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"3700","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"3000","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"5000","h5_browserName":"Safari","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"","h5_platform":"iPhone","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"9900.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0,\\"app_9999d_fi_效率_cnt\\":0,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":3000,\\"pl_loan_5o_user_avg_wait_hours\\":61.05375,\\"pl_loan_7o_phone_max_amt\\":5000,\\"pl_loan_9999d_phone_max_amt\\":5000,\\"pl_loan_9999d_user_max_amt\\":5000,\\"pl_repay_30d_phone_ontime_latest_hours\\":10.71749,\\"pl_repay_30d_user_ontime_latest_hours\\":10.71749,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.33557,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.5,\\"pl_repay_60d_phone_early_earliest_hours\\":540.41776,\\"pl_repay_60d_phone_ontime_earliest_hours\\":540.41776,\\"pl_repay_60d_phone_ontime_latest_hours\\":10.71749,\\"pl_repay_60d_user_latest_overdue_days\\":-6,\\"pl_repay_60d_user_ontime_earliest_hours\\":540.41776,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.35784,\\"pl_repay_90d_phone_early_cnt\\":5,\\"pl_repay_90d_user_early_cnt\\":5,\\"pl_repay_9999d_phone_cnt\\":14,\\"pl_repay_9999d_phone_early_cnt\\":5,\\"pl_repay_9999d_user_latest_overdue_days\\":-6,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.29567757655647514,\\"th_ios_bcard_2508_score\\":588.7234802399258}","first_borrow_whitelist_in":"1","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":932,\\"h5_available_left\\":0,\\"h5_available_top\\":0,\\"h5_available_width\\":430,\\"h5_colorDepth\\":24,\\"h5_devicePixelRatio\\":3,\\"h5_height\\":932,\\"h5_innerHeight\\":745,\\"h5_innerWidth\\":430,\\"h5_keepAwake\\":true,\\"h5_maxTouchPoints\\":5,\\"h5_orientation_angle\\":0,\\"h5_orientation_type\\":\\"portrait-primary\\",\\"h5_pixelDepth\\":24,\\"h5_width\\":430}","loan_id":"0","ios_charging_status":"","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"SVD2025100401240120KR4v2t","age":"30","last_credit_amt":"3700","h5_vendor":"Apple Computer, Inc.","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1759515842004","ph_diffzalo_num":"3","h5_hardwareConcurrency":"4","last_settlement_order_difference":"0.15","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"5","lng":"100.14802394373636","another_contact_name":"ทัศนาทิพย์","ip":"","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"c9520a45f9bb4d59b3865919b6538392","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"3","borrow_count_3":"3","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"90.8","borrow_count_1":"3","const_id":"","liveness_livenessScore":" \'100\'","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730307840001","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"3700.00","last_contract_amount_1":"5000.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"Gecko","first_credit_amount_9":"0","churn_type":"1,2","current_app_overdue_order_cnt":"0","first_credit_amount_2":"2000.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3500.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"3000.00","borrow_count":"9","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"15.125891586487317","if_live_detection":"1","automatic_face_pass":"1","same_calendarid_dif_phone":"0","ios_app_version":"","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"1","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"","h5_language":"th-TH","h5_productSub":"20030107","h5_appName":"Netscape","personal_education":"","ios_app_package":"","ios_battery_percentage":"","borrowing_loan_id":"2,3","emergency_is_others":"0"}',
        "response_data": '{"uuid":"63bea853-92ba-4ddd-8857-2467aff478a3","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25100401vm3K0TuO","extraInfo":{"_cost_time":174,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"63bea853-92ba-4ddd-8857-2467aff478a3,1759515572660,703f6f24e6c3356c08f9a59f6923e0a9"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"27","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":6000.0,"th_beh_reject_pctb2407_meta_20240717":"0.4444","credit_amt_2":4700.0,"credit_amt_3":4000.0,"Expected_loan_time_1":"1759515842004","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1759515842004","Expected_loan_time_3":"1759515842004","th_app_last_21d_social_update_cnt_meta_20240715":0,"lastoverdue_day":"-6","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6250","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.7692","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":1.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":0,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":0,"th_app_last_3m_highrisk_install_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","credit_amt":"5000","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.7692","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":0}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "SVD2025100401240120KR4v2t",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
        "verify_time": 1759515846,
    },
    {
        "app_name": "easyloan_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1757045044,
        "request_data": '{"Settlement_frequency_2":"4","Settlement_frequency_1":"3","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"3","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"49.237.78.152","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1757045044","diffid_face_num":"0","ios_manufacturer":"","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"Mozilla","app_id":"118","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"tree","ios_system_version":"","r81_60p70p81_b":"0.71","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"8","geohash_add_time":"1759825602000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148 Safari Line/15.13.0","email":"","user_due_amount_sum":"14100.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.1679398148","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[\\"th-TH\\"]","app_name":"happy cashing_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"6.5225810185","last_overdue_days_3":"-4","diffbank_id_num":"0","last_overdue_days_1":"-6","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jd7x9w0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"3700","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"3000","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"6000","h5_browserName":"Safari","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"","h5_platform":"iPhone","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"14100.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0,\\"app_9999d_fi_效率_cnt\\":0,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":500,\\"pl_loan_5o_user_avg_wait_hours\\":38.89574,\\"pl_loan_7o_phone_max_amt\\":6000,\\"pl_loan_9999d_phone_max_amt\\":6000,\\"pl_loan_9999d_user_max_amt\\":6000,\\"pl_repay_30d_phone_ontime_latest_hours\\":4.03069,\\"pl_repay_30d_user_ontime_latest_hours\\":4.03069,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":626.46403,\\"pl_repay_60d_phone_ontime_earliest_hours\\":626.46403,\\"pl_repay_60d_phone_ontime_latest_hours\\":4.03069,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":626.46403,\\"pl_repay_7o_phone_early_amt_all_pct\\":0,\\"pl_repay_90d_phone_early_cnt\\":5,\\"pl_repay_90d_user_early_cnt\\":5,\\"pl_repay_9999d_phone_cnt\\":16,\\"pl_repay_9999d_phone_early_cnt\\":5,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.34322310168376463,\\"th_ios_bcard_2508_score\\":582.4043911181227}","first_borrow_whitelist_in":"1","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":932,\\"h5_available_left\\":0,\\"h5_available_top\\":0,\\"h5_available_width\\":430,\\"h5_colorDepth\\":24,\\"h5_devicePixelRatio\\":3,\\"h5_height\\":932,\\"h5_innerHeight\\":745,\\"h5_innerWidth\\":430,\\"h5_keepAwake\\":true,\\"h5_maxTouchPoints\\":5,\\"h5_orientation_angle\\":0,\\"h5_orientation_type\\":\\"portrait-primary\\",\\"h5_pixelDepth\\":24,\\"h5_width\\":430}","loan_id":"0","ios_charging_status":"","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"SVD20251007032649WD6Lmip0","age":"30","last_credit_amt":"6000","h5_vendor":"Apple Computer, Inc.","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1759825609256","ph_diffzalo_num":"3","h5_hardwareConcurrency":"4","last_settlement_order_difference":"0.17","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"5","lng":"100.79344637980658","another_contact_name":"ทัศนาทิพย์","ip":"","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"c9520a45f9bb4d59b3865919b6538392","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"4","borrow_count_3":"3","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"90.8","borrow_count_1":"3","const_id":"","liveness_livenessScore":" \'100\'","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730308021005","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"3700.00","last_contract_amount_1":"6000.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"Gecko","first_credit_amount_9":"0","churn_type":"1","current_app_overdue_order_cnt":"0","first_credit_amount_2":"2000.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3500.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"3000.00","borrow_count":"10","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.88868089174534","if_live_detection":"1","automatic_face_pass":"1","same_calendarid_dif_phone":"0","ios_app_version":"","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"1","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"","h5_language":"th-TH","h5_productSub":"20030107","h5_appName":"Netscape","personal_education":"","ios_app_package":"","ios_battery_percentage":"","borrowing_loan_id":"1,3","emergency_is_others":"0"}',
        "response_data": '{"uuid":"eedbbfc7-6480-4c93-9b51-de914e8bb9ed","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25100703k85vJ5gp","extraInfo":{"_cost_time":146,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"eedbbfc7-6480-4c93-9b51-de914e8bb9ed,1759825539174,ff704105700f85502be4d54d9b61439b"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"26","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":6800.0,"th_beh_reject_pctb2407_meta_20240717":"0.3810","credit_amt_2":4500.0,"credit_amt_3":3800.0,"Expected_loan_time_1":"1759825609256","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1759825609256","Expected_loan_time_3":"1759825609256","th_app_last_21d_social_update_cnt_meta_20240715":0,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.5833","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.7059","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":0,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":0,"th_app_last_3m_highrisk_install_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","credit_amt":"6000","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.7059","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":0}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "SVD20251007032649WD6Lmip0",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
        "verify_time": 1759825612,
    },
    {
        "app_name": "finnix_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 0,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739937444,
        "request_data": '{"Settlement_frequency_2":"0","Settlement_frequency_1":"0","Settlement_frequency_4":"0","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","diffbank_face_num":"0","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"66","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.3.1","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"0","last_7d_self_same_face_bankcard_cnt":"0","geohash_add_time":"1739937684000","ios_device_name":"iPhone","diffphone_id_num":"0","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"0","ocr_up_type":"1","id_expiry_date":"02-08-2030","call_risk_type":"0","id_number_ocr":"00000000001892031979169771522","h5_languages":"[]","app_name":"finnix_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jdedfd0","last_overdue_days_5":"0","last_credit_amt_2":"0","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"0","h5_browserName":"","last_7d_self_same_face_id_cnt":"0","personal_city":"792","client_ip":"49.237.15.40","h5_platform":"","geohash_type":"riskengine","last_financial_product_id":"None","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","phone_number":"00000000001865571048323723445","fea_list":"{\\"app_14d_fi_工具_trend\\":0,\\"app_9999d_fi_cnt\\":53,\\"calllog_14d_tag_移动电话_time_pct\\":0,\\"calllog_9999d_frequent_phone_cnt\\":0,\\"sms_14d_cashin_sum\\":0,\\"sms_14d_贷款_phone_pct\\":0,\\"sms_14d_贷款_审核通知_pct\\":0,\\"sms_14d_赌场_cnt\\":0,\\"sms_21d_21d_营销_pct\\":0,\\"sms_21d_贷款_审核通知_pct\\":0,\\"sms_30d_30d_营销_distinct_pct\\":0,\\"sms_30d_phone_cnt\\":0,\\"sms_30d_贷款_all_pct\\":0,\\"sms_7d_7d_贷款_pct\\":0,\\"sms_7d_贷款_phone_pct\\":0,\\"sms_7d_贷款_审核通知_pct\\":0,\\"sms_7d_金融类_cnt\\":0,\\"sms_9999d_营销_cnt\\":0,\\"sms_9999d_营销_distinct_cnt\\":0,\\"sms_9999d_贷款_营销_cnt\\":0,\\"sms_赌场_pct\\":0}","user_contract_amount_sum":"0","first_borrow_whitelist_in":"0","personal_province":"788","wl_uid":"None","age_revise":"29","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"2","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"1","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"SVD20250219110128lgWBH40K","age":"29","last_credit_amt":"0","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1739937689139","ph_diffzalo_num":"1","h5_hardwareConcurrency":"","last_settlement_order_difference":"-1","last_7d_all_same_face_phone_cnt":"1","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.79348986586703","ip":"49.237.15.40","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"38c500bb4b894f9b97b3d3491500423c","other_contact_phone":"0800699457","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"0","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"1","borrow_count_9":"0","ios_rooted":"false","if_dk_event":"1","face_similarity":"93.85","borrow_count_1":"0","const_id":"07119AAB-6DEC-4E5E-BC6C-0DB1C26D5F31","liveness_livenessScore":"99.98","last_contract_amount_10":"0","diffphone_face_num":"0","borrow_create_time":"1733069341001","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"0","last_contract_amount_1":"0","h5_product":"","last_7d_self_same_face_apply_cnt":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"0","last_7d_all_same_face_id_cnt":"1","platform_check_baiduai_black":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.889220522133716","if_live_detection":"1","automatic_face_pass":"0","ios_app_version":"1.0.2","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"0","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"07119AAB-6DEC-4E5E-BC6C-0DB1C26D5F31","h5_language":"th-TH","h5_productSub":"","h5_appName":"Signature Hub","personal_education":"7","ios_app_package":"com.signature.easysign","ios_battery_percentage":"0.4","borrowing_loan_id":""}',
        "response_data": '{"uuid":"8064f413-190c-4355-a71d-8baa86631d03","status":"SUCCESS","ctuEntireResult":{"riskLevel":"REJECT","riskType":"concatnum,model_score","hitRules":[{"id":393266594775580,"leftValue":"异常校验"},{"id":405047798071301,"leftValue":"白名单_投放返回"},{"id":393266596872274,"leftValue":"异常校验"},{"id":393266594775632,"leftValue":"异常校验"},{"id":393266596872241,"leftValue":"异常校验"},{"id":393266575901154,"leftValue":"泰国_IOS_首贷_非正常安装程序_Meta_20240620"},{"id":393266575901167,"leftValue":"泰国_IOS_首贷_越狱_Meta_20240620"},{"id":393266575901182,"leftValue":"泰国_IOS_首贷_纯新户_历史拒绝次数_Meta_20240621"},{"id":393266575901197,"leftValue":"泰国_IOS_首贷_纯新户_onelink_Meta_20240621"},{"id":393266575901212,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同姓名"},{"id":393266575901225,"leftValue":"泰国_IOS_首贷_纯新户_同银行卡不同姓名"},{"id":393266575901238,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同姓名"},{"id":393266575901251,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同line"},{"id":393266575901264,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同银行卡"},{"id":393266575901277,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同银行卡"},{"id":393266575901292,"leftValue":"泰国_IOS_首贷_纯新户_通讯录相似度_Meta_20240621"},{"id":393266575901305,"leftValue":"泰国_IOS_首贷_纯新户_APP相似度_Meta_20240621"},{"id":393266575901320,"leftValue":"泰国_IOS_首贷_纯新户_同人脸不同手机号_Meta_20240621"},{"id":405023739543636,"leftValue":"电销欺诈"},{"id":393266575900741,"leftValue":"泰国_IOS_首贷_纯新户_当前逾期人脸_Meta_20240620"},{"id":393266575900754,"leftValue":"泰国_IOS_首贷_纯新户_其他平台逾期数量_Meta_20240620"},{"id":393266575900767,"leftValue":"泰国_IOS_首贷_纯新户_活体检测分_Meta_20240620"},{"id":393266575900800,"leftValue":"泰国_IOS_首贷_纯新户_人脸相似度_Meta_20240620"},{"id":393266575900810,"leftValue":"泰国_IOS_首贷_纯新户_有效联系人数量_Meta_20240620"},{"id":393266575900827,"leftValue":"泰国_IOS_首贷_纯新户_年龄_Meta_20240620"},{"id":393266575900844,"leftValue":"泰国_IOS_首贷_纯新户_人脸黑名单_Meta_20240620"},{"id":393266575900854,"leftValue":"泰国_IOS_首贷_纯新户_外部黑名单_Meta_20240620"},{"id":393266575900871,"leftValue":"泰国_IOS_首贷_纯新户_云服务黑名单_Meta_20240620"},{"id":393600954204232,"leftValue":"泰国_IOS_首贷_纯新户_无法识别身份证_Meta_20240620"},{"id":405893860491276,"leftValue":"客服人员准入"},{"id":393266575901524,"leftValue":"泰国_IOS_首贷_纯新户_当前在途订单_Meta_20240620"},{"id":393266584289809,"leftValue":"异常校验"},{"id":393266584289432,"leftValue":"异常校验"},{"id":393266586386450,"leftValue":"应用1额度赋值"},{"id":393266603163673,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25021911Bi7w7794","extraInfo":{"_observe_mode":[false],"_cost_time":527,"ruleIdCodeNamePair":"[{\\"code\\":\\"NAdmission05\\",\\"id\\":393266575900810,\\"name\\":\\"泰国_IOS_首贷_纯新户_有效联系人数量_Meta_20240620\\"},{\\"code\\":\\"ASSIGNCREDIT\\",\\"id\\":393266586386450,\\"name\\":\\"应用1额度赋值\\"}]","riskId":"8064f413-190c-4355-a71d-8baa86631d03,1739937723287,5b1efd12c5f863ca278a09c916ae652d"},"hitPolicies":[{"id":393266575900723,"name":"泰国_电子签_硬规则_首贷_纯新户_准入"},{"id":393266584289281,"name":"泰国_电子签_软规则_首贷_纯新户_模型分"},{"id":393266586386433,"name":"泰国_电子签_后处理_变量返回_基础变量"}],"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":"","th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":"","his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":552,"th_app_last_1d_fin_update_cnt_b2407_meta_20240715":"","th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"","th_ab_model_version_meta_20240628":"2403","reject_days":"3","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":"","th_app_last_7d_food_update_cnt_b2407_meta_20240715":"","is_error":"0","credit_amt_1":"2500","th_beh_reject_pctb2407_meta_20240717":"","Expected_loan_time_1":"1739937689139","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":0,"th_app_last_21d_social_update_cnt_meta_20240715":"","lastoverdue_day":"0","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"","th_app_last_1d_biz_update_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_meta_20240619":0,"financial_product_id_1":"000002","th_model_bscore_meta_20240719":"","th_app_3m_lowrisk_update_cnt_meta_20240619":3,"th_app_last_21d_highrisk_update_cnt_meta_20240715":"","th_model_version_2_meta_20240619":"2405","th_app_loan_earlist_install_days_b2407_meta_20240715":"","risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":"","th_app_last_3m_highrisk_install_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_a2405_meta_20240619":0,"credit_amt":"2500","is_advertising":"0","th_app_last_30d_install_cnt_meta_20240715":"","th_app_loan_latest_install_days_a2405_meta_20240619":9999,"th_app_last_7d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_per_repay_b2407_meta_20240717":"","appliacation_limit":"1","model_type":"2403","th_model_bversion_meta_20240719":"","th_app_last_3d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_pct_meta_20240717":"","th_model_score_2_meta_20240619":607.1,"th_device_ios_model_meta_20240619":4,"overdue_detail":"0","model_score":604.15,"th_app_last_30d_pe_update_cnt_meta_20240715":""}}}',
        "risk_level": "REJECT",
        "risk_type": "concatnum,model_score",
        "sn": "SVD20250219110128lgWBH40K",
        "user_id": "38c500bb4b894f9b97b3d3491500423c",
        "verify_time": 1739937746,
    },
    {
        "app_name": "finnix_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 0,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739937444,
        "request_data": '{"Settlement_frequency_2":"0","Settlement_frequency_1":"0","Settlement_frequency_4":"0","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","diffbank_face_num":"0","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"66","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.3.1","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"0","last_7d_self_same_face_bankcard_cnt":"0","geohash_add_time":"1739937684000","ios_device_name":"iPhone","diffphone_id_num":"0","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"0","ocr_up_type":"1","id_expiry_date":"02-08-2030","call_risk_type":"0","id_number_ocr":"00000000001892031979169771522","h5_languages":"[]","app_name":"ZeedCash","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jdedfd0","last_overdue_days_5":"0","last_credit_amt_2":"0","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"0","h5_browserName":"","last_7d_self_same_face_id_cnt":"0","personal_city":"792","client_ip":"49.237.15.40","h5_platform":"","geohash_type":"riskengine","last_financial_product_id":"None","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","phone_number":"00000000001865571048323723445","fea_list":"{\\"app_14d_fi_工具_trend\\":0,\\"app_9999d_fi_cnt\\":53,\\"calllog_14d_tag_移动电话_time_pct\\":0,\\"calllog_9999d_frequent_phone_cnt\\":0,\\"sms_14d_cashin_sum\\":0,\\"sms_14d_贷款_phone_pct\\":0,\\"sms_14d_贷款_审核通知_pct\\":0,\\"sms_14d_赌场_cnt\\":0,\\"sms_21d_21d_营销_pct\\":0,\\"sms_21d_贷款_审核通知_pct\\":0,\\"sms_30d_30d_营销_distinct_pct\\":0,\\"sms_30d_phone_cnt\\":0,\\"sms_30d_贷款_all_pct\\":0,\\"sms_7d_7d_贷款_pct\\":0,\\"sms_7d_贷款_phone_pct\\":0,\\"sms_7d_贷款_审核通知_pct\\":0,\\"sms_7d_金融类_cnt\\":0,\\"sms_9999d_营销_cnt\\":0,\\"sms_9999d_营销_distinct_cnt\\":0,\\"sms_9999d_贷款_营销_cnt\\":0,\\"sms_赌场_pct\\":0}","user_contract_amount_sum":"0","first_borrow_whitelist_in":"0","personal_province":"788","wl_uid":"None","age_revise":"29","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"2","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"0","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"SVD20250301102844dk2v478B","age":"29","last_credit_amt":"0","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1740799724281","ph_diffzalo_num":"1","h5_hardwareConcurrency":"","last_settlement_order_difference":"-1","last_7d_all_same_face_phone_cnt":"0","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.79348986586703","ip":"49.237.15.40","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"38c500bb4b894f9b97b3d3491500423c","other_contact_phone":"0800699457","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"0","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"0","borrow_count_9":"0","ios_rooted":"false","if_dk_event":"1","face_similarity":"93.85","borrow_count_1":"0","const_id":"07119AAB-6DEC-4E5E-BC6C-0DB1C26D5F31","liveness_livenessScore":"99.98","last_contract_amount_10":"0","diffphone_face_num":"0","borrow_create_time":"1733158861000","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"0","last_contract_amount_1":"0","h5_product":"","last_7d_self_same_face_apply_cnt":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"0","last_7d_all_same_face_id_cnt":"0","platform_check_baiduai_black":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.889220522133716","if_live_detection":"1","automatic_face_pass":"0","ios_app_version":"1.0.2","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"0","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"07119AAB-6DEC-4E5E-BC6C-0DB1C26D5F31","h5_language":"th-TH","h5_productSub":"","h5_appName":"Signature Hub","personal_education":"7","ios_app_package":"com.signature.easysign","ios_battery_percentage":"0.4","borrowing_loan_id":""}',
        "response_data": '{"uuid":"e3b47dec-41df-4375-bfcb-215ac63fb83d","status":"SUCCESS","ctuEntireResult":{"riskLevel":"REJECT","riskType":"concatnum","hitRules":[{"id":393266594775580,"leftValue":"异常校验"},{"id":405047798071301,"leftValue":"白名单_投放返回"},{"id":393266596872274,"leftValue":"异常校验"},{"id":393266594775632,"leftValue":"异常校验"},{"id":393266596872241,"leftValue":"异常校验"},{"id":393266575901154,"leftValue":"泰国_IOS_首贷_非正常安装程序_Meta_20240620"},{"id":393266575901167,"leftValue":"泰国_IOS_首贷_越狱_Meta_20240620"},{"id":393266575901182,"leftValue":"泰国_IOS_首贷_纯新户_历史拒绝次数_Meta_20240621"},{"id":393266575901197,"leftValue":"泰国_IOS_首贷_纯新户_onelink_Meta_20240621"},{"id":393266575901212,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同姓名"},{"id":393266575901225,"leftValue":"泰国_IOS_首贷_纯新户_同银行卡不同姓名"},{"id":393266575901238,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同姓名"},{"id":393266575901251,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同line"},{"id":393266575901264,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同银行卡"},{"id":393266575901277,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同银行卡"},{"id":393266575901292,"leftValue":"泰国_IOS_首贷_纯新户_通讯录相似度_Meta_20240621"},{"id":393266575901305,"leftValue":"泰国_IOS_首贷_纯新户_APP相似度_Meta_20240621"},{"id":393266575901320,"leftValue":"泰国_IOS_首贷_纯新户_同人脸不同手机号_Meta_20240621"},{"id":405023739543636,"leftValue":"电销欺诈"},{"id":393266575900741,"leftValue":"泰国_IOS_首贷_纯新户_当前逾期人脸_Meta_20240620"},{"id":393266575900754,"leftValue":"泰国_IOS_首贷_纯新户_其他平台逾期数量_Meta_20240620"},{"id":393266575900767,"leftValue":"泰国_IOS_首贷_纯新户_活体检测分_Meta_20240620"},{"id":393266575900800,"leftValue":"泰国_IOS_首贷_纯新户_人脸相似度_Meta_20240620"},{"id":393266575900810,"leftValue":"泰国_IOS_首贷_纯新户_有效联系人数量_Meta_20240620"},{"id":393266575900827,"leftValue":"泰国_IOS_首贷_纯新户_年龄_Meta_20240620"},{"id":393266575900844,"leftValue":"泰国_IOS_首贷_纯新户_人脸黑名单_Meta_20240620"},{"id":393266575900854,"leftValue":"泰国_IOS_首贷_纯新户_外部黑名单_Meta_20240620"},{"id":393266575900871,"leftValue":"泰国_IOS_首贷_纯新户_云服务黑名单_Meta_20240620"},{"id":393600954204232,"leftValue":"泰国_IOS_首贷_纯新户_无法识别身份证_Meta_20240620"},{"id":405893860491276,"leftValue":"客服人员准入"},{"id":393266575901524,"leftValue":"泰国_IOS_首贷_纯新户_当前在途订单_Meta_20240620"},{"id":393266584289809,"leftValue":"异常校验"},{"id":393266584289432,"leftValue":"异常校验"},{"id":393266586386450,"leftValue":"应用1额度赋值"},{"id":393266603163673,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25030110ZG0lCL4i","extraInfo":{"_observe_mode":[false],"_cost_time":770,"ruleIdCodeNamePair":"[{\\"code\\":\\"NAdmission05\\",\\"id\\":393266575900810,\\"name\\":\\"泰国_IOS_首贷_纯新户_有效联系人数量_Meta_20240620\\"},{\\"code\\":\\"ASSIGNCREDIT\\",\\"id\\":393266586386450,\\"name\\":\\"应用1额度赋值\\"}]","riskId":"e3b47dec-41df-4375-bfcb-215ac63fb83d,1740799709299,7495682453a9eb8affd4368d487254db"},"hitPolicies":[{"id":393266575900723,"name":"泰国_电子签_硬规则_首贷_纯新户_准入"},{"id":393266586386433,"name":"泰国_电子签_后处理_变量返回_基础变量"}],"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":"","th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":"","his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":562,"th_app_last_1d_fin_update_cnt_b2407_meta_20240715":"","th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"","th_ab_model_version_meta_20240628":"2403","reject_days":"3","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":"","th_app_last_7d_food_update_cnt_b2407_meta_20240715":"","is_error":"0","credit_amt_1":"2500","th_beh_reject_pctb2407_meta_20240717":"","Expected_loan_time_1":"1740799724281","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":0,"th_app_last_21d_social_update_cnt_meta_20240715":"","lastoverdue_day":"0","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"","th_app_last_1d_biz_update_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_meta_20240619":0,"financial_product_id_1":"000002","th_model_bscore_meta_20240719":"","th_app_3m_lowrisk_update_cnt_meta_20240619":3,"th_app_last_21d_highrisk_update_cnt_meta_20240715":"","th_model_version_2_meta_20240619":"2405","th_app_loan_earlist_install_days_b2407_meta_20240715":"","risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":"","th_app_last_3m_highrisk_install_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_a2405_meta_20240619":0,"credit_amt":"2500","is_advertising":"1","th_app_last_30d_install_cnt_meta_20240715":"","th_app_loan_latest_install_days_a2405_meta_20240619":9999,"th_app_last_7d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_per_repay_b2407_meta_20240717":"","appliacation_limit":"1","model_type":"2403","th_model_bversion_meta_20240719":"","th_app_last_3d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_pct_meta_20240717":"","th_model_score_2_meta_20240619":607.1,"th_device_ios_model_meta_20240619":4,"overdue_detail":"0","model_score":604.15,"th_app_last_30d_pe_update_cnt_meta_20240715":""}}}',
        "risk_level": "REJECT",
        "risk_type": "concatnum",
        "sn": "SVD20250301102844dk2v478B",
        "user_id": "38c500bb4b894f9b97b3d3491500423c",
        "verify_time": 1740799734,
    },
    {
        "app_name": "pakincash_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 0,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1751430205,
        "request_data": '{"Settlement_frequency_2":"0","Settlement_frequency_1":"0","Settlement_frequency_4":"0","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"0","diffbank_face_num":"0","register_time_second":"1751430205","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"96","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainfH5","ios_system_version":"18.3.1","r81_60p70p81_b":"0","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0809059719","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"0","borrow_rank":"0","geohash_add_time":"1751430538000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"0","ocr_up_type":"0","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0","call_risk_type":"0","last_financial_product_id_1":"-999","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"-999","last_financial_product_id_3":"-999","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"BMP Lending_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"0","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jd7txr0","last_overdue_days_5":"0","last_credit_amt_2":"0","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"0","h5_browserName":"","last_7d_self_same_face_id_cnt":"0","personal_city":"792","client_ip":"49.237.41.105","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"None","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","phone_number":"00000000001865571048323723445","fea_list":"{\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.1087,\\"app_9999d_fi_效率_cnt\\":5,\\"bj_ios_model_prob\\":0.271092952988736,\\"bj_ios_model_score\\":619.7816890864531,\\"is_available\\":1}","user_contract_amount_sum":"0","first_borrow_whitelist_in":"0","personal_province":"788","wl_uid":"None","age_revise":"29","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001940265737634947074","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"0","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD20250702112955htBe1n71","age":"29","last_credit_amt":"0","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1751430595545","ph_diffzalo_num":"2","h5_hardwareConcurrency":"","last_settlement_order_difference":"-1","last_7d_all_same_face_phone_cnt":"0","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.79369923542805","ip":"49.237.41.105","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"224a4b43210e46468ca641abb5b777de","other_contact_phone":"0800699457","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"0","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"0","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"0","if_dk_event":"1","face_similarity":"89.64","borrow_count_1":"0","const_id":"52F0BBF2-B883-42D8-8019-5117D5898E53","liveness_livenessScore":"99.99","last_contract_amount_10":"0","diffphone_face_num":"0","borrow_create_time":"1733504521001","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"0","last_contract_amount_1":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"0","last_7d_self_same_face_apply_cnt":"0","first_credit_amount_3":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"0","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"0","last_7d_all_same_face_id_cnt":"0","platform_check_baiduai_black":"0","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.88834388668166","if_live_detection":"1","automatic_face_pass":"0","ios_app_version":"1.0.3","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"0","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"52F0BBF2-B883-42D8-8019-5117D5898E53","h5_language":"th-TH","h5_productSub":"","h5_appName":"BMP Lending","personal_education":"7","ios_app_package":"com.fin.money.zc","ios_battery_percentage":"0.5","borrowing_loan_id":""}',
        "response_data": '{"uuid":"992247b1-90f3-4fa1-9c15-329fd8b8a878","status":"SUCCESS","ctuEntireResult":{"riskLevel":"REJECT","riskType":"th_fraud_his_reject_meta_20240621","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197498446007,"leftValue":"异常校验"},{"id":382197500543014,"leftValue":"异常校验"},{"id":411574860709947,"leftValue":"异常校验"},{"id":382197502640796,"leftValue":"异常校验"},{"id":382197500543135,"leftValue":"泰国_IOS_首贷_纯新户_当前逾期人脸_Meta_20240620"},{"id":382197500543148,"leftValue":"泰国_IOS_首贷_纯新户_其他平台逾期数量_Meta_20240620"},{"id":382197500543161,"leftValue":"泰国_IOS_首贷_纯新户_活体检测分_Meta_20240620"},{"id":382197500543194,"leftValue":"泰国_IOS_首贷_纯新户_人脸相似度_Meta_20240620"},{"id":382197500543217,"leftValue":"泰国_IOS_首贷_纯新户_年龄_Meta_20240620"},{"id":382197500543234,"leftValue":"泰国_IOS_首贷_纯新户_人脸黑名单_Meta_20240620"},{"id":382197500543247,"leftValue":"泰国_IOS_首贷_纯新户_外部黑名单_Meta_20240620"},{"id":382197500543260,"leftValue":"泰国_IOS_首贷_纯新户_云服务黑名单_Meta_20240620"},{"id":393600954204183,"leftValue":"ocr年龄无法识别"},{"id":405893862588439,"leftValue":"客服人员准入"},{"id":382197502640260,"leftValue":"泰国_IOS_首贷_非正常安装程序_Meta_20240620"},{"id":382197502640273,"leftValue":"泰国_IOS_首贷_越狱_Meta_20240620"},{"id":423833072762884,"leftValue":"电销包撸贷"},{"id":382197502640288,"leftValue":"泰国_IOS_首贷_纯新户_历史拒绝次数_Meta_20240621"},{"id":382197502640303,"leftValue":"泰国_IOS_首贷_纯新户_onelink_Meta_20240621"},{"id":382197502640318,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同姓名"},{"id":382197502640344,"leftValue":"泰国_IOS_首贷_纯新户_同银行卡不同姓名"},{"id":382197502640357,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同姓名"},{"id":382197502640370,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同line"},{"id":382197502640383,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同银行卡"},{"id":382197502640396,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同银行卡"},{"id":382197502640411,"leftValue":"泰国_IOS_首贷_纯新户_通讯录相似度_Meta_20240621"},{"id":382197502640424,"leftValue":"泰国_IOS_首贷_纯新户_APP相似度_Meta_20240621"},{"id":382197502640439,"leftValue":"泰国_IOS_首贷_纯新户_同人脸不同手机号_Meta_20240621"},{"id":405023737446534,"leftValue":"电销欺诈"},{"id":382197500543604,"leftValue":"泰国_IOS_首贷_纯新户_当前在途订单_Meta_20240620"},{"id":421324602146833,"leftValue":"泰国_IOS_首贷_纯新户_交易前验证_当前在途订单"},{"id":418788321198102,"leftValue":"泰国_IOS_首贷_纯新户_当前在途金额"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25070211RFoxqIje","extraInfo":{"_observe_mode":[false],"_cost_time":229,"ruleIdCodeNamePair":"[{\\"code\\":\\"rule2\\",\\"id\\":406482220679187,\\"name\\":\\"购物券包\\"},{\\"code\\":\\"NBev01\\",\\"id\\":382197502640288,\\"name\\":\\"泰国_IOS_首贷_纯新户_历史拒绝次数_Meta_20240621\\"}]","riskId":"992247b1-90f3-4fa1-9c15-329fd8b8a878,1751430554232,695bf169aeae77434289c395c8373255"},"hitPolicies":[{"id":382197500543324,"name":"泰国_IOS贷超_前处理_公共变量_赋值"},{"id":382197502640241,"name":"泰国_IOS贷超_硬规则_首贷_纯新户_反欺诈"}],"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":"","th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":"","his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":685,"th_app_last_1d_fin_update_cnt_b2407_meta_20240715":"","th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"","th_ab_model_version_meta_20240628":"2405/2503","reject_days":"30","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":"","th_app_last_7d_food_update_cnt_b2407_meta_20240715":"","bj_ios_model_score":619.7816890864531,"is_error":"0","credit_amt_1":"3000","th_beh_reject_pctb2407_meta_20240717":"","credit_amt_2":"","credit_amt_3":"","Expected_loan_time_1":"1751430595545","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":0,"Expected_loan_time_2":"","Expected_loan_time_3":"","th_app_last_21d_social_update_cnt_meta_20240715":"","lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":5.0,"coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"","th_app_last_1d_biz_update_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_meta_20240619":0,"financial_product_id_1":"000008","th_model_bscore_meta_20240719":"","financial_product_id_3":"","financial_product_id_2":"000008","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":3,"th_app_last_21d_highrisk_update_cnt_meta_20240715":"","th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":0.0,"th_app_loan_earlist_install_days_b2407_meta_20240715":"","risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":"","th_app_last_3m_highrisk_install_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_a2405_meta_20240619":0,"job_monthly_income":"8","credit_amt":"3000","is_advertising":"1","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":"","th_app_loan_latest_install_days_a2405_meta_20240619":9999,"th_app_last_7d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_per_repay_b2407_meta_20240717":"","appliacation_limit":"1","model_type":"2403","th_ios_device_model_jixia_20250321":36,"th_model_bversion_meta_20240719":"","th_app_last_3d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_pct_meta_20240717":"","th_model_score_2_meta_20240619":607.1,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":0.1087,"th_ios_first_model_score_2503_jixia_20250321":597.28,"overdue_detail":"0","model_score":604.15,"th_app_last_30d_pe_update_cnt_meta_20240715":""}}}',
        "risk_level": "REJECT",
        "risk_type": "th_fraud_his_reject_meta_20240621",
        "sn": "EDD20250702112955htBe1n71",
        "user_id": "224a4b43210e46468ca641abb5b777de",
        "verify_time": 1751430603,
    },
    {
        "app_name": "photoPortfolio_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 0,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1752507169,
        "request_data": '{"Settlement_frequency_2":"0","Settlement_frequency_1":"0","Settlement_frequency_4":"0","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"0","diffbank_face_num":"0","register_time_second":"1752507169","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"80","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"tencent","ios_system_version":"18.3.1","r81_60p70p81_b":"0","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"0","borrow_rank":"0","geohash_add_time":"1752543970000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"0","ocr_up_type":"0","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0","call_risk_type":"0","last_financial_product_id_1":"-999","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"-999","last_financial_product_id_3":"-999","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"photoPortfolio_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"0","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jd7v1q0","last_overdue_days_5":"0","last_credit_amt_2":"0","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"0","h5_browserName":"","last_7d_self_same_face_id_cnt":"0","personal_city":"792","client_ip":"49.237.177.241","h5_platform":"","face_similarity_channel":"tencent","geohash_type":"riskengine","last_financial_product_id":"None","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","phone_number":"00000000001865571048323723445","fea_list":"{\\"age_job_monthly_income\\":232,\\"app_3d_fi_pct\\":0.02326,\\"app_60d_fi_pct\\":0.06977,\\"app_9999d_fi_效率_cnt\\":5,\\"bj_ios_model_prob\\":0.271092952988736,\\"bj_ios_model_score\\":619.7816890864531,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_first_loan_model_2506_prob\\":0.34304726123809814,\\"ios_first_loan_model_2506_score\\":618.7476464689448,\\"ios_model\\":36,\\"is_available\\":1,\\"job_monthly_income\\":8}","user_contract_amount_sum":"0","first_borrow_whitelist_in":"0","personal_province":"788","wl_uid":"None","age_revise":"29","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001940265737634947074","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"0","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD20250715084709DJlCG5j2","age":"29","last_credit_amt":"0","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1752544030167","ph_diffzalo_num":"2","h5_hardwareConcurrency":"","last_settlement_order_difference":"-1","last_7d_all_same_face_phone_cnt":"0","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.79378389721704","ip":"49.237.177.241","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"ab1d48bdcbc54f9689943c5f5196e15a","other_contact_phone":"0877551743","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"0","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"0","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"0","if_dk_event":"1","face_similarity":"90.0","borrow_count_1":"0","const_id":"91337CA7-EDE6-4E77-931F-80DF419257E1","liveness_livenessScore":"90","last_contract_amount_10":"0","diffphone_face_num":"0","borrow_create_time":"1733501100008","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"0","last_contract_amount_1":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"0","last_7d_self_same_face_apply_cnt":"0","first_credit_amount_3":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"0","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"0","last_7d_all_same_face_id_cnt":"0","platform_check_baiduai_black":"0","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888251067419077","if_live_detection":"1","automatic_face_pass":"0","ios_app_version":"1.0.2","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"0","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"91337CA7-EDE6-4E77-931F-80DF419257E1","h5_language":"th-TH","h5_productSub":"","h5_appName":"Photo portfolio","personal_education":"7","ios_app_package":"tool.photoportfolio.th","ios_battery_percentage":"0.15","borrowing_loan_id":""}',
        "response_data": '{"uuid":"4784cdb3-7cc0-4366-b80e-f7cc01e43642","status":"SUCCESS","ctuEntireResult":{"riskLevel":"REJECT","riskType":"th_fraud_his_reject_meta_20240621","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197498446007,"leftValue":"异常校验"},{"id":382197500543014,"leftValue":"异常校验"},{"id":411574860709947,"leftValue":"异常校验"},{"id":382197502640796,"leftValue":"异常校验"},{"id":382197500543135,"leftValue":"泰国_IOS_首贷_纯新户_当前逾期人脸_Meta_20240620"},{"id":382197500543148,"leftValue":"泰国_IOS_首贷_纯新户_其他平台逾期数量_Meta_20240620"},{"id":382197500543161,"leftValue":"泰国_IOS_首贷_纯新户_活体检测分_Meta_20240620"},{"id":382197500543194,"leftValue":"泰国_IOS_首贷_纯新户_人脸相似度_Meta_20240620"},{"id":382197500543217,"leftValue":"泰国_IOS_首贷_纯新户_年龄_Meta_20240620"},{"id":382197500543234,"leftValue":"泰国_IOS_首贷_纯新户_人脸黑名单_Meta_20240620"},{"id":382197500543247,"leftValue":"泰国_IOS_首贷_纯新户_外部黑名单_Meta_20240620"},{"id":382197500543260,"leftValue":"泰国_IOS_首贷_纯新户_云服务黑名单_Meta_20240620"},{"id":393600954204183,"leftValue":"ocr年龄无法识别"},{"id":405893862588439,"leftValue":"客服人员准入"},{"id":382197502640260,"leftValue":"泰国_IOS_首贷_非正常安装程序_Meta_20240620"},{"id":382197502640273,"leftValue":"泰国_IOS_首贷_越狱_Meta_20240620"},{"id":423833072762884,"leftValue":"电销包撸贷"},{"id":382197502640288,"leftValue":"泰国_IOS_首贷_纯新户_历史拒绝次数_Meta_20240621"},{"id":382197502640303,"leftValue":"泰国_IOS_首贷_纯新户_onelink_Meta_20240621"},{"id":382197502640318,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同姓名"},{"id":382197502640344,"leftValue":"泰国_IOS_首贷_纯新户_同银行卡不同姓名"},{"id":382197502640357,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同姓名"},{"id":382197502640370,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同line"},{"id":382197502640383,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同银行卡"},{"id":382197502640396,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同银行卡"},{"id":382197502640411,"leftValue":"泰国_IOS_首贷_纯新户_通讯录相似度_Meta_20240621"},{"id":382197502640424,"leftValue":"泰国_IOS_首贷_纯新户_APP相似度_Meta_20240621"},{"id":382197502640439,"leftValue":"泰国_IOS_首贷_纯新户_同人脸不同手机号_Meta_20240621"},{"id":405023737446534,"leftValue":"电销欺诈"},{"id":382197500543604,"leftValue":"泰国_IOS_首贷_纯新户_当前在途订单_Meta_20240620"},{"id":421324602146833,"leftValue":"泰国_IOS_首贷_纯新户_交易前验证_当前在途订单"},{"id":418788321198102,"leftValue":"泰国_IOS_首贷_纯新户_当前在途金额"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25071508G7RcrkNN","extraInfo":{"_observe_mode":[false],"_cost_time":138,"ruleIdCodeNamePair":"[{\\"code\\":\\"NBev01\\",\\"id\\":382197502640288,\\"name\\":\\"泰国_IOS_首贷_纯新户_历史拒绝次数_Meta_20240621\\"}]","riskId":"4784cdb3-7cc0-4366-b80e-f7cc01e43642,1752543821305,4984e4c4f3b6f7a883d2434172029083"},"hitPolicies":[{"id":382197502640241,"name":"泰国_IOS贷超_硬规则_首贷_纯新户_反欺诈"}],"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":"","th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":"","his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":698,"th_app_last_1d_fin_update_cnt_b2407_meta_20240715":"","th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"","th_ab_model_version_meta_20240628":"2405/2503","reject_days":"30","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":"","th_app_last_7d_food_update_cnt_b2407_meta_20240715":"","bj_ios_model_score":619.7816890864531,"is_error":"0","credit_amt_1":"2700","th_beh_reject_pctb2407_meta_20240717":"","credit_amt_2":"","credit_amt_3":"","Expected_loan_time_1":"1752544030167","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":1,"Expected_loan_time_2":"","Expected_loan_time_3":"","th_app_last_21d_social_update_cnt_meta_20240715":"","lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":5.0,"coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"","th_app_last_1d_biz_update_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_meta_20240619":0,"financial_product_id_1":"000002","th_model_bscore_meta_20240719":"","financial_product_id_3":"","financial_product_id_2":"","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":3,"th_app_last_21d_highrisk_update_cnt_meta_20240715":"","th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":0.02326,"th_app_loan_earlist_install_days_b2407_meta_20240715":"","risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":"","th_app_last_3m_highrisk_install_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_a2405_meta_20240619":0,"job_monthly_income":"8","credit_amt":"3000","is_advertising":"1","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":"","th_app_loan_latest_install_days_a2405_meta_20240619":0,"th_app_last_7d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_per_repay_b2407_meta_20240717":"","appliacation_limit":"1","model_type":"2403","th_ios_device_model_jixia_20250321":36,"th_model_bversion_meta_20240719":"","th_app_last_3d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_pct_meta_20240717":"","th_model_score_2_meta_20240619":612.46,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":0.06977,"th_ios_first_model_score_2503_jixia_20250321":603.39,"overdue_detail":"0","model_score":609.77,"th_app_last_30d_pe_update_cnt_meta_20240715":""}}}',
        "risk_level": "REJECT",
        "risk_type": "th_fraud_his_reject_meta_20240621",
        "sn": "EDD20250715084709DJlCG5j2",
        "user_id": "ab1d48bdcbc54f9689943c5f5196e15a",
        "verify_time": 1752544045,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 0,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"0","Settlement_frequency_1":"0","Settlement_frequency_4":"0","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","diffbank_face_num":"0","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.3.1","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","last_7d_self_same_face_bankcard_cnt":"0","geohash_add_time":"1739930770000","ios_device_name":"iPhone","diffphone_id_num":"0","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"0","ocr_up_type":"1","id_expiry_date":"02-08-2030","call_risk_type":"0","id_number_ocr":"00000000001892031979169771522","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jd7xth0","last_overdue_days_5":"0","last_credit_amt_2":"0","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"0","h5_browserName":"","last_7d_self_same_face_id_cnt":"0","personal_city":"792","client_ip":"49.237.15.40","h5_platform":"","geohash_type":"riskengine","last_financial_product_id":"None","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","phone_number":"00000000001865571048323723445","fea_list":"{\\"app_14d_fi_工具_trend\\":0,\\"app_9999d_fi_cnt\\":53,\\"calllog_14d_tag_移动电话_time_pct\\":0,\\"calllog_9999d_frequent_phone_cnt\\":0,\\"sms_14d_cashin_sum\\":0,\\"sms_14d_贷款_phone_pct\\":0,\\"sms_14d_贷款_审核通知_pct\\":0,\\"sms_14d_赌场_cnt\\":0,\\"sms_21d_21d_营销_pct\\":0,\\"sms_21d_贷款_审核通知_pct\\":0,\\"sms_30d_30d_营销_distinct_pct\\":0,\\"sms_30d_phone_cnt\\":0,\\"sms_30d_贷款_all_pct\\":0,\\"sms_7d_7d_贷款_pct\\":0,\\"sms_7d_贷款_phone_pct\\":0,\\"sms_7d_贷款_审核通知_pct\\":0,\\"sms_7d_金融类_cnt\\":0,\\"sms_9999d_营销_cnt\\":0,\\"sms_9999d_营销_distinct_cnt\\":0,\\"sms_9999d_贷款_营销_cnt\\":0,\\"sms_赌场_pct\\":0}","user_contract_amount_sum":"0","first_borrow_whitelist_in":"0","personal_province":"788","wl_uid":"None","age_revise":"29","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"0","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD202502190906118l05toei","age":"29","last_credit_amt":"0","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1739930771580","ph_diffzalo_num":"1","h5_hardwareConcurrency":"","last_settlement_order_difference":"-1","last_7d_all_same_face_phone_cnt":"0","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.79360514632866","ip":"49.237.15.40","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"0","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"0","borrow_count_9":"0","ios_rooted":"false","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"0","const_id":"F705A9C1-6662-40FF-9732-2E30192FBECB","liveness_livenessScore":"99.97","last_contract_amount_10":"0","diffphone_face_num":"0","borrow_create_time":"1733069340009","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"0","last_contract_amount_1":"0","h5_product":"","last_7d_self_same_face_apply_cnt":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"0","last_7d_all_same_face_id_cnt":"0","platform_check_baiduai_black":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.88867237770087","if_live_detection":"1","automatic_face_pass":"0","ios_app_version":"1.0.1","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"0","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"F705A9C1-6662-40FF-9732-2E30192FBECB","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","ios_battery_percentage":"0.25","borrowing_loan_id":""}',
        "response_data": '{"uuid":"76cd7fe8-e142-443f-a4cf-76dc52fcb5a2","status":"SUCCESS","ctuEntireResult":{"riskLevel":"REJECT","riskType":"concatnum,model_score","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":382197498446007,"leftValue":"异常校验"},{"id":382197500543014,"leftValue":"异常校验"},{"id":382197502640796,"leftValue":"异常校验"},{"id":382197500543135,"leftValue":"泰国_IOS_首贷_纯新户_当前逾期人脸_Meta_20240620"},{"id":382197500543148,"leftValue":"泰国_IOS_首贷_纯新户_其他平台逾期数量_Meta_20240620"},{"id":382197500543161,"leftValue":"泰国_IOS_首贷_纯新户_活体检测分_Meta_20240620"},{"id":382197500543194,"leftValue":"泰国_IOS_首贷_纯新户_人脸相似度_Meta_20240620"},{"id":382197500543204,"leftValue":"泰国_IOS_首贷_纯新户_有效联系人数量_Meta_20240620"},{"id":382197500543217,"leftValue":"泰国_IOS_首贷_纯新户_年龄_Meta_20240620"},{"id":382197500543234,"leftValue":"泰国_IOS_首贷_纯新户_人脸黑名单_Meta_20240620"},{"id":382197500543247,"leftValue":"泰国_IOS_首贷_纯新户_外部黑名单_Meta_20240620"},{"id":382197500543260,"leftValue":"泰国_IOS_首贷_纯新户_云服务黑名单_Meta_20240620"},{"id":393600954204183,"leftValue":"ocr年龄无法识别"},{"id":405893862588439,"leftValue":"客服人员准入"},{"id":382197502640260,"leftValue":"泰国_IOS_首贷_非正常安装程序_Meta_20240620"},{"id":382197502640273,"leftValue":"泰国_IOS_首贷_越狱_Meta_20240620"},{"id":382197502640288,"leftValue":"泰国_IOS_首贷_纯新户_历史拒绝次数_Meta_20240621"},{"id":382197502640303,"leftValue":"泰国_IOS_首贷_纯新户_onelink_Meta_20240621"},{"id":382197502640318,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同姓名"},{"id":382197502640344,"leftValue":"泰国_IOS_首贷_纯新户_同银行卡不同姓名"},{"id":382197502640357,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同姓名"},{"id":382197502640370,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同line"},{"id":382197502640383,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同银行卡"},{"id":382197502640396,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同银行卡"},{"id":382197502640411,"leftValue":"泰国_IOS_首贷_纯新户_通讯录相似度_Meta_20240621"},{"id":382197502640424,"leftValue":"泰国_IOS_首贷_纯新户_APP相似度_Meta_20240621"},{"id":382197502640439,"leftValue":"泰国_IOS_首贷_纯新户_同人脸不同手机号_Meta_20240621"},{"id":405023737446534,"leftValue":"电销欺诈"},{"id":382197500543604,"leftValue":"泰国_IOS_首贷_纯新户_当前在途订单_Meta_20240620"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197500543313,"leftValue":"应用1额度赋值"},{"id":382197498446054,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25021909N46EQDxu","extraInfo":{"_observe_mode":[false],"_cost_time":495,"ruleIdCodeNamePair":"[{\\"code\\":\\"NAdmission05\\",\\"id\\":382197500543204,\\"name\\":\\"泰国_IOS_首贷_纯新户_有效联系人数量_Meta_20240620\\"},{\\"code\\":\\"ASSIGNCREDIT\\",\\"id\\":382197500543313,\\"name\\":\\"应用1额度赋值\\"}]","riskId":"76cd7fe8-e142-443f-a4cf-76dc52fcb5a2,1739930956777,665fa28b1d8636114a573094b9cf09fe"},"hitPolicies":[{"id":382197500543117,"name":"泰国_IOS贷超_硬规则_首贷_纯新户_准入"},{"id":382197502640619,"name":"泰国_IOS贷超_软规则_首贷_纯新户_模型分"},{"id":382197500543300,"name":"泰国_IOS贷超_后处理_变量返回_基础变量"}],"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":"","th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":"","his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":552,"th_app_last_1d_fin_update_cnt_b2407_meta_20240715":"","th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"","th_ab_model_version_meta_20240628":"2405","reject_days":"3","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":"","th_app_last_7d_food_update_cnt_b2407_meta_20240715":"","is_error":"0","credit_amt_1":"3000","th_beh_reject_pctb2407_meta_20240717":"","Expected_loan_time_1":"1739930771580","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":0,"th_app_last_21d_social_update_cnt_meta_20240715":"","lastoverdue_day":"0","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"","th_app_last_1d_biz_update_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_meta_20240619":0,"financial_product_id_1":"000002","th_model_bscore_meta_20240719":"","th_app_3m_lowrisk_update_cnt_meta_20240619":3,"th_app_last_21d_highrisk_update_cnt_meta_20240715":"","th_model_version_2_meta_20240619":"2405","th_app_loan_earlist_install_days_b2407_meta_20240715":"","risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":"","th_app_last_3m_highrisk_install_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_a2405_meta_20240619":0,"credit_amt":"3000","is_advertising":"0","th_app_last_30d_install_cnt_meta_20240715":"","th_app_loan_latest_install_days_a2405_meta_20240619":9999,"th_app_last_7d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_per_repay_b2407_meta_20240717":"","appliacation_limit":"1","model_type":"2403","th_model_bversion_meta_20240719":"","th_app_last_3d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_pct_meta_20240717":"","th_model_score_2_meta_20240619":607.1,"th_device_ios_model_meta_20240619":4,"overdue_detail":"0","model_score":604.15,"th_app_last_30d_pe_update_cnt_meta_20240715":""}}}',
        "risk_level": "REJECT",
        "risk_type": "concatnum,model_score",
        "sn": "EDD202502190906118l05toei",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1739930854,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 0,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"0","Settlement_frequency_1":"0","Settlement_frequency_4":"0","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"0","diffbank_face_num":"0","register_time_second":"1739928003","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.3.1","r81_60p70p81_b":"0","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"0","borrow_rank":"0","geohash_add_time":"1752721220000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"0","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"02-08-2030","last81_repaytime_diff_30":"0","call_risk_type":"0","last_financial_product_id_1":"-999","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"-999","last_financial_product_id_3":"-999","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"0","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jd7tyf0","last_overdue_days_5":"0","last_credit_amt_2":"0","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"0","h5_browserName":"","last_7d_self_same_face_id_cnt":"0","personal_city":"792","client_ip":"49.237.42.193","h5_platform":"","face_similarity_channel":"tongdun","geohash_type":"riskengine","last_financial_product_id":"None","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","phone_number":"00000000001865571048323723445","fea_list":"{\\"age_job_monthly_income\\":232,\\"app_3d_fi_pct\\":0.04444,\\"app_60d_fi_pct\\":0.06667,\\"app_9999d_fi_效率_cnt\\":5,\\"bj_ios_model_prob\\":0.271092952988736,\\"bj_ios_model_score\\":619.7816890864531,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_first_loan_model_2506_prob\\":0.37504634261131287,\\"ios_first_loan_model_2506_score\\":614.7336040745106,\\"ios_model\\":36,\\"is_available\\":1,\\"job_monthly_income\\":8}","user_contract_amount_sum":"0","first_borrow_whitelist_in":"0","personal_province":"788","wl_uid":"None","age_revise":"29","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"1","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD20250717100022q85f862o","age":"29","last_credit_amt":"0","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1752721222611","ph_diffzalo_num":"2","h5_hardwareConcurrency":"","last_settlement_order_difference":"-1","last_7d_all_same_face_phone_cnt":"1","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.79367390095855","ip":"49.237.42.193","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"0","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"1","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"0","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"0","const_id":"482C65D9-65C3-46B7-8222-3FAB9FC194FE","liveness_livenessScore":"99.97","last_contract_amount_10":"0","diffphone_face_num":"0","borrow_create_time":"1733501221000","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"0","last_contract_amount_1":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"0","last_7d_self_same_face_apply_cnt":"0","first_credit_amount_3":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"0","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"0","last_7d_all_same_face_id_cnt":"1","platform_check_baiduai_black":"0","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.88836210147709","if_live_detection":"1","automatic_face_pass":"0","ios_app_version":"1.0.2","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"0","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"482C65D9-65C3-46B7-8222-3FAB9FC194FE","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","ios_battery_percentage":"0.55","borrowing_loan_id":""}',
        "response_data": '{"uuid":"1ea63c86-afc9-4750-b2e1-8f6d36b3e41e","status":"SUCCESS","ctuEntireResult":{"riskLevel":"REJECT","riskType":"th_fraud_his_reject_meta_20240621","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197498446007,"leftValue":"异常校验"},{"id":382197500543014,"leftValue":"异常校验"},{"id":411574860709947,"leftValue":"异常校验"},{"id":382197502640796,"leftValue":"异常校验"},{"id":382197500543135,"leftValue":"泰国_IOS_首贷_纯新户_当前逾期人脸_Meta_20240620"},{"id":382197500543148,"leftValue":"泰国_IOS_首贷_纯新户_其他平台逾期数量_Meta_20240620"},{"id":382197500543161,"leftValue":"泰国_IOS_首贷_纯新户_活体检测分_Meta_20240620"},{"id":382197500543194,"leftValue":"泰国_IOS_首贷_纯新户_人脸相似度_Meta_20240620"},{"id":382197500543217,"leftValue":"泰国_IOS_首贷_纯新户_年龄_Meta_20240620"},{"id":382197500543234,"leftValue":"泰国_IOS_首贷_纯新户_人脸黑名单_Meta_20240620"},{"id":382197500543247,"leftValue":"泰国_IOS_首贷_纯新户_外部黑名单_Meta_20240620"},{"id":382197500543260,"leftValue":"泰国_IOS_首贷_纯新户_云服务黑名单_Meta_20240620"},{"id":393600954204183,"leftValue":"ocr年龄无法识别"},{"id":405893862588439,"leftValue":"客服人员准入"},{"id":382197502640260,"leftValue":"泰国_IOS_首贷_非正常安装程序_Meta_20240620"},{"id":382197502640273,"leftValue":"泰国_IOS_首贷_越狱_Meta_20240620"},{"id":423833072762884,"leftValue":"电销包撸贷"},{"id":382197502640288,"leftValue":"泰国_IOS_首贷_纯新户_历史拒绝次数_Meta_20240621"},{"id":382197502640303,"leftValue":"泰国_IOS_首贷_纯新户_onelink_Meta_20240621"},{"id":382197502640318,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同姓名"},{"id":382197502640344,"leftValue":"泰国_IOS_首贷_纯新户_同银行卡不同姓名"},{"id":382197502640357,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同姓名"},{"id":382197502640370,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同line"},{"id":382197502640383,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同银行卡"},{"id":382197502640396,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同银行卡"},{"id":382197502640411,"leftValue":"泰国_IOS_首贷_纯新户_通讯录相似度_Meta_20240621"},{"id":382197502640424,"leftValue":"泰国_IOS_首贷_纯新户_APP相似度_Meta_20240621"},{"id":382197502640439,"leftValue":"泰国_IOS_首贷_纯新户_同人脸不同手机号_Meta_20240621"},{"id":405023737446534,"leftValue":"电销欺诈"},{"id":432743022264339,"leftValue":"中介窝点拒绝"},{"id":382197500543604,"leftValue":"泰国_IOS_首贷_纯新户_当前在途订单_Meta_20240620"},{"id":421324602146833,"leftValue":"泰国_IOS_首贷_纯新户_交易前验证_当前在途订单"},{"id":418788321198102,"leftValue":"泰国_IOS_首贷_纯新户_当前在途金额"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25071710SqyGJAox","extraInfo":{"_observe_mode":[false],"_cost_time":121,"ruleIdCodeNamePair":"[{\\"code\\":\\"NBev01\\",\\"id\\":382197502640288,\\"name\\":\\"泰国_IOS_首贷_纯新户_历史拒绝次数_Meta_20240621\\"}]","riskId":"1ea63c86-afc9-4750-b2e1-8f6d36b3e41e,1752721416791,e4843c5b88784a97a9d00ae4d1c78d17"},"hitPolicies":[{"id":382197502640241,"name":"泰国_IOS贷超_硬规则_首贷_纯新户_反欺诈"}],"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":"","th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":"","his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":700,"th_app_last_1d_fin_update_cnt_b2407_meta_20240715":"","th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"","th_ab_model_version_meta_20240628":"2405/2503","reject_days":"30","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":"","th_app_last_7d_food_update_cnt_b2407_meta_20240715":"","bj_ios_model_score":619.7816890864531,"is_error":"0","credit_amt_1":"2700","th_beh_reject_pctb2407_meta_20240717":"","credit_amt_2":"","credit_amt_3":"","Expected_loan_time_1":"1752721222611","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":0,"Expected_loan_time_2":"","Expected_loan_time_3":"","th_app_last_21d_social_update_cnt_meta_20240715":"","lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":5.0,"coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"","th_app_last_1d_biz_update_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_meta_20240619":0,"financial_product_id_1":"000002","th_model_bscore_meta_20240719":"","financial_product_id_3":"","financial_product_id_2":"","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":3,"th_app_last_21d_highrisk_update_cnt_meta_20240715":"","th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":0.04444,"th_app_loan_earlist_install_days_b2407_meta_20240715":"","risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":"","th_app_last_3m_highrisk_install_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_a2405_meta_20240619":0,"job_monthly_income":"8","credit_amt":"3000","is_advertising":"1","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":"","th_app_loan_latest_install_days_a2405_meta_20240619":9999,"th_app_last_7d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_per_repay_b2407_meta_20240717":"","appliacation_limit":"1","model_type":"2403","th_ios_device_model_jixia_20250321":36,"th_model_bversion_meta_20240719":"","th_app_last_3d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_pct_meta_20240717":"","th_model_score_2_meta_20240619":607.1,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":0.06667,"th_ios_first_model_score_2503_jixia_20250321":599.23,"overdue_detail":"0","model_score":604.15,"th_app_last_30d_pe_update_cnt_meta_20240715":""}}}',
        "risk_level": "REJECT",
        "risk_type": "th_fraud_his_reject_meta_20240621",
        "sn": "EDD20250717100022q85f862o",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1752721236,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"0","Settlement_frequency_1":"0","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"0","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1739928003","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"1.00","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"0","emergency_contact_name":"เบอร์ 2","borrow_rank":"0","geohash_add_time":"1761152804000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"0","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"0","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"02-08-2030","last81_repaytime_diff_30":"15.1402199074","call_risk_type":"0","last_financial_product_id_1":"-999","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"-999","last_financial_product_id_3":"-999","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"0","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w4x6ct8tn90","last_overdue_days_5":"0","other_contact_name":"Chai","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"0","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"0","h5_browserName":"","last_7d_self_same_face_id_cnt":"0","personal_city":"792","client_ip":"49.48.218.185","h5_platform":"","face_similarity_channel":"tongdun","geohash_type":"riskengine","last_financial_product_id":"None","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"0","fea_list":"{\\"age_job_monthly_income\\":240,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.13158,\\"app_9999d_fi_效率_cnt\\":5,\\"bj_ios_model_prob\\":0.271092952988736,\\"bj_ios_model_score\\":619.7816890864531,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_first_loan_model_2506_prob\\":0.29186856746673584,\\"ios_first_loan_model_2506_score\\":625.5739663685529,\\"ios_model\\":36,\\"is_available\\":1,\\"job_monthly_income\\":8,\\"pl_loan_1o_phone_max_amt\\":4500,\\"pl_loan_7o_phone_max_amt\\":6000,\\"pl_loan_9999d_phone_max_amt\\":6000,\\"pl_loan_9999d_user_max_amt\\":0,\\"pl_loan_9999d_user_onway_cnt\\":0,\\"pl_repay_1o_phone_early_latest_hours\\":363.44125,\\"pl_repay_1o_phone_ontime_latest_hours\\":363.44125,\\"pl_repay_2o_phone_early_earliest_hours\\":363.46792,\\"pl_repay_2o_phone_ontime_latest_hours\\":363.44125,\\"pl_repay_30d_phone_ontime_latest_hours\\":363.44125,\\"pl_repay_3o_phone_early_cnt\\":3,\\"pl_repay_3o_phone_s-1_cnt\\":2,\\"pl_repay_4o_phone_ontime_amt_pct\\":1,\\"pl_repay_5o_phone_s-1_cnt\\":4,\\"pl_repay_5o_phone_s-1_pct\\":0.8,\\"pl_repay_60d_phone_early_earliest_hours\\":995.13181,\\"pl_repay_60d_phone_ontime_earliest_hours\\":995.13181,\\"pl_repay_60d_phone_ontime_latest_hours\\":363.44125,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.35046,\\"pl_repay_90d_phone_early_cnt\\":11,\\"pl_repay_90d_user_early_cnt\\":0,\\"pl_repay_9999d_phone_cnt\\":23,\\"pl_repay_9999d_phone_early_cnt\\":11,\\"pure_new_tag\\":0}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"2","if_id_number_modified":"0","whitelist_type":"2","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"0","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD20251023120644V8II57no","age":"30","last_credit_amt":"0","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1761152804872","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"-1","last_7d_all_same_face_phone_cnt":"0","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.2622051026155","another_contact_name":"Manni","ip":"49.48.218.185","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"0","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"0","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"0","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"0","const_id":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","liveness_livenessScore":"99.97","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730300580000","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"0","last_contract_amount_1":"0","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"1,5,6","current_app_overdue_order_cnt":"0","first_credit_amount_2":"0","last_7d_self_same_face_apply_cnt":"0","first_credit_amount_3":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"0","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"0","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"0","platform_check_baiduai_black":"0","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"14.576977395630946","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"0","ios_app_version":"1.0.2","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"0","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","ios_battery_percentage":"0.15","borrowing_loan_id":"","emergency_is_others":"0"}',
        "response_data": '{"uuid":"89515dcb-2e20-4594-b5ca-b4fcc129402d","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197498446007,"leftValue":"异常校验"},{"id":382197500543014,"leftValue":"异常校验"},{"id":411574860709947,"leftValue":"异常校验"},{"id":382197502640796,"leftValue":"异常校验"},{"id":382197500543452,"leftValue":"泰国_IOS_首贷_新包老客_活体检测分_Meta_20240621"},{"id":382197500543485,"leftValue":"泰国_IOS_首贷_新包老客_人脸相似度_Meta_20240621"},{"id":382197500543495,"leftValue":"泰国_IOS_首贷_新包老客_其他平台当前逾期数量_Meta_20240621"},{"id":382197500543508,"leftValue":"泰国_IOS_首贷_新包老客_当前逾期人脸_Meta_20240620"},{"id":382197500543521,"leftValue":"泰国_IOS_首贷_新包老客_年龄准入_Meta_20240620"},{"id":382197500543538,"leftValue":"泰国_IOS_首贷_新包老客_外部黑名单_Meta_20240621"},{"id":382197500543551,"leftValue":"泰国_IOS_首贷_新包老客_云服务黑名单_Meta_20240621"},{"id":389414262866011,"leftValue":"泰国_IOS_首贷_新包老客_人脸黑名单_Meta_20241111"},{"id":393581725417533,"leftValue":"ocr无法识别年龄"},{"id":436544793935884,"leftValue":"电销包新包老客拒绝"},{"id":382197498446159,"leftValue":"泰国_IOS_首贷_新包老客_非正常安装程序_Meta_20240620"},{"id":382197498446172,"leftValue":"泰国_IOS_首贷_新包老客_越狱_Meta_20240620"},{"id":405023739543570,"leftValue":"电销欺诈"},{"id":432743791919143,"leftValue":"中介窝点风险"},{"id":433108941733894,"leftValue":"近7天同geohash不同手机号数量"},{"id":434209938145287,"leftValue":"泰国_IOS_首贷_纯新户_APP相似度"},{"id":434209938145299,"leftValue":"泰国_IOS_首贷_纯新户_通讯录相似度"},{"id":434209938145311,"leftValue":"泰国_IOS_首贷_纯新户_人脸聚集风险"},{"id":434209938145323,"leftValue":"泰国_IOS_首贷_纯新户_同银行卡不同姓名"},{"id":434209938145335,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同姓名"},{"id":382197496348689,"leftValue":"泰国_IOS_首贷_新包老客_在途订单_Meta_20240620"},{"id":418788323295235,"leftValue":"泰国_IOS_首贷_新包老客_在途金额"},{"id":432349613326340,"leftValue":"泰国_IOS_首贷_新包老客_在途订单"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25102312or1Q8K66","extraInfo":{"_observe_mode":[false],"_cost_time":171,"ruleIdCodeNamePair":"[{\\"code\\":\\"rule1\\",\\"id\\":405047798071317,\\"name\\":\\"白名单_投放返回\\"}]","riskId":"89515dcb-2e20-4594-b5ca-b4fcc129402d,1761153048806,a7daa4298367357bd51cbb16e85bdb3d"},"hitPolicies":[{"id":382197500543324,"name":"泰国_IOS贷超_前处理_公共变量_赋值"}],"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":"","th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":"","his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":606,"th_app_last_1d_fin_update_cnt_b2407_meta_20240715":"","th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":"","th_app_last_7d_food_update_cnt_b2407_meta_20240715":"","bj_ios_model_score":619.7816890864531,"is_error":"0","credit_amt_1":"4300","th_beh_reject_pctb2407_meta_20240717":"","credit_amt_2":"","credit_amt_3":"","Expected_loan_time_1":"1761152804872","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":0,"Expected_loan_time_2":"","Expected_loan_time_3":"","th_app_last_21d_social_update_cnt_meta_20240715":"","lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":5.0,"coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"","th_app_last_1d_biz_update_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_meta_20240619":1,"financial_product_id_1":"000002","th_model_bscore_meta_20240719":"","financial_product_id_3":"","financial_product_id_2":"","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":2,"th_app_last_21d_highrisk_update_cnt_meta_20240715":"","th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":0.0,"th_app_loan_earlist_install_days_b2407_meta_20240715":"","risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":"","th_app_last_3m_highrisk_install_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_a2405_meta_20240619":0,"job_monthly_income":"8","th_externel_onway_cnt":"0","credit_amt":"3000","is_advertising":"1","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":"","th_app_loan_latest_install_days_a2405_meta_20240619":9999,"th_app_last_7d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_per_repay_b2407_meta_20240717":"","appliacation_limit":"1","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":36,"th_model_bversion_meta_20240719":"","th_app_last_3d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_pct_meta_20240717":"","th_model_score_2_meta_20240619":607.1,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":0.13158,"th_ios_first_model_score_2503_jixia_20250321":597.28,"ABtest":"","overdue_detail":"0","model_score":590.17,"th_app_last_30d_pe_update_cnt_meta_20240715":""}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251023120644V8II57no",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1761152818,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 2,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"0","Settlement_frequency_1":"1","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1739928003","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.85","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"เบอร์ 2","borrow_rank":"1","geohash_add_time":"1761789121000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"1","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"23600.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"02-08-2030","last81_repaytime_diff_30":"2.1853935185","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"-999","last_financial_product_id_3":"-999","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"4.3677777778","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"-1","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jde91t0","last_overdue_days_5":"0","other_contact_name":"Chai","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"0","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"5600","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.73.144","h5_platform":"","face_similarity_channel":"tongdun","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"23600.00","fea_list":"{\\"age_job_monthly_income\\":240,\\"app_3d_fi_pct\\":0.02564,\\"app_60d_fi_pct\\":0.15385,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_model\\":36,\\"job_monthly_income\\":8,\\"pl_loan_1o_phone_max_amt\\":8200,\\"pl_loan_5o_user_avg_wait_hours\\":44.68778,\\"pl_loan_7o_phone_max_amt\\":8200,\\"pl_loan_9999d_phone_max_amt\\":8200,\\"pl_loan_9999d_user_max_amt\\":5600,\\"pl_loan_9999d_user_onway_cnt\\":1,\\"pl_repay_30d_phone_ontime_latest_hours\\":122.23386,\\"pl_repay_30d_user_ontime_latest_hours\\":52.58247,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":1179.7833,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1179.7833,\\"pl_repay_60d_phone_ontime_latest_hours\\":122.23386,\\"pl_repay_60d_user_latest_overdue_days\\":-1,\\"pl_repay_60d_user_ontime_earliest_hours\\":52.58247,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.19033,\\"pl_repay_90d_phone_early_cnt\\":15,\\"pl_repay_90d_user_early_cnt\\":1,\\"pl_repay_9999d_phone_cnt\\":27,\\"pl_repay_9999d_phone_early_cnt\\":15,\\"pl_repay_9999d_user_latest_overdue_days\\":-1,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.29321222294656013,\\"th_ios_bcard_2508_score\\":589.0658936012092}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD202510300445535T1fhlpT","age":"30","last_credit_amt":"5600","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1761817553783","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"2.19","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.79345287790314","another_contact_name":"Manni","ip":"49.237.73.144","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"0","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"10","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"1","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"1","const_id":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","liveness_livenessScore":"99.97","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730296801006","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"0","last_contract_amount_1":"5600.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"0","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"4300.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"1","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888935311883493","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"0","ios_app_version":"1.0.2","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","ios_battery_percentage":"0.4","borrowing_loan_id":"1","emergency_is_others":"0"}',
        "response_data": '{"uuid":"9c1f9fea-7033-4989-9c8a-dd6a559eedda","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25103004G8452dFD","extraInfo":{"_cost_time":221,"riskId":"9c1f9fea-7033-4989-9c8a-dd6a559eedda,1761817797775,58680994fc754e3d588e89f8a97f57cd"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":1680.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"21","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":1,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":6400.0,"th_beh_reject_pctb2407_meta_20240717":"0.2963","credit_amt_2":800.0,"credit_amt_3":"500","Expected_loan_time_1":"1761817553783","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1761817553783","Expected_loan_time_3":"1761817553783","th_app_last_21d_social_update_cnt_meta_20240715":7,"lastoverdue_day":"-1","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.5000","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.8462","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":618.4482944660313,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":1,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":3,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"8","th_externel_onway_cnt":"4","credit_amt":"5600","is_advertising":"0","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":4,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.8462","th_model_score_2_meta_20240619":607.79,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.97,"ABtest":"实验组","overdue_detail":"0","model_score":576.08,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD202510300445535T1fhlpT",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1761817563,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"0","Settlement_frequency_1":"1","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"0","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1739928003","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.92","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"เบอร์ 2","borrow_rank":"1","geohash_add_time":"1761628288000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"1","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"9800.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"02-08-2030","last81_repaytime_diff_30":"1.8469328704","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"-999","last_financial_product_id_3":"-999","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"4.0293171296","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"-1","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jde91t0","last_overdue_days_5":"0","other_contact_name":"Chai","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"0","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"4300","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.91.222","h5_platform":"","face_similarity_channel":"tongdun","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"9800.00","fea_list":"{\\"age_job_monthly_income\\":240,\\"app_3d_fi_pct\\":0.02564,\\"app_60d_fi_pct\\":0.15385,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_model\\":36,\\"job_monthly_income\\":8,\\"pl_loan_1o_phone_max_amt\\":4800,\\"pl_loan_5o_user_avg_wait_hours\\":0,\\"pl_loan_7o_phone_max_amt\\":6700,\\"pl_loan_9999d_phone_max_amt\\":6700,\\"pl_loan_9999d_user_max_amt\\":4300,\\"pl_loan_9999d_user_onway_cnt\\":0,\\"pl_repay_30d_phone_ontime_latest_hours\\":114.11061,\\"pl_repay_30d_user_ontime_latest_hours\\":44.45922,\\"pl_repay_3o_phone_early_cnt\\":1,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.40964,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.33333,\\"pl_repay_60d_phone_early_earliest_hours\\":1171.66006,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1171.66006,\\"pl_repay_60d_phone_ontime_latest_hours\\":114.11061,\\"pl_repay_60d_user_latest_overdue_days\\":-1,\\"pl_repay_60d_user_ontime_earliest_hours\\":44.45922,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.31439,\\"pl_repay_90d_phone_early_cnt\\":15,\\"pl_repay_90d_user_early_cnt\\":1,\\"pl_repay_9999d_phone_cnt\\":27,\\"pl_repay_9999d_phone_early_cnt\\":15,\\"pl_repay_9999d_user_latest_overdue_days\\":-1,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.21352662619176882,\\"th_ios_bcard_2508_score\\":601.2988955478945}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD20251030083830Wng34qRx","age":"30","last_credit_amt":"4300","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1761788310668","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"1.85","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.79345287790314","another_contact_name":"Manni","ip":"49.237.91.222","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"0","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"10","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"1","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"1","const_id":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","liveness_livenessScore":"99.97","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730296800008","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"0","last_contract_amount_1":"4300.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"0","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"4300.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"1","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888935311883493","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"0","ios_app_version":"1.0.2","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","ios_battery_percentage":"0.55","borrowing_loan_id":"","emergency_is_others":"0"}',
        "response_data": '{"uuid":"c75112da-57ea-46be-961a-d0843751ac86","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25103008zqP7h9U4","extraInfo":{"_cost_time":281,"riskId":"c75112da-57ea-46be-961a-d0843751ac86,1761788553881,c86e5286d44957be536d5b309d9c0b26"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":1680.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"15","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":1,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":5600.0,"th_beh_reject_pctb2407_meta_20240717":"0.3200","credit_amt_2":1300.0,"credit_amt_3":"500","Expected_loan_time_1":"1761788310668","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1761788310668","Expected_loan_time_3":"1761788310668","th_app_last_21d_social_update_cnt_meta_20240715":7,"lastoverdue_day":"-1","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6667","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9167","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":619.0012489265642,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":1,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":3,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"8","th_externel_onway_cnt":"2","credit_amt":"4300","is_advertising":"0","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":4,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.9167","th_model_score_2_meta_20240619":607.79,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.97,"ABtest":"实验组","overdue_detail":"0","model_score":576.08,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251030083830Wng34qRx",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1761788319,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"1","Settlement_frequency_1":"1","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"1","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1739928003","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.89","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"เบอร์ 2","borrow_rank":"2","geohash_add_time":"1761925486000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"18800.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"02-08-2030","last81_repaytime_diff_30":"1.3447453704","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"-6","first60_borrowtime_diff":"5.7673958333","last_overdue_days_3":"-6","diffbank_id_num":"0","last_overdue_days_1":"-1","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w4xs1w04fm0","last_overdue_days_5":"0","other_contact_name":"Chai","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"800","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"500","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"5600","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"223.24.196.203","h5_platform":"","face_similarity_channel":"tongdun","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"18800.00","fea_list":"{\\"age_job_monthly_income\\":240,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.12821,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_model\\":36,\\"job_monthly_income\\":8,\\"pl_loan_1o_phone_max_amt\\":500,\\"pl_loan_5o_user_avg_wait_hours\\":49.95574,\\"pl_loan_7o_phone_max_amt\\":8200,\\"pl_loan_9999d_phone_max_amt\\":8200,\\"pl_loan_9999d_user_max_amt\\":5600,\\"pl_loan_9999d_user_onway_cnt\\":1,\\"pl_repay_1o_phone_early_latest_hours\\":32.3059,\\"pl_repay_1o_phone_ontime_latest_hours\\":32.3059,\\"pl_repay_2o_phone_early_earliest_hours\\":32.30646,\\"pl_repay_2o_phone_ontime_latest_hours\\":32.3059,\\"pl_repay_30d_phone_ontime_latest_hours\\":32.3059,\\"pl_repay_30d_user_ontime_latest_hours\\":32.3059,\\"pl_repay_3o_phone_early_cnt\\":2,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.08609,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.33333,\\"pl_repay_60d_phone_early_earliest_hours\\":1213.37396,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1213.37396,\\"pl_repay_60d_phone_ontime_latest_hours\\":32.3059,\\"pl_repay_60d_user_latest_overdue_days\\":-6,\\"pl_repay_60d_user_ontime_earliest_hours\\":86.17312,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.12203,\\"pl_repay_90d_phone_early_cnt\\":18,\\"pl_repay_90d_user_early_cnt\\":3,\\"pl_repay_9999d_phone_cnt\\":30,\\"pl_repay_9999d_phone_early_cnt\\":18,\\"pl_repay_9999d_user_latest_overdue_days\\":-6,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.2392709537438736,\\"th_ios_bcard_2508_score\\":597.0540050864837}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD20251101022122sHWMYnFM","age":"30","last_credit_amt":"800","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1761938482917","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"1.35","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.61289803422177","another_contact_name":"Manni","ip":"223.24.196.203","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"1","borrow_count_3":"1","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"11","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"1","const_id":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","liveness_livenessScore":"99.97","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730394060002","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"800.00","last_contract_amount_1":"5600.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"800.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"4300.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"500.00","borrow_count":"3","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"14.799084120936074","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"0","ios_app_version":"1.0.2","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","ios_battery_percentage":"0.4","borrowing_loan_id":"1","emergency_is_others":"0"}',
        "response_data": '{"uuid":"ebdde190-bddd-4cb6-a070-70912fc3a351","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25110102Za9pabIS","extraInfo":{"_cost_time":234,"riskId":"ebdde190-bddd-4cb6-a070-70912fc3a351,1761938198546,dcdd19084ab5cb8c836c28d0fca2d9f1"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":1680.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":100,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"21","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":2,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":7100.0,"th_beh_reject_pctb2407_meta_20240717":"0.2857","credit_amt_2":2300.0,"credit_amt_3":"500","Expected_loan_time_1":"1761938482917","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1761938482917","Expected_loan_time_3":"1761938482917","th_app_last_21d_social_update_cnt_meta_20240715":8,"lastoverdue_day":"-6","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.7000","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.8929","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":616.8870472519766,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":1,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":3,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"8","th_externel_onway_cnt":"3","credit_amt":"5600","is_advertising":"0","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":4,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.8929","th_model_score_2_meta_20240619":607.79,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.97,"ABtest":"实验组","overdue_detail":"0","model_score":576.08,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251101022122sHWMYnFM",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1761938489,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"2","Settlement_frequency_1":"1","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"1","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1739928003","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.93","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"เบอร์ 2","borrow_rank":"3","geohash_add_time":"1761938558000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"13800.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"02-08-2030","last81_repaytime_diff_30":"0.5534143519","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"-6","first60_borrowtime_diff":"2.6467245370","last_overdue_days_3":"-6","diffbank_id_num":"0","last_overdue_days_1":"-1","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w4x6ct8wsv0","last_overdue_days_5":"0","other_contact_name":"Chai","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"2300","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"500","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"5600","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.180.208","h5_platform":"","face_similarity_channel":"tongdun","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"13800.00","fea_list":"{\\"age_job_monthly_income\\":240,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.12821,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_model\\":36,\\"job_monthly_income\\":8,\\"pl_loan_1o_phone_max_amt\\":2300,\\"pl_loan_5o_user_avg_wait_hours\\":45.54847,\\"pl_loan_7o_phone_max_amt\\":8200,\\"pl_loan_9999d_phone_max_amt\\":8200,\\"pl_loan_9999d_user_max_amt\\":5600,\\"pl_loan_9999d_user_onway_cnt\\":1,\\"pl_repay_1o_phone_early_latest_hours\\":13.28214,\\"pl_repay_1o_phone_ontime_latest_hours\\":13.28214,\\"pl_repay_2o_phone_early_earliest_hours\\":54.11408,\\"pl_repay_2o_phone_ontime_latest_hours\\":13.28214,\\"pl_repay_30d_phone_ontime_latest_hours\\":13.28214,\\"pl_repay_30d_user_ontime_latest_hours\\":13.28214,\\"pl_repay_3o_phone_early_cnt\\":3,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.30508,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":1235.18214,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1235.18214,\\"pl_repay_60d_phone_ontime_latest_hours\\":13.28214,\\"pl_repay_60d_user_latest_overdue_days\\":-6,\\"pl_repay_60d_user_ontime_earliest_hours\\":107.9813,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.09961,\\"pl_repay_90d_phone_early_cnt\\":19,\\"pl_repay_90d_user_early_cnt\\":4,\\"pl_repay_9999d_phone_cnt\\":32,\\"pl_repay_9999d_phone_early_cnt\\":19,\\"pl_repay_9999d_user_latest_overdue_days\\":-6,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.2475551239551334,\\"th_ios_bcard_2508_score\\":595.7559783638322}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"2","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD202511021209510RrnsWT6","age":"30","last_credit_amt":"2300","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1762016991876","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"0.55","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.26212290877643","another_contact_name":"Manni","ip":"49.237.180.208","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"2","borrow_count_3":"1","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"11","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"1","const_id":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","liveness_livenessScore":"99.97","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730394120000","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"2300.00","last_contract_amount_1":"5600.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"800.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"4300.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"500.00","borrow_count":"4","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"14.577253564877806","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"0","ios_app_version":"1.0.2","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","ios_battery_percentage":"0.25","borrowing_loan_id":"1","emergency_is_others":"0"}',
        "response_data": '{"uuid":"db14c452-ed26-4fe3-bdec-068122951ad2","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25110212nVFt64kn","extraInfo":{"_cost_time":227,"riskId":"db14c452-ed26-4fe3-bdec-068122951ad2,1762016918181,a7a879a747cb4d3c0e64ca8794566dad"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":1680.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"21","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":3,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":7100.0,"th_beh_reject_pctb2407_meta_20240717":"0.2759","credit_amt_2":3800.0,"credit_amt_3":2000.0,"Expected_loan_time_1":"1762016991876","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1762016991876","Expected_loan_time_3":"1762016991876","th_app_last_21d_social_update_cnt_meta_20240715":8,"lastoverdue_day":"-6","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.8182","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9310","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":627.78240080034,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":1,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":3,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"8","th_externel_onway_cnt":"2","credit_amt":"5600","is_advertising":"0","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":4,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":1,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":1,"th_beh_ontime_repay_pct_meta_20240717":"0.9310","th_model_score_2_meta_20240619":607.79,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.97,"ABtest":"实验组","overdue_detail":"0","model_score":576.08,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD202511021209510RrnsWT6",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1762016998,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"2","Settlement_frequency_1":"2","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"1","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1739928003","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.91","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"เบอร์ 2","borrow_rank":"4","geohash_add_time":"1762302087000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"11800.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"02-08-2030","last81_repaytime_diff_30":"0.4137037037","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"-6","first60_borrowtime_diff":"3.6656018519","last_overdue_days_3":"-6","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jde99m0","last_overdue_days_5":"0","other_contact_name":"Chai","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"3800","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"2000","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"5600","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.47.123","h5_platform":"","face_similarity_channel":"tongdun","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"11800.00","fea_list":"{\\"age_job_monthly_income\\":240,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.10811,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_model\\":36,\\"job_monthly_income\\":8,\\"pl_loan_1o_phone_max_amt\\":3800,\\"pl_loan_5o_user_avg_wait_hours\\":46.02046,\\"pl_loan_7o_phone_max_amt\\":8200,\\"pl_loan_9999d_phone_max_amt\\":8200,\\"pl_loan_9999d_user_max_amt\\":5600,\\"pl_loan_9999d_user_onway_cnt\\":2,\\"pl_repay_30d_phone_ontime_latest_hours\\":101.25717,\\"pl_repay_30d_user_ontime_latest_hours\\":101.25717,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.16312,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":967.42717,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1036.75634,\\"pl_repay_60d_phone_ontime_latest_hours\\":101.25717,\\"pl_repay_60d_user_latest_overdue_days\\":-6,\\"pl_repay_60d_user_ontime_earliest_hours\\":195.95634,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.03669,\\"pl_repay_90d_phone_early_cnt\\":19,\\"pl_repay_90d_user_early_cnt\\":4,\\"pl_repay_9999d_phone_cnt\\":34,\\"pl_repay_9999d_phone_early_cnt\\":19,\\"pl_repay_9999d_user_latest_overdue_days\\":-6,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.26270730610891263,\\"th_ios_bcard_2508_score\\":593.454879441979}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"2","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD20251105040822BFAW3c03","age":"30","last_credit_amt":"3800","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1762333702460","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"0.41","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.7934379607149","another_contact_name":"Manni","ip":"49.237.47.123","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"2","borrow_count_3":"1","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"2","const_id":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","liveness_livenessScore":"99.97","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730394301006","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"3800.00","last_contract_amount_1":"5600.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"800.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"4300.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"2000.00","borrow_count":"5","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.889020870765773","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"0","ios_app_version":"1.0.2","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","ios_battery_percentage":"0.1","borrowing_loan_id":"3,2","emergency_is_others":"0"}',
        "response_data": '{"uuid":"23f009a4-d1c5-4cfc-95e9-7ee6aa4fbec0","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz251105048PWT97xF","extraInfo":{"_cost_time":1618,"riskId":"23f009a4-d1c5-4cfc-95e9-7ee6aa4fbec0,1762333947097,be9db4c49bcb3bfa555ad9570cfedc2e"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":1680.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"23","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":2,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":7100.0,"th_beh_reject_pctb2407_meta_20240717":"0.2500","credit_amt_2":5300.0,"credit_amt_3":3500.0,"Expected_loan_time_1":"1762333702460","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1762333702460","Expected_loan_time_3":"1762333702460","th_app_last_21d_social_update_cnt_meta_20240715":10,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.7857","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9063","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":621.4626387115192,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":1,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":2,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"8","th_externel_onway_cnt":"3","credit_amt":"5600","is_advertising":"0","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":3,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":1,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.9063","th_model_score_2_meta_20240619":607.79,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.97,"ABtest":"实验组","overdue_detail":"0","model_score":576.08,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251105040822BFAW3c03",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1762333709,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"2","Settlement_frequency_1":"2","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"1","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1739928003","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.94","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"เบอร์ 2","borrow_rank":"4","geohash_add_time":"1762017015000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"8000.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"02-08-2030","last81_repaytime_diff_30":"0.0463194444","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"-6","first60_borrowtime_diff":"3.2982175926","last_overdue_days_3":"-6","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p9qy2mvu0","last_overdue_days_5":"0","other_contact_name":"Chai","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"2300","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"2000","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"5600","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"27.55.83.159","h5_platform":"","face_similarity_channel":"tongdun","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"8000.00","fea_list":"{\\"age_job_monthly_income\\":240,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.12821,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_model\\":36,\\"job_monthly_income\\":8,\\"pl_loan_1o_phone_max_amt\\":6000,\\"pl_loan_5o_user_avg_wait_hours\\":22.80722,\\"pl_loan_7o_phone_max_amt\\":8200,\\"pl_loan_9999d_phone_max_amt\\":8200,\\"pl_loan_9999d_user_max_amt\\":5600,\\"pl_loan_9999d_user_onway_cnt\\":1,\\"pl_repay_30d_phone_ontime_latest_hours\\":92.43991,\\"pl_repay_30d_user_ontime_latest_hours\\":92.43991,\\"pl_repay_3o_phone_early_cnt\\":1,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.25926,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":958.60991,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1027.93907,\\"pl_repay_60d_phone_ontime_latest_hours\\":92.43991,\\"pl_repay_60d_user_latest_overdue_days\\":-6,\\"pl_repay_60d_user_ontime_earliest_hours\\":187.13907,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.03669,\\"pl_repay_90d_phone_early_cnt\\":19,\\"pl_repay_90d_user_early_cnt\\":4,\\"pl_repay_9999d_phone_cnt\\":34,\\"pl_repay_9999d_phone_early_cnt\\":19,\\"pl_repay_9999d_user_latest_overdue_days\\":-6,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.25434955817182314,\\"th_ios_bcard_2508_score\\":594.7129941117787}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD20251105071919i4ys41O9","age":"30","last_credit_amt":"2000","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1762301960186","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"0.05","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.8440977270081","another_contact_name":"Manni","ip":"27.55.83.159","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"2","borrow_count_3":"1","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"2","const_id":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","liveness_livenessScore":"99.97","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730394300007","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"2300.00","last_contract_amount_1":"5600.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"800.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"4300.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"2000.00","borrow_count":"5","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"17.130071681621637","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"0","ios_app_version":"1.0.2","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","ios_battery_percentage":"0.65","borrowing_loan_id":"3","emergency_is_others":"0"}',
        "response_data": '{"uuid":"daf6cf52-2399-4d76-8739-282a5d6c9eaf","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz251105077SQu5zHW","extraInfo":{"_cost_time":235,"riskId":"daf6cf52-2399-4d76-8739-282a5d6c9eaf,1762301673162,18f7da1a971a43964bb759d6d72b926a"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":1680.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"20","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":3,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":7100.0,"th_beh_reject_pctb2407_meta_20240717":"0.2581","credit_amt_2":3800.0,"credit_amt_3":3500.0,"Expected_loan_time_1":"1762301960186","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1762301960186","Expected_loan_time_3":"1762301960186","th_app_last_21d_social_update_cnt_meta_20240715":8,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.8462","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9355","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":631.6716888191936,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":1,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":3,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"8","th_externel_onway_cnt":"2","credit_amt":"5600","is_advertising":"0","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":3,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":1,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.9355","th_model_score_2_meta_20240619":607.79,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.97,"ABtest":"实验组","overdue_detail":"0","model_score":576.08,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251105071919i4ys41O9",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1762301966,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"2","Settlement_frequency_1":"2","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"2","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1739928003","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.86","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"เบอร์ 2","borrow_rank":"5","geohash_add_time":"1762513545000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"32100.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"02-08-2030","last81_repaytime_diff_30":"0.2151736111","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"-6","first60_borrowtime_diff":"3.0336574074","last_overdue_days_3":"-1","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w4zxvnypcu0","last_overdue_days_5":"0","other_contact_name":"Chai","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"3800","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"2000","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"7100","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.47.100","h5_platform":"","face_similarity_channel":"tongdun","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"32100.00","fea_list":"{\\"age_job_monthly_income\\":240,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.08108,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_model\\":36,\\"job_monthly_income\\":8,\\"pl_loan_1o_phone_max_amt\\":6000,\\"pl_loan_5o_user_avg_wait_hours\\":59.83354,\\"pl_loan_7o_phone_max_amt\\":9200,\\"pl_loan_9999d_phone_max_amt\\":9200,\\"pl_loan_9999d_user_max_amt\\":7100,\\"pl_loan_9999d_user_onway_cnt\\":2,\\"pl_repay_30d_phone_ontime_latest_hours\\":5.16428,\\"pl_repay_30d_user_ontime_latest_hours\\":5.16428,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":1017.38511,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1086.71428,\\"pl_repay_60d_phone_ontime_latest_hours\\":5.16428,\\"pl_repay_60d_user_latest_overdue_days\\":-1,\\"pl_repay_60d_user_ontime_earliest_hours\\":245.91428,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.04294,\\"pl_repay_90d_phone_early_cnt\\":20,\\"pl_repay_90d_user_early_cnt\\":5,\\"pl_repay_9999d_phone_cnt\\":35,\\"pl_repay_9999d_phone_early_cnt\\":20,\\"pl_repay_9999d_user_latest_overdue_days\\":-1,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.3159384025043967,\\"th_ios_bcard_2508_score\\":585.9689153508673}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD20251107060550zCpl1Bj3","age":"30","last_credit_amt":"7100","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1762513551208","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"0.22","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.77492424626953","another_contact_name":"Manni","ip":"49.237.47.100","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"2","borrow_count_3":"2","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"2","const_id":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","liveness_livenessScore":"99.97","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730394421008","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"3800.00","last_contract_amount_1":"7100.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"800.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"4300.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"2000.00","borrow_count":"6","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.869485930921414","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"0","ios_app_version":"1.0.2","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"51DE823B-F9CA-436B-BB3D-FEC9978068A5","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","ios_battery_percentage":"0.05","borrowing_loan_id":"1,2","emergency_is_others":"0"}',
        "response_data": '{"uuid":"491cb34a-2ead-4c8e-bfd3-1b13033c9c06","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25110706P2g5Uxtk","extraInfo":{"_cost_time":252,"riskId":"491cb34a-2ead-4c8e-bfd3-1b13033c9c06,1762513796421,a8b1f05b314d27f8409ae03ace43e547"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":1680.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":100,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"25","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":1,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":8400.0,"th_beh_reject_pctb2407_meta_20240717":"0.2353","credit_amt_2":5100.0,"credit_amt_3":3300.0,"Expected_loan_time_1":"1762513551208","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1762513551208","Expected_loan_time_3":"1762513551208","th_app_last_21d_social_update_cnt_meta_20240715":11,"lastoverdue_day":"-1","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6875","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.8571","th_app_last_1d_biz_update_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":617.5205038570502,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":1,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":2,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"8","th_externel_onway_cnt":"5","credit_amt":"7100","is_advertising":"0","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":2,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":1,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.8571","th_model_score_2_meta_20240619":607.79,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.97,"ABtest":"实验组","overdue_detail":"0","model_score":576.08,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251107060550zCpl1Bj3",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1762513557,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 3,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"3","Settlement_frequency_1":"3","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"3","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"0","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1739928003","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"1.00","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"เบอร์ 2","borrow_rank":"6","geohash_add_time":"","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"0","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"02-08-2030","last81_repaytime_diff_30":"13.8995023148","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"0","last_overdue_days_3":"-1","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"","last_overdue_days_5":"0","other_contact_name":"Chai","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"3800","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"3300","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"7100","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.177.71","h5_platform":"","face_similarity_channel":"tongdun","geohash_type":"","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"0","fea_list":"{\\"age_job_monthly_income\\":240,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.09677,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_model\\":36,\\"job_monthly_income\\":8,\\"pl_loan_1o_phone_max_amt\\":3300,\\"pl_loan_5o_user_avg_wait_hours\\":53.04458,\\"pl_loan_7o_phone_max_amt\\":9200,\\"pl_loan_9999d_phone_max_amt\\":9200,\\"pl_loan_9999d_user_max_amt\\":7100,\\"pl_loan_9999d_user_onway_cnt\\":0,\\"pl_repay_1o_phone_early_latest_hours\\":333.58816,\\"pl_repay_1o_phone_ontime_latest_hours\\":333.58816,\\"pl_repay_2o_phone_early_earliest_hours\\":333.58816,\\"pl_repay_2o_phone_ontime_latest_hours\\":333.58816,\\"pl_repay_30d_phone_ontime_latest_hours\\":333.58816,\\"pl_repay_30d_user_ontime_latest_hours\\":333.58816,\\"pl_repay_3o_phone_early_cnt\\":1,\\"pl_repay_3o_phone_s-1_cnt\\":1,\\"pl_repay_4o_phone_ontime_amt_pct\\":1,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.2,\\"pl_repay_60d_phone_early_earliest_hours\\":1404.04122,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1404.04122,\\"pl_repay_60d_phone_ontime_latest_hours\\":333.58816,\\"pl_repay_60d_user_latest_overdue_days\\":-1,\\"pl_repay_60d_user_ontime_earliest_hours\\":697.35427,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.03897,\\"pl_repay_90d_phone_early_cnt\\":21,\\"pl_repay_90d_user_early_cnt\\":6,\\"pl_repay_9999d_phone_cnt\\":41,\\"pl_repay_9999d_phone_early_cnt\\":21,\\"pl_repay_9999d_user_latest_overdue_days\\":-1,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.21084213855305223,\\"th_ios_bcard_2508_score\\":601.7622705438331}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD202511260132149ToYiQO5","age":"30","last_credit_amt":"3300","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1764138734895","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"13.90","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"","another_contact_name":"Manni","ip":"49.237.177.71","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"3","borrow_count_3":"3","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"3","const_id":"7885C220-A19B-41C6-A0A3-EE13859809A8","liveness_livenessScore":"99.97","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730387161003","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"3800.00","last_contract_amount_1":"7100.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"5,6","current_app_overdue_order_cnt":"0","first_credit_amount_2":"800.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"4300.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"3300.00","borrow_count":"9","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"1","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"0","ios_app_version":"1.0.2","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"7885C220-A19B-41C6-A0A3-EE13859809A8","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","last_overdue_days":"-1","ios_battery_percentage":"0.3","borrowing_loan_id":"","emergency_is_others":"0"}',
        "response_data": '{"uuid":"8c8e738b-9bc0-461f-af9a-7104ecae57a6","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":456658654265348,"leftValue":"未开启放款包拒绝"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz251126015PzLp97F","extraInfo":{"_cost_time":199,"riskId":"8c8e738b-9bc0-461f-af9a-7104ecae57a6,1764138653624,f5b2f209fcbe8cea65e9dfb29182ef82"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":1680.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"24","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":8700.0,"th_beh_reject_pctb2407_meta_20240717":"0.2286","credit_amt_2":5400.0,"credit_amt_3":4900.0,"Expected_loan_time_1":"1764138734895","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1764138734895","Expected_loan_time_3":"1764138734895","th_app_last_21d_social_update_cnt_meta_20240715":16,"lastoverdue_day":"-1","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.0000","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9722","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":620.119513685833,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":2,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":1,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"8","th_externel_onway_cnt":"0","credit_amt":"7100","is_advertising":"0","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":1,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"0.9722","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.9722","th_model_score_2_meta_20240619":607.79,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.97,"ABtest":"实验组","overdue_detail":"0","model_score":576.08,"th_app_last_30d_pe_update_cnt_meta_20240715":0}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD202511260132149ToYiQO5",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1764138741,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"3","Settlement_frequency_1":"3","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"4","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1739928003","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.88","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"เบอร์ 2","borrow_rank":"7","geohash_add_time":"1764138771000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"38300.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"02-08-2030","last81_repaytime_diff_30":"1.4320601852","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"1.7964930556","last_overdue_days_3":"-6","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jd7dsg0","last_overdue_days_5":"0","other_contact_name":"Chai","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"5400","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"4900","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"8700","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.178.149","h5_platform":"","face_similarity_channel":"tongdun","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"38300.00","fea_list":"{\\"age_job_monthly_income\\":240,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.09677,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_model\\":36,\\"job_monthly_income\\":8,\\"pl_loan_1o_phone_max_amt\\":7000,\\"pl_loan_5o_user_avg_wait_hours\\":333.63167,\\"pl_loan_7o_phone_max_amt\\":10200,\\"pl_loan_9999d_phone_max_amt\\":10200,\\"pl_loan_9999d_user_max_amt\\":8700,\\"pl_loan_9999d_user_onway_cnt\\":2,\\"pl_repay_30d_phone_ontime_latest_hours\\":34.36955,\\"pl_repay_30d_user_ontime_latest_hours\\":34.36955,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.16838,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":1245.20455,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1245.23817,\\"pl_repay_60d_phone_ontime_latest_hours\\":34.36955,\\"pl_repay_60d_user_latest_overdue_days\\":-6,\\"pl_repay_60d_user_ontime_earliest_hours\\":740.47067,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.05819,\\"pl_repay_90d_phone_early_cnt\\":22,\\"pl_repay_90d_user_early_cnt\\":7,\\"pl_repay_9999d_phone_cnt\\":42,\\"pl_repay_9999d_phone_early_cnt\\":22,\\"pl_repay_9999d_user_latest_overdue_days\\":-6,\\"pure_new_tag\\":1,\\"th_ios_bcard_2508_prob\\":0.24554765649301266,\\"th_ios_bcard_2508_score\\":596.0677910846991}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD20251128083910uhzeE8X2","age":"30","last_credit_amt":"8700","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1764293951045","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"1.43","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.79358743232571","another_contact_name":"Manni","ip":"49.237.178.149","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"3","borrow_count_3":"4","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"3","const_id":"7885C220-A19B-41C6-A0A3-EE13859809A8","liveness_livenessScore":"99.97","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730387280008","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"5400.00","last_contract_amount_1":"8700.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"800.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"4300.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"4900.00","borrow_count":"10","geo_hash_30d_100meter_overdue_user_cnt":"1","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"1","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.887805755240052","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"0","ios_app_version":"1.0.2","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"7885C220-A19B-41C6-A0A3-EE13859809A8","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","last_overdue_days":"-6","ios_battery_percentage":"0.9","borrowing_loan_id":"1,2","emergency_is_others":"0"}',
        "response_data": '{"uuid":"00bd4385-ea28-4b49-bab1-a143a25407f4","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":456658654265348,"leftValue":"未开启放款包拒绝"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz251128088c41BLKP","extraInfo":{"_cost_time":241,"riskId":"00bd4385-ea28-4b49-bab1-a143a25407f4,1764293651932,ef1c8c38bfee6e3ac542a0a316868022"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":1680.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"30","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":10300.0,"th_beh_reject_pctb2407_meta_20240717":"0.2162","credit_amt_2":7000.0,"credit_amt_3":6500.0,"Expected_loan_time_1":"1764293951045","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1764293951045","Expected_loan_time_3":"1764293951045","th_app_last_21d_social_update_cnt_meta_20240715":16,"lastoverdue_day":"-6","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.1667","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.8571","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":615.365986218884,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":2,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":1,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"8","th_externel_onway_cnt":"5","credit_amt":"8700","is_advertising":"0","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":1,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"0.9730","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.8571","th_model_score_2_meta_20240619":607.79,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.97,"ABtest":"实验组","overdue_detail":"0","model_score":576.08,"th_app_last_30d_pe_update_cnt_meta_20240715":0}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251128083910uhzeE8X2",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1764293960,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"4","Settlement_frequency_1":"4","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"4","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1739928003","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.93","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"เบอร์ 2","borrow_rank":"8","geohash_add_time":"1764635576000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"23300.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"02-08-2030","last81_repaytime_diff_30":"0.2863657407","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"-1","first60_borrowtime_diff":"4.3365625000","last_overdue_days_3":"-6","diffbank_id_num":"0","last_overdue_days_1":"-1","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jd7wpp0","last_overdue_days_5":"0","other_contact_name":"Chai","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"5400","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"6500","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"8700","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.93.65","h5_platform":"","face_similarity_channel":"tongdun","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"23300.00","fea_list":"{\\"age_job_monthly_income\\":240,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.06667,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_model\\":36,\\"job_monthly_income\\":8,\\"pl_loan_1o_phone_max_amt\\":8600,\\"pl_loan_5o_user_avg_wait_hours\\":258.81882,\\"pl_loan_7o_phone_max_amt\\":10200,\\"pl_loan_9999d_phone_max_amt\\":10200,\\"pl_loan_9999d_user_max_amt\\":8700,\\"pl_loan_9999d_user_onway_cnt\\":1,\\"pl_repay_30d_phone_ontime_latest_hours\\":9.2426,\\"pl_repay_30d_user_ontime_latest_hours\\":138.4476,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.23102,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.5,\\"pl_repay_60d_phone_early_earliest_hours\\":1340.05927,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1340.05927,\\"pl_repay_60d_phone_ontime_latest_hours\\":9.2426,\\"pl_repay_60d_user_latest_overdue_days\\":-6,\\"pl_repay_60d_user_ontime_earliest_hours\\":844.54871,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.0664,\\"pl_repay_90d_phone_early_cnt\\":25,\\"pl_repay_90d_user_early_cnt\\":9,\\"pl_repay_9999d_phone_cnt\\":47,\\"pl_repay_9999d_phone_early_cnt\\":25,\\"pl_repay_9999d_user_latest_overdue_days\\":-6,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.2392002412413347,\\"th_ios_bcard_2508_score\\":597.0652155807186}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD20251202044354y4hw86bP","age":"30","last_credit_amt":"6500","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1764668635112","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"1.39","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.79368275789083","another_contact_name":"Manni","ip":"49.237.93.65","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"4","borrow_count_3":"4","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"4","const_id":"7885C220-A19B-41C6-A0A3-EE13859809A8","liveness_livenessScore":"99.97","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730480521006","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"5400.00","last_contract_amount_1":"8700.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"800.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"4300.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"6500.00","borrow_count":"12","geo_hash_30d_100meter_overdue_user_cnt":"1","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"1","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888429181061536","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"0","ios_app_version":"1.0.5","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"7885C220-A19B-41C6-A0A3-EE13859809A8","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","last_overdue_days":"-1","ios_battery_percentage":"0.6","borrowing_loan_id":"3","emergency_is_others":"0"}',
        "response_data": '{"uuid":"09fece90-1e7d-47a2-b200-f4eb5af614ac","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":456658654265348,"leftValue":"未开启放款包拒绝"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz251202042R66r7iZ","extraInfo":{"_cost_time":184,"riskId":"09fece90-1e7d-47a2-b200-f4eb5af614ac,1764668330293,dc84290b41b241b96cba830a4fea5a65"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":1680.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"39","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":10300.0,"th_beh_reject_pctb2407_meta_20240717":"0.1860","credit_amt_2":7000.0,"credit_amt_3":8100.0,"Expected_loan_time_1":"1764668635112","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1764668635112","Expected_loan_time_3":"1764668635112","th_app_last_21d_social_update_cnt_meta_20240715":20,"lastoverdue_day":"-1","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6667","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9111","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":618.5437763734612,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":1,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":1,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"8","th_externel_onway_cnt":"3","credit_amt":"8700","is_advertising":"0","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":1,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"0.9762","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.9111","th_model_score_2_meta_20240619":607.79,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.97,"ABtest":"实验组","overdue_detail":"0","model_score":576.08,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251202044354y4hw86bP",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1764668641,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 0,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"4","Settlement_frequency_1":"4","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"4","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1739928003","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.95","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"เบอร์ 2","borrow_rank":"8","geohash_add_time":"1764548254000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"16700.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"02-08-2030","last81_repaytime_diff_30":"0.0022453704","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"-1","first60_borrowtime_diff":"5.4771412037","last_overdue_days_3":"-6","diffbank_id_num":"0","last_overdue_days_1":"-1","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jde8gp0","last_overdue_days_5":"0","other_contact_name":"Chai","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"5400","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"6500","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"8700","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.84.205","h5_platform":"","face_similarity_channel":"tongdun","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"16700.00","fea_list":"{\\"age_job_monthly_income\\":240,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.06667,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_model\\":36,\\"job_monthly_income\\":8,\\"pl_loan_1o_phone_max_amt\\":6500,\\"pl_loan_5o_user_avg_wait_hours\\":258.81882,\\"pl_loan_7o_phone_max_amt\\":10200,\\"pl_loan_9999d_phone_max_amt\\":10200,\\"pl_loan_9999d_user_max_amt\\":8700,\\"pl_loan_9999d_user_onway_cnt\\":1,\\"pl_repay_2o_phone_ontime_latest_hours\\":0.05397,\\"pl_repay_30d_phone_ontime_latest_hours\\":0.05397,\\"pl_repay_30d_user_ontime_latest_hours\\":129.25897,\\"pl_repay_3o_phone_early_cnt\\":1,\\"pl_repay_3o_phone_s-1_cnt\\":1,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.45603,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.33333,\\"pl_repay_60d_phone_early_earliest_hours\\":1330.87064,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1330.87064,\\"pl_repay_60d_phone_ontime_latest_hours\\":0.05397,\\"pl_repay_60d_user_latest_overdue_days\\":-6,\\"pl_repay_60d_user_ontime_earliest_hours\\":835.36008,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.15384,\\"pl_repay_90d_phone_early_cnt\\":25,\\"pl_repay_90d_user_early_cnt\\":9,\\"pl_repay_9999d_phone_cnt\\":46,\\"pl_repay_9999d_phone_early_cnt\\":25,\\"pl_repay_9999d_user_latest_overdue_days\\":-6,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.21364949417625093,\\"th_ios_bcard_2508_score\\":601.2777890485635}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD20251202073236XdcJddry","age":"30","last_credit_amt":"6500","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1764635556281","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"1.01","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.79351426088307","another_contact_name":"Manni","ip":"49.237.84.205","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"4","borrow_count_3":"4","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"4","const_id":"7885C220-A19B-41C6-A0A3-EE13859809A8","liveness_livenessScore":"99.97","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730480520007","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"5400.00","last_contract_amount_1":"8700.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"800.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"4300.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"6500.00","borrow_count":"12","geo_hash_30d_100meter_overdue_user_cnt":"1","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"1","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888901521484534","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"0","ios_app_version":"1.0.5","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"7885C220-A19B-41C6-A0A3-EE13859809A8","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","last_overdue_days":"-1","ios_battery_percentage":"0.7","borrowing_loan_id":"3","emergency_is_others":"0"}',
        "response_data": '{"uuid":"d3de7a48-69e4-4b21-bbd5-a9b9f896cfd0","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":456658654265348,"leftValue":"未开启放款包拒绝"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25120207C7IOQeq8","extraInfo":{"_cost_time":202,"riskId":"d3de7a48-69e4-4b21-bbd5-a9b9f896cfd0,1764635813196,e53d6245550955d685cb81c92126f612"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":1680.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"27","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":10300.0,"th_beh_reject_pctb2407_meta_20240717":"0.2051","credit_amt_2":7000.0,"credit_amt_3":8100.0,"Expected_loan_time_1":"1764635556281","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1764635556281","Expected_loan_time_3":"1764635556281","th_app_last_21d_social_update_cnt_meta_20240715":20,"lastoverdue_day":"-1","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.7143","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9302","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":623.2973024916181,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":1,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":1,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"8","th_externel_onway_cnt":"2","credit_amt":"8700","is_advertising":"0","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":1,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"0.9756","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.9302","th_model_score_2_meta_20240619":607.79,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.97,"ABtest":"实验组","overdue_detail":"0","model_score":576.08,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251202073236XdcJddry",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1764635561,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"4","Settlement_frequency_1":"4","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"4","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1739928003","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.91","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"เบอร์ 2","borrow_rank":"8","geohash_add_time":"1764668659000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"30300.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"02-08-2030","last81_repaytime_diff_30":"0.4892476852","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"-1","first60_borrowtime_diff":"4.5394444444","last_overdue_days_3":"-6","diffbank_id_num":"0","last_overdue_days_1":"-1","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jde80c0","last_overdue_days_5":"0","other_contact_name":"Chai","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"7000","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"6500","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"8700","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.83.112","h5_platform":"","face_similarity_channel":"tongdun","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"30300.00","fea_list":"{\\"age_job_monthly_income\\":240,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.06667,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_model\\":36,\\"job_monthly_income\\":8,\\"pl_loan_1o_phone_max_amt\\":7000,\\"pl_loan_5o_user_avg_wait_hours\\":86.47139,\\"pl_loan_7o_phone_max_amt\\":10200,\\"pl_loan_9999d_phone_max_amt\\":10200,\\"pl_loan_9999d_user_max_amt\\":8700,\\"pl_loan_9999d_user_onway_cnt\\":2,\\"pl_repay_30d_phone_ontime_latest_hours\\":14.11171,\\"pl_repay_30d_user_ontime_latest_hours\\":143.31671,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":1344.92838,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1344.92838,\\"pl_repay_60d_phone_ontime_latest_hours\\":14.11171,\\"pl_repay_60d_user_latest_overdue_days\\":-6,\\"pl_repay_60d_user_ontime_earliest_hours\\":849.41782,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.03906,\\"pl_repay_90d_phone_early_cnt\\":25,\\"pl_repay_90d_user_early_cnt\\":9,\\"pl_repay_9999d_phone_cnt\\":47,\\"pl_repay_9999d_phone_early_cnt\\":25,\\"pl_repay_9999d_user_latest_overdue_days\\":-6,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.27528411939258707,\\"th_ios_bcard_2508_score\\":591.609139141058}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"2","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD20251202093602f90FM9L5","age":"30","last_credit_amt":"7000","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1764686162890","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"1.60","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.79341866488873","another_contact_name":"Manni","ip":"49.237.83.112","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"4","borrow_count_3":"4","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"4","const_id":"7885C220-A19B-41C6-A0A3-EE13859809A8","liveness_livenessScore":"99.97","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730480522001","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"7000.00","last_contract_amount_1":"8700.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"800.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"4300.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"6500.00","borrow_count":"12","geo_hash_30d_100meter_overdue_user_cnt":"1","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"1","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888741769998145","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"0","ios_app_version":"1.0.5","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"7885C220-A19B-41C6-A0A3-EE13859809A8","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","last_overdue_days":"-1","ios_battery_percentage":"0.1","borrowing_loan_id":"3,2","emergency_is_others":"0"}',
        "response_data": '{"uuid":"13ea04f4-e929-4043-849e-656eb090449a","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":456658654265348,"leftValue":"未开启放款包拒绝"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25120209aWN8bfv3","extraInfo":{"_cost_time":211,"riskId":"13ea04f4-e929-4043-849e-656eb090449a,1764686080315,13e65783c9529825c5f665da1086902d"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":1680.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"42","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":9900.0,"th_beh_reject_pctb2407_meta_20240717":"0.1818","credit_amt_2":8200.0,"credit_amt_3":7700.0,"Expected_loan_time_1":"1764686162890","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1764686162890","Expected_loan_time_3":"1764686162890","th_app_last_21d_social_update_cnt_meta_20240715":20,"lastoverdue_day":"-1","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6000","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.8913","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":618.5437763734612,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":1,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":1,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"8","th_externel_onway_cnt":"4","credit_amt":"8700","is_advertising":"0","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":1,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"0.9762","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.8913","th_model_score_2_meta_20240619":607.79,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.97,"ABtest":"实验组","overdue_detail":"0","model_score":576.08,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251202093602f90FM9L5",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1764686169,
    },
    {
        "app_name": "radonmoney_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1739928003,
        "request_data": '{"Settlement_frequency_2":"4","Settlement_frequency_1":"4","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"0","Settlement_frequency_3":"5","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"0","diffbank_face_num":"0","register_time_second":"1739928003","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"82","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.90","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"เบอร์ 2","borrow_rank":"9","geohash_add_time":"1764686510000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"44900.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"02-08-2030","last81_repaytime_diff_30":"0.4076736111","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"APM money_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"-1","first60_borrowtime_diff":"2.3678472222","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"-1","unique_identifier_calendar":"","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jd7sj70","last_overdue_days_5":"0","other_contact_name":"Chai","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"7000","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"6500","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"9900","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.87.74","h5_platform":"","face_similarity_channel":"tongdun","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"44900.00","fea_list":"{\\"age_job_monthly_income\\":240,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.06667,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":8,\\"industry\\":1,\\"ios_model\\":36,\\"job_monthly_income\\":8,\\"pl_loan_1o_phone_max_amt\\":11200,\\"pl_loan_5o_user_avg_wait_hours\\":105.47565,\\"pl_loan_7o_phone_max_amt\\":11200,\\"pl_loan_9999d_phone_max_amt\\":11200,\\"pl_loan_9999d_user_max_amt\\":9900,\\"pl_loan_9999d_user_onway_cnt\\":2,\\"pl_repay_30d_phone_ontime_latest_hours\\":9.78424,\\"pl_repay_30d_user_ontime_latest_hours\\":9.78424,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":1387.77647,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1387.77647,\\"pl_repay_60d_phone_ontime_latest_hours\\":9.78424,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":892.23924,\\"pl_repay_7o_phone_early_amt_all_pct\\":0,\\"pl_repay_90d_phone_early_cnt\\":24,\\"pl_repay_90d_user_early_cnt\\":9,\\"pl_repay_9999d_phone_cnt\\":48,\\"pl_repay_9999d_phone_early_cnt\\":25,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.30528164222349247,\\"th_ios_bcard_2508_score\\":587.4050043659897}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"0","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"0","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD20251204042520K4J0HycX","age":"30","last_credit_amt":"9900","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1764840320612","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"0.41","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.79361411577143","another_contact_name":"Manni","ip":"49.237.87.74","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"aecefd177b4844ed929d78168698cc2f","other_contact_phone":"0800699457","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"4","borrow_count_3":"5","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"96.0","borrow_count_1":"4","const_id":"7885C220-A19B-41C6-A0A3-EE13859809A8","liveness_livenessScore":"99.97","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730480641006","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"7000.00","last_contract_amount_1":"9900.00","same_calendarid_dif_idcard":"0","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"800.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"4300.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"6500.00","borrow_count":"13","geo_hash_30d_100meter_overdue_user_cnt":"1","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"1","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888064680560756","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"0","ios_app_version":"1.0.5","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"7885C220-A19B-41C6-A0A3-EE13859809A8","h5_language":"th-TH","h5_productSub":"","h5_appName":"APM Money","personal_education":"7","ios_app_package":"com.radon.kzth.lop","last_overdue_days":"0","ios_battery_percentage":"0.5","borrowing_loan_id":"1,2","emergency_is_others":"0"}',
        "response_data": '{"uuid":"4480ca31-bc2e-49ea-8702-86b6bfe6d552","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":456658654265348,"leftValue":"未开启放款包拒绝"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25120404zrcaIQFJ","extraInfo":{"_cost_time":214,"riskId":"4480ca31-bc2e-49ea-8702-86b6bfe6d552,1764840579374,c742c601f66246e164c02e83b4fc6aed"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":1680.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":0,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":0,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"48","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":0,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":10900.0,"th_beh_reject_pctb2407_meta_20240717":"0.1739","credit_amt_2":8000.0,"credit_amt_3":7500.0,"Expected_loan_time_1":"1764840320612","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1764840320612","Expected_loan_time_3":"1764840320612","th_app_last_21d_social_update_cnt_meta_20240715":20,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.5833","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.8696","th_app_last_1d_biz_update_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":618.5437763734612,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":1,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":0,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"8","th_externel_onway_cnt":"5","credit_amt":"9900","is_advertising":"0","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":1,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"0.9767","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":0,"th_beh_ontime_repay_pct_meta_20240717":"0.8750","th_model_score_2_meta_20240619":607.79,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.97,"ABtest":"实验组","overdue_detail":"0","model_score":576.08,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251204042520K4J0HycX",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
        "verify_time": 1764840326,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"0","Settlement_frequency_1":"0","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone 14 Pro Max","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"0","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.70","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"0","emergency_contact_name":"สมับติ","borrow_rank":"0","geohash_add_time":"1758994148000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"0","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"9700.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"1.6573263889","call_risk_type":"0","last_financial_product_id_1":"-999","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"-999","last_financial_product_id_3":"-999","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"4.2343518519","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1758994146980","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w4x6ctb86h0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"0","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"0","h5_browserName":"","last_7d_self_same_face_id_cnt":"0","personal_city":"792","client_ip":"2405:9800:b920:1705:c4c7:d35f:1e40:5001","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"None","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"9700.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.125,\\"app_9999d_fi_效率_cnt\\":5,\\"bj_ios_model_prob\\":0.271092952988736,\\"bj_ios_model_score\\":619.7816890864531,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_first_loan_model_2506_prob\\":0.3438604474067688,\\"ios_first_loan_model_2506_score\\":618.6435900858668,\\"ios_model\\":-1,\\"is_available\\":1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":5000,\\"pl_loan_7o_phone_max_amt\\":5000,\\"pl_loan_9999d_phone_max_amt\\":5000,\\"pl_loan_9999d_user_max_amt\\":0,\\"pl_repay_30d_phone_ontime_latest_hours\\":79.38937,\\"pl_repay_3o_phone_early_cnt\\":1,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.33103,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.5,\\"pl_repay_60d_phone_early_earliest_hours\\":395.50604,\\"pl_repay_60d_phone_ontime_earliest_hours\\":395.50604,\\"pl_repay_60d_phone_ontime_latest_hours\\":79.38937,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.18872,\\"pl_repay_90d_phone_early_cnt\\":3,\\"pl_repay_90d_user_early_cnt\\":0,\\"pl_repay_9999d_phone_cnt\\":10,\\"pl_repay_9999d_phone_early_cnt\\":3,\\"pure_new_tag\\":0}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"3","if_id_number_modified":"0","whitelist_type":"2","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"0","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD20250928122912e7IS3Oan","age":"30","last_credit_amt":"0","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1758994152792","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"-1","last_7d_all_same_face_phone_cnt":"0","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.26200825668015","another_contact_name":"ทัศนาทิพย์","ip":"2405:9800:b920:1705:c4c7:d35f:1e40:5001","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"0","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"0","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"0","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"0","const_id":"58F49037-86E7-4D84-A9C3-E95FFF9BBFF6","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1733670480000","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"0","last_contract_amount_1":"0","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"1,2","current_app_overdue_order_cnt":"0","first_credit_amount_2":"0","last_7d_self_same_face_apply_cnt":"0","first_credit_amount_3":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"0","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"0","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"0","platform_check_baiduai_black":"0","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"14.577553142780621","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.4","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"0","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"58F49037-86E7-4D84-A9C3-E95FFF9BBFF6","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","ios_battery_percentage":"0.15000000596046448","borrowing_loan_id":"","emergency_is_others":"0"}',
        "response_data": '{"uuid":"10bb1cf6-5702-4899-9f85-d230ea424fa8","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197498446007,"leftValue":"异常校验"},{"id":382197500543014,"leftValue":"异常校验"},{"id":411574860709947,"leftValue":"异常校验"},{"id":382197502640796,"leftValue":"异常校验"},{"id":382197500543452,"leftValue":"泰国_IOS_首贷_新包老客_活体检测分_Meta_20240621"},{"id":382197500543485,"leftValue":"泰国_IOS_首贷_新包老客_人脸相似度_Meta_20240621"},{"id":382197500543495,"leftValue":"泰国_IOS_首贷_新包老客_其他平台当前逾期数量_Meta_20240621"},{"id":382197500543508,"leftValue":"泰国_IOS_首贷_新包老客_当前逾期人脸_Meta_20240620"},{"id":382197500543521,"leftValue":"泰国_IOS_首贷_新包老客_年龄准入_Meta_20240620"},{"id":382197500543538,"leftValue":"泰国_IOS_首贷_新包老客_外部黑名单_Meta_20240621"},{"id":382197500543551,"leftValue":"泰国_IOS_首贷_新包老客_云服务黑名单_Meta_20240621"},{"id":389414262866011,"leftValue":"泰国_IOS_首贷_新包老客_人脸黑名单_Meta_20241111"},{"id":393581725417533,"leftValue":"ocr无法识别年龄"},{"id":436544793935884,"leftValue":"电销包新包老客拒绝"},{"id":382197498446159,"leftValue":"泰国_IOS_首贷_新包老客_非正常安装程序_Meta_20240620"},{"id":382197498446172,"leftValue":"泰国_IOS_首贷_新包老客_越狱_Meta_20240620"},{"id":405023739543570,"leftValue":"电销欺诈"},{"id":432743791919143,"leftValue":"中介窝点风险"},{"id":433108941733894,"leftValue":"近7天同geohash不同手机号数量"},{"id":434209938145287,"leftValue":"泰国_IOS_首贷_纯新户_APP相似度"},{"id":434209938145299,"leftValue":"泰国_IOS_首贷_纯新户_通讯录相似度"},{"id":434209938145311,"leftValue":"泰国_IOS_首贷_纯新户_人脸聚集风险"},{"id":434209938145323,"leftValue":"泰国_IOS_首贷_纯新户_同银行卡不同姓名"},{"id":434209938145335,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同姓名"},{"id":382197496348689,"leftValue":"泰国_IOS_首贷_新包老客_在途订单_Meta_20240620"},{"id":418788323295235,"leftValue":"泰国_IOS_首贷_新包老客_在途金额"},{"id":432349613326340,"leftValue":"泰国_IOS_首贷_新包老客_在途订单"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25092812y7P5cP9c","extraInfo":{"_observe_mode":[false],"_cost_time":116,"ruleIdCodeNamePair":"[{\\"code\\":\\"rule1\\",\\"id\\":405047798071317,\\"name\\":\\"白名单_投放返回\\"}]","riskId":"10bb1cf6-5702-4899-9f85-d230ea424fa8,1758994382822,c36f15c7cd8f16b9ad01648b6a431815"},"hitPolicies":[{"id":382197500543324,"name":"泰国_IOS贷超_前处理_公共变量_赋值"}],"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":"","th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":"","his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":772,"th_app_last_1d_fin_update_cnt_b2407_meta_20240715":"","th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":"","th_app_last_7d_food_update_cnt_b2407_meta_20240715":"","bj_ios_model_score":619.7816890864531,"is_error":"0","credit_amt_1":"3200","th_beh_reject_pctb2407_meta_20240717":"","credit_amt_2":"","credit_amt_3":"","Expected_loan_time_1":"1758994152792","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":1,"Expected_loan_time_2":"","Expected_loan_time_3":"","th_app_last_21d_social_update_cnt_meta_20240715":"","lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":5.0,"coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"","th_app_last_1d_biz_update_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_meta_20240619":0,"financial_product_id_1":"000002","th_model_bscore_meta_20240719":"","financial_product_id_3":"","financial_product_id_2":"","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":3,"th_app_last_21d_highrisk_update_cnt_meta_20240715":"","th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":0.0,"th_app_loan_earlist_install_days_b2407_meta_20240715":"","risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":"","th_app_last_3m_highrisk_install_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_a2405_meta_20240619":0,"job_monthly_income":"None","credit_amt":"3000","is_advertising":"1","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":"","th_app_loan_latest_install_days_a2405_meta_20240619":25,"th_app_last_7d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_per_repay_b2407_meta_20240717":"","appliacation_limit":"1","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":-1,"th_model_bversion_meta_20240719":"","th_app_last_3d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_pct_meta_20240717":"","th_model_score_2_meta_20240619":575.16,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":0.125,"th_ios_first_model_score_2503_jixia_20250321":594.06,"ABtest":"","overdue_detail":"0","model_score":601.79,"th_app_last_30d_pe_update_cnt_meta_20240715":""}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20250928122912e7IS3Oan",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1758994164,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"0","Settlement_frequency_1":"1","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone 14 Pro Max","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"0","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.79","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"1","geohash_add_time":"1759644288000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"1","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"12700.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.9222685185","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"-999","last_financial_product_id_3":"-999","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"4.4241319444","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1758994146980","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w4x6ctb8550","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"0","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"3200","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"2405:9800:b920:a58d:85b:bbf4:2f3d:43fc","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"12700.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0.025,\\"app_60d_fi_pct\\":0.125,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":6000,\\"pl_loan_5o_user_avg_wait_hours\\":0,\\"pl_loan_7o_phone_max_amt\\":6000,\\"pl_loan_9999d_phone_max_amt\\":6000,\\"pl_loan_9999d_user_max_amt\\":3200,\\"pl_repay_30d_phone_ontime_latest_hours\\":22.1346,\\"pl_repay_30d_user_ontime_latest_hours\\":22.1346,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.20126,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":576.10126,\\"pl_repay_60d_phone_ontime_earliest_hours\\":576.10126,\\"pl_repay_60d_phone_ontime_latest_hours\\":22.1346,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":22.1346,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.32068,\\"pl_repay_90d_phone_early_cnt\\":5,\\"pl_repay_90d_user_early_cnt\\":0,\\"pl_repay_9999d_phone_cnt\\":15,\\"pl_repay_9999d_phone_early_cnt\\":5,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.3355337785501282,\\"th_ios_bcard_2508_score\\":583.3940136900374}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"3","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD20251005010500zow0HW05","age":"30","last_credit_amt":"3200","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1759644300455","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"0.92","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.26205534958163","another_contact_name":"ทัศนาทิพย์","ip":"2405:9800:b920:a58d:85b:bbf4:2f3d:43fc","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"0","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"10","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"1","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"1","const_id":"58F49037-86E7-4D84-A9C3-E95FFF9BBFF6","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730307901003","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"0","last_contract_amount_1":"3200.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"1","current_app_overdue_order_cnt":"0","first_credit_amount_2":"0","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"1","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"14.577501438464434","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.4","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"58F49037-86E7-4D84-A9C3-E95FFF9BBFF6","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","ios_battery_percentage":"0.44999998807907104","borrowing_loan_id":"","emergency_is_others":"0"}',
        "response_data": '{"uuid":"97e41aa4-eb8a-4a35-886a-300e0d24719f","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25100501Sa256894","extraInfo":{"_cost_time":380,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"97e41aa4-eb8a-4a35-886a-300e0d24719f,1759644032852,95a054fb52f76dfac1e988d6ab1099b1"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":8,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":10,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"25","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":5,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":2,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":3900.0,"th_beh_reject_pctb2407_meta_20240717":"0.4211","credit_amt_2":700.0,"credit_amt_3":"500","Expected_loan_time_1":"1759644300455","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1759644300455","Expected_loan_time_3":"1759644300455","th_app_last_21d_social_update_cnt_meta_20240715":25,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6667","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.7857","th_app_last_1d_biz_update_cnt_meta_20240715":5,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":2,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":7,"th_app_last_3m_highrisk_install_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","credit_amt":"3200","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":2,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_pct_meta_20240717":"0.7857","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251005010500zow0HW05",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1759644307,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 2,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"0","Settlement_frequency_1":"1","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone 14 Pro Max","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.73","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"1","geohash_add_time":"1759646347000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"1","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"16600.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"1.1655902778","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"-999","last_financial_product_id_3":"-999","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"4.6674537037","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1758994146980","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w4x6ctb8560","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"0","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"3900","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"2405:9800:b920:a58d:84dd:14f3:84fe:c565","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"16600.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0.025,\\"app_60d_fi_pct\\":0.125,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":3900,\\"pl_loan_5o_user_avg_wait_hours\\":22.14944,\\"pl_loan_7o_phone_max_amt\\":6000,\\"pl_loan_9999d_phone_max_amt\\":6000,\\"pl_loan_9999d_user_max_amt\\":3900,\\"pl_repay_30d_phone_ontime_latest_hours\\":27.97418,\\"pl_repay_30d_user_ontime_latest_hours\\":27.97418,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":581.94085,\\"pl_repay_60d_phone_ontime_earliest_hours\\":581.94085,\\"pl_repay_60d_phone_ontime_latest_hours\\":27.97418,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":27.97418,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.2272,\\"pl_repay_90d_phone_early_cnt\\":5,\\"pl_repay_90d_user_early_cnt\\":0,\\"pl_repay_9999d_phone_cnt\\":15,\\"pl_repay_9999d_phone_early_cnt\\":5,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.40965312326284076,\\"th_ios_bcard_2508_score\\":574.222458495208}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"2","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD2025100506552304T8ebDM","age":"30","last_credit_amt":"3900","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1759665323353","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"1.17","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.26205785715462","another_contact_name":"ทัศนาทิพย์","ip":"2405:9800:b920:a58d:84dd:14f3:84fe:c565","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"0","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"10","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"1","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"1","const_id":"58F49037-86E7-4D84-A9C3-E95FFF9BBFF6","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730307901008","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"0","last_contract_amount_1":"3900.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"1","current_app_overdue_order_cnt":"0","first_credit_amount_2":"0","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"1","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"14.577499284244148","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.4","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"58F49037-86E7-4D84-A9C3-E95FFF9BBFF6","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","ios_battery_percentage":"0.15000000596046448","borrowing_loan_id":"1","emergency_is_others":"0"}',
        "response_data": '{"uuid":"630a54f7-6044-4510-b93c-e799b483cc6c","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25100506uP3JIPH6","extraInfo":{"_cost_time":500,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"630a54f7-6044-4510-b93c-e799b483cc6c,1759665552092,f7366276ccf451dba1c017db95f359e7"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":8,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":10,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"25","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":5,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":2,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":4600.0,"th_beh_reject_pctb2407_meta_20240717":"0.4000","credit_amt_2":700.0,"credit_amt_3":"500","Expected_loan_time_1":"1759665323353","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1759665323353","Expected_loan_time_3":"1759665323353","th_app_last_21d_social_update_cnt_meta_20240715":25,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6000","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.7333","th_app_last_1d_biz_update_cnt_meta_20240715":5,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":2,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":7,"th_app_last_3m_highrisk_install_cnt_meta_20240715":0,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","credit_amt":"3900","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":2,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_pct_meta_20240717":"0.7333","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD2025100506552304T8ebDM",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1759665330,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 3,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"1","Settlement_frequency_1":"2","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"1","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone 14 Pro Max","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"0","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.95","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"2","geohash_add_time":"1759853914000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"4300.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"16.6571990741","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"-4","first60_borrowtime_diff":"1.5169444444","last_overdue_days_3":"-4","diffbank_id_num":"0","last_overdue_days_1":"-4","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1761283872811","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w4x6ct8tnf0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"1","last_credit_amt_2":"700","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"500","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"3900","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"2403:6200:8890:3bd8:d886:d9db:df83:a086","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"4300.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.13158,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":4300,\\"pl_loan_5o_user_avg_wait_hours\\":26.06556,\\"pl_loan_7o_phone_max_amt\\":6000,\\"pl_loan_9999d_phone_max_amt\\":6000,\\"pl_loan_9999d_user_max_amt\\":3900,\\"pl_loan_9999d_user_onway_cnt\\":0,\\"pl_repay_2o_phone_early_earliest_hours\\":399.84863,\\"pl_repay_2o_phone_ontime_latest_hours\\":399.84863,\\"pl_repay_30d_phone_ontime_latest_hours\\":399.84863,\\"pl_repay_30d_user_ontime_latest_hours\\":399.8753,\\"pl_repay_3o_phone_early_cnt\\":2,\\"pl_repay_3o_phone_s-1_cnt\\":1,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.57,\\"pl_repay_5o_phone_s-1_cnt\\":3,\\"pl_repay_5o_phone_s-1_pct\\":0.75,\\"pl_repay_60d_phone_early_earliest_hours\\":1031.53919,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1031.53919,\\"pl_repay_60d_phone_ontime_latest_hours\\":399.84863,\\"pl_repay_60d_user_latest_overdue_days\\":-4,\\"pl_repay_60d_user_ontime_earliest_hours\\":477.57252,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.35046,\\"pl_repay_90d_phone_early_cnt\\":11,\\"pl_repay_90d_user_early_cnt\\":3,\\"pl_repay_9999d_phone_cnt\\":23,\\"pl_repay_9999d_phone_early_cnt\\":11,\\"pl_repay_9999d_user_latest_overdue_days\\":-4,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.17282124628528342,\\"th_ios_bcard_2508_score\\":608.8576274932793}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"3","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD202510241231179Ft29VzP","age":"30","last_credit_amt":"700","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1761283877279","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"16.66","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.26221064014683","another_contact_name":"ทัศนาทิพย์","ip":"2403:6200:8890:3bd8:d886:d9db:df83:a086","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"1","borrow_count_3":"1","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"12","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"2","const_id":"A470A979-1E39-4805-960E-A080DDAB5795","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730300641002","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"700.00","last_contract_amount_1":"3900.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"1","current_app_overdue_order_cnt":"0","first_credit_amount_2":"700.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"1","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"500.00","borrow_count":"4","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"14.576982384605783","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.4","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"A470A979-1E39-4805-960E-A080DDAB5795","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","ios_battery_percentage":"0.6499999761581421","borrowing_loan_id":"","emergency_is_others":"0"}',
        "response_data": '{"uuid":"7f9f4867-0377-4b6d-920d-b0705a99a874","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25102412Knk3jeF0","extraInfo":{"_cost_time":400,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"7f9f4867-0377-4b6d-920d-b0705a99a874,1761283805714,e194b92c806d82e7a9e3ba51a6c3684f"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":9,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":10,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"18","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":4,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":2,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":6700.0,"th_beh_reject_pctb2407_meta_20240717":"0.3478","credit_amt_2":3500.0,"credit_amt_3":3300.0,"Expected_loan_time_1":"1761283877279","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1761283877279","Expected_loan_time_3":"1761283877279","th_app_last_21d_social_update_cnt_meta_20240715":26,"lastoverdue_day":"-4","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.0000","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9474","th_app_last_1d_biz_update_cnt_meta_20240715":4,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":1.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":3,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":7,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","th_externel_onway_cnt":"1","credit_amt":"3900","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":3,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":3,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":3,"th_beh_ontime_repay_pct_meta_20240717":"0.9474","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD202510241231179Ft29VzP",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1761283884,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 2,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"2","Settlement_frequency_1":"2","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"2","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone 14 Pro Max","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.91","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"3","geohash_add_time":"1761286203000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"11000.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.7252662037","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"-5","first60_borrowtime_diff":"3.3259722222","last_overdue_days_3":"-5","diffbank_id_num":"0","last_overdue_days_1":"-4","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1761283872811","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w4x6ctb8570","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"3500","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"3300","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"6700","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"2405:9800:b920:93cd:f4a9:6708:4fae:5c1a","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"11000.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.13158,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":3300,\\"pl_loan_5o_user_avg_wait_hours\\":400.31417,\\"pl_loan_7o_phone_max_amt\\":6700,\\"pl_loan_9999d_phone_max_amt\\":6700,\\"pl_loan_9999d_user_max_amt\\":6700,\\"pl_loan_9999d_user_onway_cnt\\":1,\\"pl_repay_1o_phone_early_latest_hours\\":17.4065,\\"pl_repay_1o_phone_ontime_latest_hours\\":17.4065,\\"pl_repay_2o_phone_early_earliest_hours\\":17.40678,\\"pl_repay_2o_phone_ontime_latest_hours\\":17.4065,\\"pl_repay_30d_phone_ontime_latest_hours\\":17.4065,\\"pl_repay_30d_user_ontime_latest_hours\\":17.4065,\\"pl_repay_3o_phone_early_cnt\\":2,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.38202,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":1074.95595,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1074.95595,\\"pl_repay_60d_phone_ontime_latest_hours\\":17.4065,\\"pl_repay_60d_user_latest_overdue_days\\":-5,\\"pl_repay_60d_user_ontime_earliest_hours\\":520.98928,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.20858,\\"pl_repay_90d_phone_early_cnt\\":13,\\"pl_repay_90d_user_early_cnt\\":5,\\"pl_repay_9999d_phone_cnt\\":25,\\"pl_repay_9999d_phone_early_cnt\\":13,\\"pl_repay_9999d_user_latest_overdue_days\\":-5,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.24831619137918726,\\"th_ios_bcard_2508_score\\":595.6382085618437}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"3","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD20251026075617GSm9pAd6","age":"30","last_credit_amt":"3500","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1761440177525","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"0.73","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.26205839180297","another_contact_name":"ทัศนาทิพย์","ip":"2405:9800:b920:93cd:f4a9:6708:4fae:5c1a","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"2","borrow_count_3":"2","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"11","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"2","const_id":"A470A979-1E39-4805-960E-A080DDAB5795","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730300760007","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"3500.00","last_contract_amount_1":"6700.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"700.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"3300.00","borrow_count":"6","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"14.577504488194565","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.4","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"A470A979-1E39-4805-960E-A080DDAB5795","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","ios_battery_percentage":"0.75","borrowing_loan_id":"1","emergency_is_others":"0"}',
        "response_data": '{"uuid":"1b5a3423-984f-48df-9265-dbf7bb06b350","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25102607131iwWu0","extraInfo":{"_cost_time":389,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"1b5a3423-984f-48df-9265-dbf7bb06b350,1761440416907,aa0cb34ddc5297cca9c4b95aaeab2316"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":9,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":10,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"15","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":4,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":2,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":8200.0,"th_beh_reject_pctb2407_meta_20240717":"0.3333","credit_amt_2":5000.0,"credit_amt_3":4800.0,"Expected_loan_time_1":"1761440177525","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1761440177525","Expected_loan_time_3":"1761440177525","th_app_last_21d_social_update_cnt_meta_20240715":26,"lastoverdue_day":"-5","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.5000","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9091","th_app_last_1d_biz_update_cnt_meta_20240715":4,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":1.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":3,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":7,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","th_externel_onway_cnt":"2","credit_amt":"6700","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":3,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":3,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":3,"th_beh_ontime_repay_pct_meta_20240717":"0.9091","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251026075617GSm9pAd6",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1761440184,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"2","Settlement_frequency_1":"3","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"2","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone 14 Pro Max","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.92","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"4","geohash_add_time":"1761630292000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"9800.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"1.8529976852","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"-5","first60_borrowtime_diff":"4.0353819444","last_overdue_days_3":"-5","diffbank_id_num":"0","last_overdue_days_1":"-2","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1761788832626","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jde2pu0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"5000","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"4800","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"6700","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.91.222","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"9800.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0.02564,\\"app_60d_fi_pct\\":0.15385,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":4800,\\"pl_loan_5o_user_avg_wait_hours\\":17.80611,\\"pl_loan_7o_phone_max_amt\\":6700,\\"pl_loan_9999d_phone_max_amt\\":6700,\\"pl_loan_9999d_user_max_amt\\":6700,\\"pl_loan_9999d_user_onway_cnt\\":2,\\"pl_repay_30d_phone_ontime_latest_hours\\":114.25638,\\"pl_repay_30d_user_ontime_latest_hours\\":114.25638,\\"pl_repay_3o_phone_early_cnt\\":1,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.40964,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.33333,\\"pl_repay_60d_phone_early_earliest_hours\\":1171.80582,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1171.80582,\\"pl_repay_60d_phone_ontime_latest_hours\\":114.25638,\\"pl_repay_60d_user_latest_overdue_days\\":-5,\\"pl_repay_60d_user_ontime_earliest_hours\\":617.83916,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.31439,\\"pl_repay_90d_phone_early_cnt\\":15,\\"pl_repay_90d_user_early_cnt\\":6,\\"pl_repay_9999d_phone_cnt\\":27,\\"pl_repay_9999d_phone_early_cnt\\":15,\\"pl_repay_9999d_user_latest_overdue_days\\":-5,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.25451265247738997,\\"th_ios_bcard_2508_score\\":594.6881864770542}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"3","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD20251030084716Wi5sRAml","age":"30","last_credit_amt":"4800","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1761788837144","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"1.85","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.7933792626072","another_contact_name":"ทัศนาทิพย์","ip":"49.237.91.222","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"2","borrow_count_3":"2","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"3","const_id":"A470A979-1E39-4805-960E-A080DDAB5795","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730296800008","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"5000.00","last_contract_amount_1":"6700.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"700.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"4800.00","borrow_count":"7","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888756083039333","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.4","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"A470A979-1E39-4805-960E-A080DDAB5795","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","ios_battery_percentage":"0.550000011920929","borrowing_loan_id":"3,2","emergency_is_others":"0"}',
        "response_data": '{"uuid":"e9cad9d9-35ab-4a1c-ab1b-184f741d57ff","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25103008PQ9N6e61","extraInfo":{"_cost_time":468,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"e9cad9d9-35ab-4a1c-ab1b-184f741d57ff,1761789078590,41f7f80b65187d9a64fa8fe0ff8d049e"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":9,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":10,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"18","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":4,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":2,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":8200.0,"th_beh_reject_pctb2407_meta_20240717":"0.3077","credit_amt_2":6500.0,"credit_amt_3":6300.0,"Expected_loan_time_1":"1761788837144","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1761788837144","Expected_loan_time_3":"1761788837144","th_app_last_21d_social_update_cnt_meta_20240715":25,"lastoverdue_day":"-2","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6667","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9167","th_app_last_1d_biz_update_cnt_meta_20240715":4,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":3,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":7,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","th_externel_onway_cnt":"2","credit_amt":"6700","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":4,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":3,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":3,"th_beh_ontime_repay_pct_meta_20240717":"0.9167","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251030084716Wi5sRAml",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1761788844,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"3","Settlement_frequency_1":"3","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"3","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone 14 Pro Max","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.90","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"5","geohash_add_time":"1761826796000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"15800.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"3.2669444444","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"5.3602546296","last_overdue_days_3":"-2","diffbank_id_num":"0","last_overdue_days_1":"-2","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1762251436307","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jdz3rz0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"5000","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"4800","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"8200","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"223.24.200.54","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"15800.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.12821,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":2000,\\"pl_loan_5o_user_avg_wait_hours\\":50.62315,\\"pl_loan_7o_phone_max_amt\\":8200,\\"pl_loan_9999d_phone_max_amt\\":8200,\\"pl_loan_9999d_user_max_amt\\":8200,\\"pl_loan_9999d_user_onway_cnt\\":1,\\"pl_repay_2o_phone_early_earliest_hours\\":78.40665,\\"pl_repay_2o_phone_ontime_latest_hours\\":78.40665,\\"pl_repay_30d_phone_ontime_latest_hours\\":78.40665,\\"pl_repay_30d_user_ontime_latest_hours\\":119.20665,\\"pl_repay_3o_phone_early_cnt\\":2,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.64286,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":944.57665,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1013.90582,\\"pl_repay_60d_phone_ontime_latest_hours\\":78.40665,\\"pl_repay_60d_user_latest_overdue_days\\":-2,\\"pl_repay_60d_user_ontime_earliest_hours\\":746.33998,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.09961,\\"pl_repay_90d_phone_early_cnt\\":19,\\"pl_repay_90d_user_early_cnt\\":7,\\"pl_repay_9999d_phone_cnt\\":32,\\"pl_repay_9999d_phone_early_cnt\\":19,\\"pl_repay_9999d_user_latest_overdue_days\\":-2,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.25608783487775727,\\"th_ios_bcard_2508_score\\":594.4491280753124}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"3","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD20251104051720nQW4hRKs","age":"30","last_credit_amt":"8200","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1762251440921","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"3.50","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.79886532998901","another_contact_name":"ทัศนาทิพย์","ip":"223.24.200.54","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"3","borrow_count_3":"3","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"3","const_id":"A470A979-1E39-4805-960E-A080DDAB5795","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730394241007","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"5000.00","last_contract_amount_1":"8200.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"700.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"4800.00","borrow_count":"9","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.8903619303027","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.4","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"A470A979-1E39-4805-960E-A080DDAB5795","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","ios_battery_percentage":"0.05000000074505806","borrowing_loan_id":"1","emergency_is_others":"0"}',
        "response_data": '{"uuid":"b774b426-0303-4b55-88e9-5bd0feeb3d38","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz251104052Wldc5a9","extraInfo":{"_cost_time":491,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"b774b426-0303-4b55-88e9-5bd0feeb3d38,1762251153910,fe5e0ce34886bff086edb563d3ef0383"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":9,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":10,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"17","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":4,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":2,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":9400.0,"th_beh_reject_pctb2407_meta_20240717":"0.2667","credit_amt_2":6200.0,"credit_amt_3":6000.0,"Expected_loan_time_1":"1762251440921","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1762251440921","Expected_loan_time_3":"1762251440921","th_app_last_21d_social_update_cnt_meta_20240715":25,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.7500","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9000","th_app_last_1d_biz_update_cnt_meta_20240715":4,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":3,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":7,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","th_externel_onway_cnt":"3","credit_amt":"8200","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":3,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":3,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":3,"th_beh_ontime_repay_pct_meta_20240717":"0.9000","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251104051720nQW4hRKs",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1762251447,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 2,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"3","Settlement_frequency_1":"4","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"3","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone 14 Pro Max","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.88","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"6","geohash_add_time":"1762296069000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"18900.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"1.4782175926","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"4.7301157407","last_overdue_days_3":"-2","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1762425672354","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jfmbfe0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"5000","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"6000","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"8200","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"2405:9800:b500:88d8:55e6:ce2f:9ea6:c0fd","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"18900.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0.02632,\\"app_60d_fi_pct\\":0.13158,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":-1,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":7100,\\"pl_loan_5o_user_avg_wait_hours\\":67.77049,\\"pl_loan_7o_phone_max_amt\\":7100,\\"pl_loan_9999d_phone_max_amt\\":8200,\\"pl_loan_9999d_user_max_amt\\":8200,\\"pl_loan_9999d_user_onway_cnt\\":1,\\"pl_repay_30d_phone_ontime_latest_hours\\":126.80548,\\"pl_repay_30d_user_ontime_latest_hours\\":35.49326,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":992.97548,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1062.30465,\\"pl_repay_60d_phone_ontime_latest_hours\\":126.80548,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":794.73881,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.03669,\\"pl_repay_90d_phone_early_cnt\\":19,\\"pl_repay_90d_user_early_cnt\\":7,\\"pl_repay_9999d_phone_cnt\\":34,\\"pl_repay_9999d_phone_early_cnt\\":19,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.29897535774760875,\\"th_ios_bcard_2508_score\\":588.2680282848125}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"3","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD2025110605411654dpvwvL","age":"30","last_credit_amt":"6000","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1762425676797","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"1.48","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.80757214241142","another_contact_name":"ทัศนาทิพย์","ip":"2405:9800:b500:88d8:55e6:ce2f:9ea6:c0fd","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"3","borrow_count_3":"3","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"4","const_id":"A470A979-1E39-4805-960E-A080DDAB5795","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730394361007","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"5000.00","last_contract_amount_1":"8200.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"700.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"6000.00","borrow_count":"10","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.887506802403333","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.4","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"A470A979-1E39-4805-960E-A080DDAB5795","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","ios_battery_percentage":"0.05000000074505806","borrowing_loan_id":"3","emergency_is_others":"0"}',
        "response_data": '{"uuid":"b67d0d2f-da7c-4d8b-8715-af18acc2bcbb","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25110605q5Y7BkHm","extraInfo":{"_cost_time":414,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"b67d0d2f-da7c-4d8b-8715-af18acc2bcbb,1762425600987,b7a497f45f26e090ce49a6d005bf2777"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":8,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":10,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"22","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":3,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":2,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":9200.0,"th_beh_reject_pctb2407_meta_20240717":"0.2424","credit_amt_2":6000.0,"credit_amt_3":7000.0,"Expected_loan_time_1":"1762425676797","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1762425676797","Expected_loan_time_3":"1762425676797","th_app_last_21d_social_update_cnt_meta_20240715":26,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.7333","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.8788","th_app_last_1d_biz_update_cnt_meta_20240715":4,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":3,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":5,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","th_externel_onway_cnt":"4","credit_amt":"8200","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":4,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":3,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"1.0000","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":3,"th_beh_ontime_repay_pct_meta_20240717":"0.8788","th_model_score_2_meta_20240619":568.08,"th_device_ios_model_meta_20240619":0,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":568.1,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD2025110605411654dpvwvL",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1762425683,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 3,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"4","Settlement_frequency_1":"5","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"4","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"0","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.92","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"7","geohash_add_time":"1764162308000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"19000.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"14.1725810185","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"0.2730555556","last_overdue_days_3":"1","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1764162325104","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jde3mm0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"6000","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"6000","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"9200","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.32.135","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"19000.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.09677,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":36,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":4900,\\"pl_loan_5o_user_avg_wait_hours\\":63.405,\\"pl_loan_7o_phone_max_amt\\":9200,\\"pl_loan_9999d_phone_max_amt\\":9200,\\"pl_loan_9999d_user_max_amt\\":9200,\\"pl_loan_9999d_user_onway_cnt\\":0,\\"pl_repay_30d_phone_ontime_latest_hours\\":340.14202,\\"pl_repay_30d_user_ontime_latest_hours\\":346.5423,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.14798,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.5,\\"pl_repay_60d_phone_early_earliest_hours\\":1208.64202,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1277.1423,\\"pl_repay_60d_phone_ontime_latest_hours\\":340.14202,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":1277.1423,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.02426,\\"pl_repay_90d_phone_early_cnt\\":21,\\"pl_repay_90d_user_early_cnt\\":7,\\"pl_repay_9999d_phone_cnt\\":41,\\"pl_repay_9999d_phone_early_cnt\\":21,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.31419519032296106,\\"th_ios_bcard_2508_score\\":586.2019952182051}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"2","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD20251126080529K54KDE50","age":"30","last_credit_amt":"9200","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1764162329551","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"14.43","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.79326854221418","another_contact_name":"ทัศนาทิพย์","ip":"49.237.32.135","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"4","borrow_count_3":"4","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"5","const_id":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730387162000","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"6000.00","last_contract_amount_1":"9200.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"700.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"6000.00","borrow_count":"13","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"1","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.88897802545289","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.8","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","last_overdue_days":"0","ios_battery_percentage":"0.2","borrowing_loan_id":"","emergency_is_others":"0"}',
        "response_data": '{"uuid":"5b7dd99e-e8d8-4a53-ad6a-e03a6667b163","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":456658654265348,"leftValue":"未开启放款包拒绝"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25112608n3g2gHIX","extraInfo":{"_cost_time":440,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"5b7dd99e-e8d8-4a53-ad6a-e03a6667b163,1764162027966,771782ea2b38974cab2f1cc13fbc6d9c"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":8,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":9,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"27","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":2,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":3,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":10200.0,"th_beh_reject_pctb2407_meta_20240717":"0.2222","credit_amt_2":7000.0,"credit_amt_3":7000.0,"Expected_loan_time_1":"1764162329551","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1764162329551","Expected_loan_time_3":"1764162329551","th_app_last_21d_social_update_cnt_meta_20240715":25,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.0000","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.8974","th_app_last_1d_biz_update_cnt_meta_20240715":4,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":2,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":3,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","th_externel_onway_cnt":"3","credit_amt":"9200","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":1,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"0.9722","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_pct_meta_20240717":"0.8974","th_model_score_2_meta_20240619":589.58,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":575.01,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251126080529K54KDE50",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1764162335,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 0,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"5","Settlement_frequency_1":"6","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"5","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.93","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"9","geohash_add_time":"1764665422000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"23300.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.2860185185","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"4.3362152778","last_overdue_days_3":"-1","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1764162325104","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jd7wnm0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"8600","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"8200","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"10200","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.93.65","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"23300.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.06667,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":36,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":8600,\\"pl_loan_5o_user_avg_wait_hours\\":2.16861,\\"pl_loan_7o_phone_max_amt\\":10200,\\"pl_loan_9999d_phone_max_amt\\":10200,\\"pl_loan_9999d_user_max_amt\\":10200,\\"pl_loan_9999d_user_onway_cnt\\":2,\\"pl_repay_30d_phone_ontime_latest_hours\\":9.23416,\\"pl_repay_30d_user_ontime_latest_hours\\":9.23416,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.23102,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.5,\\"pl_repay_60d_phone_early_earliest_hours\\":1340.05083,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1340.05083,\\"pl_repay_60d_phone_ontime_latest_hours\\":9.23416,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":1340.07778,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.0664,\\"pl_repay_90d_phone_early_cnt\\":25,\\"pl_repay_90d_user_early_cnt\\":8,\\"pl_repay_9999d_phone_cnt\\":47,\\"pl_repay_9999d_phone_early_cnt\\":25,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.2683270996432455,\\"th_ios_bcard_2508_score\\":592.6233778720075}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"3","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD202512020443257gBpPzE8","age":"30","last_credit_amt":"8600","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1764668605764","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"0.29","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.7936522568163","another_contact_name":"ทัศนาทิพย์","ip":"49.237.93.65","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"5","borrow_count_3":"5","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"6","const_id":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730480521006","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"8600.00","last_contract_amount_1":"10200.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"700.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"8200.00","borrow_count":"16","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"1","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888419337316076","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.8","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","last_overdue_days":"0","ios_battery_percentage":"0.6","borrowing_loan_id":"3,2","emergency_is_others":"0"}',
        "response_data": '{"uuid":"57f0db5f-4398-4953-aacb-b0022bb98f60","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":456658654265348,"leftValue":"未开启放款包拒绝"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz251202048qlswF83","extraInfo":{"_cost_time":366,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"57f0db5f-4398-4953-aacb-b0022bb98f60,1764668299753,d6974667696cdd56c81fe52f6490776a"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":8,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":6,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"36","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":2,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":3,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":11400.0,"th_beh_reject_pctb2407_meta_20240717":"0.1905","credit_amt_2":9800.0,"credit_amt_3":9400.0,"Expected_loan_time_1":"1764668605764","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1764668605764","Expected_loan_time_3":"1764668605764","th_app_last_21d_social_update_cnt_meta_20240715":26,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6667","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9111","th_app_last_1d_biz_update_cnt_meta_20240715":4,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":2,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":3,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","th_externel_onway_cnt":"3","credit_amt":"10200","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"0.9762","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_pct_meta_20240717":"0.9111","th_model_score_2_meta_20240619":589.58,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":575.01,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD202512020443257gBpPzE8",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1764668610,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 0,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"4","Settlement_frequency_1":"5","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"5","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.93","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"8","geohash_add_time":"1764162392000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"23700.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.8155324074","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"5.4736689815","last_overdue_days_3":"-1","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1764162325104","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jde80x0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"7000","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"7000","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"10200","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.84.205","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"23700.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.06667,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":36,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":6500,\\"pl_loan_5o_user_avg_wait_hours\\":346.55806,\\"pl_loan_7o_phone_max_amt\\":10200,\\"pl_loan_9999d_phone_max_amt\\":10200,\\"pl_loan_9999d_user_max_amt\\":10200,\\"pl_loan_9999d_user_onway_cnt\\":2,\\"pl_repay_30d_phone_ontime_latest_hours\\":19.57299,\\"pl_repay_30d_user_ontime_latest_hours\\":19.57299,\\"pl_repay_3o_phone_early_cnt\\":1,\\"pl_repay_3o_phone_s-1_cnt\\":1,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.22801,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.5,\\"pl_repay_60d_phone_early_earliest_hours\\":1330.78743,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1330.78743,\\"pl_repay_60d_phone_ontime_latest_hours\\":19.57299,\\"pl_repay_60d_user_latest_overdue_days\\":-1,\\"pl_repay_60d_user_ontime_earliest_hours\\":1330.81438,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.16048,\\"pl_repay_90d_phone_early_cnt\\":25,\\"pl_repay_90d_user_early_cnt\\":8,\\"pl_repay_9999d_phone_cnt\\":45,\\"pl_repay_9999d_phone_early_cnt\\":25,\\"pl_repay_9999d_user_latest_overdue_days\\":-1,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.23975875354316425,\\"th_ios_bcard_2508_score\\":596.9767329211808}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"3","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD20251202072736HW8uRE93","age":"30","last_credit_amt":"10200","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1764635256841","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"0.82","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.7934103429105","another_contact_name":"ทัศนาทิพย์","ip":"49.237.84.205","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"4","borrow_count_3":"5","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"5","const_id":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730480520007","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"7000.00","last_contract_amount_1":"10200.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"700.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"7000.00","borrow_count":"14","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"1","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888773892457852","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.8","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","last_overdue_days":"-1","ios_battery_percentage":"0.7","borrowing_loan_id":"2,1","emergency_is_others":"0"}',
        "response_data": '{"uuid":"b5d2d5d3-cbf1-450c-b280-8dfa41744701","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":456658654265348,"leftValue":"未开启放款包拒绝"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25120207Oim1bqso","extraInfo":{"_cost_time":340,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"b5d2d5d3-cbf1-450c-b280-8dfa41744701,1764634951730,caa373b117d562dc1f7092e16d1278d8"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":8,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":6,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"24","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":2,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":3,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":11800.0,"th_beh_reject_pctb2407_meta_20240717":"0.2105","credit_amt_2":8600.0,"credit_amt_3":8600.0,"Expected_loan_time_1":"1764635256841","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1764635256841","Expected_loan_time_3":"1764635256841","th_app_last_21d_social_update_cnt_meta_20240715":26,"lastoverdue_day":"-1","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.5714","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9070","th_app_last_1d_biz_update_cnt_meta_20240715":4,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":2,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":3,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","th_externel_onway_cnt":"3","credit_amt":"10200","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"0.9750","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_pct_meta_20240717":"0.9070","th_model_score_2_meta_20240619":589.58,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":575.01,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251202072736HW8uRE93",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1764635262,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"5","Settlement_frequency_1":"5","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"5","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.95","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"8","geohash_add_time":"1764635733000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"16700.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.0043518519","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"5.4792476852","last_overdue_days_3":"-1","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1764162325104","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jd7x5f0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"7000","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"7000","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"10200","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.84.205","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"16700.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.06667,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":36,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":6500,\\"pl_loan_5o_user_avg_wait_hours\\":346.55806,\\"pl_loan_7o_phone_max_amt\\":10200,\\"pl_loan_9999d_phone_max_amt\\":10200,\\"pl_loan_9999d_user_max_amt\\":10200,\\"pl_loan_9999d_user_onway_cnt\\":1,\\"pl_repay_2o_phone_ontime_latest_hours\\":0.10446,\\"pl_repay_30d_phone_ontime_latest_hours\\":0.10446,\\"pl_repay_30d_user_ontime_latest_hours\\":0.10446,\\"pl_repay_3o_phone_early_cnt\\":1,\\"pl_repay_3o_phone_s-1_cnt\\":1,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.45603,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.33333,\\"pl_repay_60d_phone_early_earliest_hours\\":1330.92113,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1330.92113,\\"pl_repay_60d_phone_ontime_latest_hours\\":0.10446,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":1330.94807,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.15384,\\"pl_repay_90d_phone_early_cnt\\":25,\\"pl_repay_90d_user_early_cnt\\":8,\\"pl_repay_9999d_phone_cnt\\":46,\\"pl_repay_9999d_phone_early_cnt\\":25,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.27537673625217873,\\"th_ios_bcard_2508_score\\":591.5957454476646}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"3","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD202512020735369eJAaT3y","age":"30","last_credit_amt":"10200","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1764635737075","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"0.00","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.79354554493578","another_contact_name":"ทัศนาทิพย์","ip":"49.237.84.205","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"5","borrow_count_3":"5","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"5","const_id":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730480520007","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"7000.00","last_contract_amount_1":"10200.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"700.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"7000.00","borrow_count":"15","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"1","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888575218855273","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.8","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","last_overdue_days":"0","ios_battery_percentage":"0.7","borrowing_loan_id":"1","emergency_is_others":"0"}',
        "response_data": '{"uuid":"aacb0197-3e25-46d1-a459-d94bb181534e","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":456658654265348,"leftValue":"未开启放款包拒绝"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25120207P7CLF6f0","extraInfo":{"_cost_time":341,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"aacb0197-3e25-46d1-a459-d94bb181534e,1764635654195,b1de9b09140991dd1f83b0c5cca45534"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":8,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":6,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"30","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":2,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":3,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":11400.0,"th_beh_reject_pctb2407_meta_20240717":"0.2000","credit_amt_2":8200.0,"credit_amt_3":8200.0,"Expected_loan_time_1":"1764635737075","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1764635737075","Expected_loan_time_3":"1764635737075","th_app_last_21d_social_update_cnt_meta_20240715":26,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.7143","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9302","th_app_last_1d_biz_update_cnt_meta_20240715":4,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":2,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":3,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","th_externel_onway_cnt":"2","credit_amt":"10200","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"0.9756","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_pct_meta_20240717":"0.9302","th_model_score_2_meta_20240619":589.58,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":575.01,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD202512020735369eJAaT3y",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1764635743,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"5","Settlement_frequency_1":"5","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"5","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.93","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"8","geohash_add_time":"1764635762000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"24900.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.0917361111","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"5.5666319444","last_overdue_days_3":"-1","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1764162325104","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jd7edv0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"7000","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"8200","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"10200","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.80.72","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"24900.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.06667,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":36,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":8200,\\"pl_loan_5o_user_avg_wait_hours\\":260.44035,\\"pl_loan_7o_phone_max_amt\\":10200,\\"pl_loan_9999d_phone_max_amt\\":10200,\\"pl_loan_9999d_user_max_amt\\":10200,\\"pl_loan_9999d_user_onway_cnt\\":2,\\"pl_repay_30d_phone_ontime_latest_hours\\":2.20181,\\"pl_repay_30d_user_ontime_latest_hours\\":2.20181,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.4878,\\"pl_repay_5o_phone_s-1_cnt\\":1,\\"pl_repay_5o_phone_s-1_pct\\":0.5,\\"pl_repay_60d_phone_early_earliest_hours\\":1333.01848,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1333.01848,\\"pl_repay_60d_phone_ontime_latest_hours\\":2.20181,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":1333.04542,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.10236,\\"pl_repay_90d_phone_early_cnt\\":25,\\"pl_repay_90d_user_early_cnt\\":8,\\"pl_repay_9999d_phone_cnt\\":46,\\"pl_repay_9999d_phone_early_cnt\\":25,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.2479104290135814,\\"th_ios_bcard_2508_score\\":595.700967258256}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"3","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD20251202094127Y1694oq6","age":"30","last_credit_amt":"8200","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1764643288171","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"0.09","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.79350018406579","another_contact_name":"ทัศนาทิพย์","ip":"49.237.80.72","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"5","borrow_count_3":"5","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"5","const_id":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730480520009","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"7000.00","last_contract_amount_1":"10200.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"700.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"8200.00","borrow_count":"15","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"1","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.887991684556578","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.8","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","last_overdue_days":"0","ios_battery_percentage":"0.35","borrowing_loan_id":"3,1","emergency_is_others":"0"}',
        "response_data": '{"uuid":"80af190f-cc85-4410-8c3f-16a7297bc084","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":456658654265348,"leftValue":"未开启放款包拒绝"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25120209FrA46SV2","extraInfo":{"_cost_time":362,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"80af190f-cc85-4410-8c3f-16a7297bc084,1764642983282,3d08160be14e3219e0d0082713470fbe"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":8,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":6,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"33","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":2,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":3,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":11800.0,"th_beh_reject_pctb2407_meta_20240717":"0.1951","credit_amt_2":8600.0,"credit_amt_3":9800.0,"Expected_loan_time_1":"1764643288171","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1764643288171","Expected_loan_time_3":"1764643288171","th_app_last_21d_social_update_cnt_meta_20240715":26,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6250","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9091","th_app_last_1d_biz_update_cnt_meta_20240715":4,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":2,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":3,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","th_externel_onway_cnt":"3","credit_amt":"10200","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"0.9756","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_pct_meta_20240717":"0.9091","th_model_score_2_meta_20240619":589.58,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":575.01,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251202094127Y1694oq6",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1764643293,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"5","Settlement_frequency_1":"6","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"5","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.91","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"9","geohash_add_time":"1764686517000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"30300.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.4933912037","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"4.5435879630","last_overdue_days_3":"-1","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1764162325104","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jde80s0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"8600","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"8200","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"10200","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.83.112","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"30300.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.06667,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":36,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":7000,\\"pl_loan_5o_user_avg_wait_hours\\":2.16861,\\"pl_loan_7o_phone_max_amt\\":10200,\\"pl_loan_9999d_phone_max_amt\\":10200,\\"pl_loan_9999d_user_max_amt\\":10200,\\"pl_loan_9999d_user_onway_cnt\\":2,\\"pl_repay_30d_phone_ontime_latest_hours\\":14.21122,\\"pl_repay_30d_user_ontime_latest_hours\\":14.21122,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":1345.02789,\\"pl_repay_60d_phone_ontime_earliest_hours\\":1345.02789,\\"pl_repay_60d_phone_ontime_latest_hours\\":14.21122,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":1345.05484,\\"pl_repay_7o_phone_early_amt_all_pct\\":0.03906,\\"pl_repay_90d_phone_early_cnt\\":25,\\"pl_repay_90d_user_early_cnt\\":8,\\"pl_repay_9999d_phone_cnt\\":47,\\"pl_repay_9999d_phone_early_cnt\\":25,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.28868714028718817,\\"th_ios_bcard_2508_score\\":589.698804748818}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"2","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD20251202094202Mu4a3Ot6","age":"30","last_credit_amt":"8600","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1764686522793","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"0.49","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.79340972777808","another_contact_name":"ทัศนาทิพย์","ip":"49.237.83.112","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"5","borrow_count_3":"5","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"6","const_id":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730480522001","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"8600.00","last_contract_amount_1":"10200.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"700.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"8200.00","borrow_count":"16","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"1","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888755100603436","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.8","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","last_overdue_days":"0","ios_battery_percentage":"0.1","borrowing_loan_id":"3,2","emergency_is_others":"0"}',
        "response_data": '{"uuid":"a4b93959-ae4a-4afe-ac6a-f564dcb31ef0","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":456658654265348,"leftValue":"未开启放款包拒绝"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz251202095qw7qZu7","extraInfo":{"_cost_time":343,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"a4b93959-ae4a-4afe-ac6a-f564dcb31ef0,1764686779603,041e90aee7457f57d2be55f93c5fc459"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":8,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":6,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"45","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":2,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":3,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":11200.0,"th_beh_reject_pctb2407_meta_20240717":"0.1778","credit_amt_2":9600.0,"credit_amt_3":9200.0,"Expected_loan_time_1":"1764686522793","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1764686522793","Expected_loan_time_3":"1764686522793","th_app_last_21d_social_update_cnt_meta_20240715":26,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6000","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.8913","th_app_last_1d_biz_update_cnt_meta_20240715":4,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":2,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":3,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","th_externel_onway_cnt":"4","credit_amt":"10200","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":0,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"0.9762","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_pct_meta_20240717":"0.8913","th_model_score_2_meta_20240619":589.58,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":575.01,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251202094202Mu4a3Ot6",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1764686528,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 1,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"6","Settlement_frequency_1":"7","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"6","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"0","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.98","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"10","geohash_add_time":"1765184399000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"7500.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.2453356481","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"3.9824537037","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1764162325104","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jd79gc0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"8600","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"8200","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"11200","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.41.146","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"7500.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0.03704,\\"app_60d_fi_pct\\":0.11111,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":36,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":7500,\\"pl_loan_5o_user_avg_wait_hours\\":6.20759,\\"pl_loan_7o_phone_max_amt\\":11200,\\"pl_loan_9999d_phone_max_amt\\":11200,\\"pl_loan_9999d_user_max_amt\\":11200,\\"pl_loan_9999d_user_onway_cnt\\":0,\\"pl_repay_2o_phone_ontime_latest_hours\\":6.18775,\\"pl_repay_30d_phone_ontime_latest_hours\\":6.18775,\\"pl_repay_30d_user_ontime_latest_hours\\":6.18775,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.78933,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":987.81858,\\"pl_repay_60d_phone_ontime_earliest_hours\\":987.81858,\\"pl_repay_60d_phone_ontime_latest_hours\\":6.18775,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":987.68581,\\"pl_repay_7o_phone_early_amt_all_pct\\":0,\\"pl_repay_90d_phone_early_cnt\\":24,\\"pl_repay_90d_user_early_cnt\\":8,\\"pl_repay_9999d_phone_cnt\\":53,\\"pl_repay_9999d_phone_early_cnt\\":25,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.3047735672758927,\\"th_ios_bcard_2508_score\\":587.4741597014648}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"3","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD202512080400079011sq1H","age":"30","last_credit_amt":"11200","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1765184407622","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"0.25","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.79355178420441","another_contact_name":"ทัศนาทิพย์","ip":"49.237.41.146","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"6","borrow_count_3":"6","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"7","const_id":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730480881006","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"8600.00","last_contract_amount_1":"11200.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"700.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"8200.00","borrow_count":"19","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.887667470623473","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.8","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","last_overdue_days":"0","ios_battery_percentage":"0.25","borrowing_loan_id":"","emergency_is_others":"0"}',
        "response_data": '{"uuid":"419eaff3-0d2f-46c9-a6f7-a528923b1a5b","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":456658654265348,"leftValue":"未开启放款包拒绝"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25120804rql5ykAz","extraInfo":{"_cost_time":330,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"419eaff3-0d2f-46c9-a6f7-a528923b1a5b,1765184098015,fcf271370c83a4dd2e34aeb2f8a8f49f"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":8,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":11,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"42","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":2,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":12200.0,"th_beh_reject_pctb2407_meta_20240717":"0.1633","credit_amt_2":9600.0,"credit_amt_3":9200.0,"Expected_loan_time_1":"1765184407622","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1765184407622","Expected_loan_time_3":"1765184407622","th_app_last_21d_social_update_cnt_meta_20240715":22,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.9231","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.9574","th_app_last_1d_biz_update_cnt_meta_20240715":2,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":2,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":4,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","th_externel_onway_cnt":"1","credit_amt":"11200","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":2,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"0.9792","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_pct_meta_20240717":"0.9592","th_model_score_2_meta_20240619":589.58,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":575.01,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD202512080400079011sq1H",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1765184412,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 0,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"5","Settlement_frequency_1":"6","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"6","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"2","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.90","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"10","geohash_add_time":"1764686557000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"44200.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.2906365741","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"5.9578356481","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1764162325104","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jded2p0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"8600","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"8200","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"11200","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.43.189","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"44200.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0.07407,\\"app_60d_fi_pct\\":0.11111,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":36,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":7500,\\"pl_loan_5o_user_avg_wait_hours\\":6.20759,\\"pl_loan_7o_phone_max_amt\\":11200,\\"pl_loan_9999d_phone_max_amt\\":11200,\\"pl_loan_9999d_user_max_amt\\":11200,\\"pl_loan_9999d_user_onway_cnt\\":2,\\"pl_repay_30d_phone_ontime_latest_hours\\":6.97527,\\"pl_repay_30d_user_ontime_latest_hours\\":6.97527,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":980.49611,\\"pl_repay_60d_phone_ontime_earliest_hours\\":980.49611,\\"pl_repay_60d_phone_ontime_latest_hours\\":6.97527,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":980.36333,\\"pl_repay_7o_phone_early_amt_all_pct\\":0,\\"pl_repay_90d_phone_early_cnt\\":24,\\"pl_repay_90d_user_early_cnt\\":8,\\"pl_repay_9999d_phone_cnt\\":49,\\"pl_repay_9999d_phone_early_cnt\\":25,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.324318007981107,\\"th_ios_bcard_2508_score\\":584.8579621548402}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"3","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD20251208084045X641uH71","age":"30","last_credit_amt":"11200","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1765158046101","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"0.29","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.79338687145487","another_contact_name":"ทัศนาทิพย์","ip":"49.237.43.189","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"5","borrow_count_3":"6","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"6","const_id":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730480880008","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"8600.00","last_contract_amount_1":"11200.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"700.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"8200.00","borrow_count":"17","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.889157670566128","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.8","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","last_overdue_days":"0","ios_battery_percentage":"0.35","borrowing_loan_id":"2,1","emergency_is_others":"0"}',
        "response_data": '{"uuid":"89fa0bc2-2893-4de0-a722-cb6a483a007a","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":456658654265348,"leftValue":"未开启放款包拒绝"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25120808J5n2428M","extraInfo":{"_cost_time":325,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"89fa0bc2-2893-4de0-a722-cb6a483a007a,1765157737340,b811f39c0306c7103a1eef557f3b3509"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":8,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":11,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"36","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":2,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":12000.0,"th_beh_reject_pctb2407_meta_20240717":"0.1702","credit_amt_2":9400.0,"credit_amt_3":9000.0,"Expected_loan_time_1":"1765158046101","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1765158046101","Expected_loan_time_3":"1765158046101","th_app_last_21d_social_update_cnt_meta_20240715":22,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6154","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.8723","th_app_last_1d_biz_update_cnt_meta_20240715":2,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":2,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":4,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","th_externel_onway_cnt":"5","credit_amt":"11200","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":2,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"0.9773","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_pct_meta_20240717":"0.8776","th_model_score_2_meta_20240619":589.58,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":575.01,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD20251208084045X641uH71",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1765158051,
    },
    {
        "app_name": "yofund_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "自然渠道",
        "confirm_cnt": 0,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1758994085,
        "request_data": '{"Settlement_frequency_2":"5","Settlement_frequency_1":"7","Settlement_frequency_4":"0","same_calendarid_dif_idcard_180day":"1","Settlement_frequency_3":"6","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"1","same_calendarid_dif_bankcard_180day":"1","diffbank_face_num":"0","register_time_second":"1758994085","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"102","if_bank_card_modified":"0","current_overdue_face_count":"0","liveness_liveChannel":"brainf","ios_system_version":"18.5","r81_60p70p81_b":"0.92","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","last_all_id_diff_bankcard_num":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","same_gsfid_dif_phone":"-999","last_7d_self_same_face_bankcard_cnt":"-99","emergency_contact_name":"สมับติ","borrow_rank":"10","geohash_add_time":"1765162155000","last_financial_product_id_9":"-999","last_financial_product_id_5":"-999","ios_device_name":"iPhone","last_financial_product_id_6":"-999","current_app_settle_apply_cnt":"3","diffphone_id_num":"0","last_financial_product_id_7":"-999","last_financial_product_id_8":"-999","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"33000.00","ocr_up_type":"1","all_id_diffidnumber_num":"0","if_0_rate":"","same_gaid_dif_phone":"-999","id_expiry_date":"7 Aug. 2030","last81_repaytime_diff_30":"0.0003240741","call_risk_type":"0","last_financial_product_id_1":"000002","id_number_ocr":"00000000001892031979169771522","last_financial_product_id_2":"000002","last_financial_product_id_3":"000002","last_financial_product_id_4":"-999","h5_languages":"[]","app_name":"Wing สินเชื่อสะดวกสบาย","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","first60_borrowtime_diff":"6.0054398148","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","unique_identifier_calendar":"4d31360ad3214011ae7c4967a7e17fef_1764162325104","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w5p8jde81g0","last_overdue_days_5":"0","other_contact_name":"","geo_hash_48h_100meter_apply_cnt_co":"0","last_credit_amt_2":"8600","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"8200","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"11200","h5_browserName":"","last_7d_self_same_face_id_cnt":"-99","personal_city":"792","client_ip":"49.237.43.189","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"000002","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","campaign":"","phone_number":"00000000001865571048323723445","user_contract_amount_sum":"33000.00","fea_list":"{\\"age_job_monthly_income\\":0,\\"app_3d_fi_pct\\":0.07407,\\"app_60d_fi_pct\\":0.11111,\\"app_9999d_fi_效率_cnt\\":5,\\"gender_job_monthly_income\\":0,\\"industry\\":0,\\"ios_model\\":36,\\"job_monthly_income\\":0,\\"pl_loan_1o_phone_max_amt\\":7500,\\"pl_loan_5o_user_avg_wait_hours\\":6.20759,\\"pl_loan_7o_phone_max_amt\\":11200,\\"pl_loan_9999d_phone_max_amt\\":11200,\\"pl_loan_9999d_user_max_amt\\":11200,\\"pl_loan_9999d_user_onway_cnt\\":1,\\"pl_repay_2o_phone_ontime_latest_hours\\":0.00784,\\"pl_repay_30d_phone_ontime_latest_hours\\":0.00784,\\"pl_repay_30d_user_ontime_latest_hours\\":0.00784,\\"pl_repay_3o_phone_early_cnt\\":0,\\"pl_repay_3o_phone_s-1_cnt\\":0,\\"pl_repay_4o_phone_ontime_amt_pct\\":0.31461,\\"pl_repay_5o_phone_s-1_cnt\\":0,\\"pl_repay_5o_phone_s-1_pct\\":0,\\"pl_repay_60d_phone_early_earliest_hours\\":981.63868,\\"pl_repay_60d_phone_ontime_earliest_hours\\":981.63868,\\"pl_repay_60d_phone_ontime_latest_hours\\":0.00784,\\"pl_repay_60d_user_latest_overdue_days\\":0,\\"pl_repay_60d_user_ontime_earliest_hours\\":981.5059,\\"pl_repay_7o_phone_early_amt_all_pct\\":0,\\"pl_repay_90d_phone_early_cnt\\":24,\\"pl_repay_90d_user_early_cnt\\":8,\\"pl_repay_9999d_phone_cnt\\":50,\\"pl_repay_9999d_phone_early_cnt\\":25,\\"pl_repay_9999d_user_latest_overdue_days\\":0,\\"pure_new_tag\\":0,\\"th_ios_bcard_2508_prob\\":0.3261882173021715,\\"th_ios_bcard_2508_score\\":584.6120764399889}","first_borrow_whitelist_in":"2","same_calendarid_dif_phone_180day":"1","personal_province":"788","wl_uid":"None","age_revise":"30","same_calendarid_dif_bankcard":"1","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001963815638851346433","front_user_label":"","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"2","if_id_number_modified":"0","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"-99","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"1","wl_amount":"None","request_id":"EDD202512080949196fw6Nt5R","age":"30","last_credit_amt":"11200","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1765162159641","ph_diffzalo_num":"3","h5_hardwareConcurrency":"","last_settlement_order_difference":"0.00","last_7d_all_same_face_phone_cnt":"-99","emergency_contact_phone":"0958798467","h5_maxTouchPoints":"","lng":"100.79346537749262","another_contact_name":"ทัศนาทิพย์","ip":"49.237.43.189","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"4d31360ad3214011ae7c4967a7e17fef","other_contact_phone":"","current_status":"1","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"5","borrow_count_3":"6","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","type_telemarketing_list":"-99","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"-99","borrow_count_9":"0","ios_rooted":"false","user_max_appliacation_limit":"3","if_dk_event":"1","face_similarity":"89.03","borrow_count_1":"7","const_id":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","liveness_livenessScore":"99.98","last_contract_amount_10":"0","appointment_category":"","diffphone_face_num":"0","borrow_create_time":"1730480880009","personal_children_nums":"None","job_monthly_income":"None","id_diffidnumber_num":"0","channel_name":"Organic","h5_deviceMemory":"","last_contract_amount_2":"8600.00","last_contract_amount_1":"11200.00","same_calendarid_dif_idcard":"1","first_credit_amount_8":"0","h5_product":"","first_credit_amount_9":"0","churn_type":"0","current_app_overdue_order_cnt":"0","first_credit_amount_2":"700.00","last_7d_self_same_face_apply_cnt":"-99","first_credit_amount_3":"500.00","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","first_credit_amount_1":"3200.00","first_credit_amount_6":"0","first_credit_amount_7":"0","first_credit_amount_4":"0","first_credit_amount_5":"0","id_birthday_year":"1995","is_organic":"1","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","geo_hash_7d_100meter_apply_cnt":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"8200.00","borrow_count":"18","geo_hash_30d_100meter_overdue_user_cnt":"0","last_7d_all_same_face_id_cnt":"-99","platform_check_baiduai_black":"-99","last_financial_product_id_10":"-999","overdue_days_max_30":"0","first_credit_amount_10":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.888749559647","if_live_detection":"1","automatic_face_pass":"0","same_calendarid_dif_phone":"1","ios_app_version":"1.0.8","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"None","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"-99","bankcard_code":"KTB","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"9818DDCC-3E44-4F3A-A5FA-85B9EF329143","h5_language":"","h5_productSub":"","h5_appName":"Wing สินเชื่อสะดวกสบาย","personal_education":"","ios_app_package":"com.youfind.bigcash.mobile","last_overdue_days":"0","ios_battery_percentage":"0.15","borrowing_loan_id":"2","emergency_is_others":"0"}',
        "response_data": '{"uuid":"761a93b9-ae63-4e20-90f3-8835bede9c00","status":"SUCCESS","ctuEntireResult":{"riskLevel":"ACCEPT","riskType":"","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":456658654265348,"leftValue":"未开启放款包拒绝"},{"id":382197500543050,"leftValue":"异常校验"},{"id":382197500543059,"leftValue":"复贷异常用户处理"},{"id":382197498446347,"leftValue":"异常校验"},{"id":382197498446130,"leftValue":"异常校验"},{"id":382197502640507,"leftValue":"泰国_IOS_复贷_当前逾期人脸_Meta_20240620"},{"id":382197502640520,"leftValue":"泰国_IOS_复贷_其他平台当前逾期数量_Meta_20240620"},{"id":382197502640533,"leftValue":"泰国_IOS_复贷_云服务黑名单_Meta_20240620"},{"id":389414277545991,"leftValue":"泰国_IOS_复贷_人脸黑名单_Meta_20241119"},{"id":382197494251535,"leftValue":"泰国_IOS_复贷_非正常安装程序_Meta_20240620"},{"id":382197494251548,"leftValue":"泰国_IOS_复贷_越狱_Meta_20240620"},{"id":405023739543597,"leftValue":"电销欺诈"},{"id":432743791919109,"leftValue":"中介窝点风险"},{"id":433108941733922,"leftValue":"同geohash不同手机号"},{"id":382197498446030,"leftValue":"泰国_IOS_复贷_当前在途首复贷_Meta_20240620"},{"id":418788321198083,"leftValue":"泰国_IOS_复贷_当前在途首金额"},{"id":432349042900996,"leftValue":"泰国_IOS_复贷_当前在途首复贷_人审"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197498446096,"leftValue":"异常校验"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25120809gC9L1tDC","extraInfo":{"_cost_time":319,"_model_eval_error":"32:java.lang.NumberFormatException: empty String","riskId":"761a93b9-ae63-4e20-90f3-8835bede9c00,1765162073741,1acb2373320f49027cf149124aa728cd"},"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":0.0,"th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":8,"his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":"","th_app_last_1d_fin_update_cnt_b2407_meta_20240715":11,"th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"39","th_ab_model_version_meta_20240628":"","reject_days":"0","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":2,"th_app_last_7d_food_update_cnt_b2407_meta_20240715":0,"bj_ios_model_score":-1.0,"is_error":"0","credit_amt_1":12000.0,"th_beh_reject_pctb2407_meta_20240717":"0.1667","credit_amt_2":9400.0,"credit_amt_3":9000.0,"Expected_loan_time_1":"1765162159641","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":"","Expected_loan_time_2":"1765162159641","Expected_loan_time_3":"1765162159641","th_app_last_21d_social_update_cnt_meta_20240715":22,"lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":"","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"0.6923","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"0.8936","th_app_last_1d_biz_update_cnt_meta_20240715":2,"th_app_3m_highrisk_install_cnt_meta_20240619":"","financial_product_id_1":"000002","th_model_bscore_meta_20240719":0,"th_ios_acard_2509_score":-1.0,"financial_product_id_3":"000002","financial_product_id_2":"000002","th_ios_frist_model_jixia_20250321":"2503","th_externel_onway_cnt_meta_20241018":0.0,"th_app_3m_lowrisk_update_cnt_meta_20240619":"","th_app_last_21d_highrisk_update_cnt_meta_20240715":2,"th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":"","th_app_loan_earlist_install_days_b2407_meta_20240715":0,"risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":4,"th_app_last_3m_highrisk_install_cnt_meta_20240715":1,"th_app_3m_highrisk_install_cnt_a2405_meta_20240619":"","job_monthly_income":"None","th_externel_onway_cnt":"4","credit_amt":"11200","is_advertising":"0","personal_education":"","th_app_last_30d_install_cnt_meta_20240715":2,"th_app_loan_latest_install_days_a2405_meta_20240619":"","th_app_last_7d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_per_repay_b2407_meta_20240717":"0.9778","appliacation_limit":"3","th_reject_recycle_tag":"","model_type":"2403","th_ios_device_model_jixia_20250321":"","th_model_bversion_meta_20240719":"b2407","th_app_last_3d_highrisk_update_cnt_meta_20240715":2,"th_beh_ontime_repay_pct_meta_20240717":"0.8980","th_model_score_2_meta_20240619":589.58,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":"","th_ios_first_model_score_2503_jixia_20250321":605.61,"ABtest":"实验组","overdue_detail":"0","model_score":575.01,"th_app_last_30d_pe_update_cnt_meta_20240715":1}}}',
        "risk_level": "ACCEPT",
        "risk_type": "",
        "sn": "EDD202512080949196fw6Nt5R",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
        "verify_time": 1765162164,
    },
    {
        "app_name": "zedwallet_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 0,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1740804309,
        "request_data": '{"Settlement_frequency_2":"0","Settlement_frequency_1":"0","Settlement_frequency_4":"0","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","diffbank_face_num":"0","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"40","current_overdue_face_count":"0","liveness_liveChannel":"brainfH5","ios_system_version":"18.3.1","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","last_7d_self_same_face_bankcard_cnt":"0","geohash_add_time":"1740804350000","ios_device_name":"iPhone","diffphone_id_num":"0","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"0","ocr_up_type":"1","id_expiry_date":"02-08-2030","call_risk_type":"0","id_number_ocr":"00000000001892031979169771522","h5_languages":"[]","app_name":"kale lending","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w4x6ct8szg0","last_overdue_days_5":"0","last_credit_amt_2":"0","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"0","h5_browserName":"","last_7d_self_same_face_id_cnt":"0","personal_city":"792","client_ip":"49.48.242.152","h5_platform":"","geohash_type":"riskengine","last_financial_product_id":"None","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","phone_number":"00000000001865571048323723445","fea_list":"{\\"app_14d_fi_工具_trend\\":0,\\"app_9999d_fi_cnt\\":56,\\"calllog_14d_tag_移动电话_time_pct\\":0,\\"calllog_9999d_frequent_phone_cnt\\":0,\\"sms_14d_cashin_sum\\":0,\\"sms_14d_贷款_phone_pct\\":0,\\"sms_14d_贷款_审核通知_pct\\":0,\\"sms_14d_赌场_cnt\\":0,\\"sms_21d_21d_营销_pct\\":0,\\"sms_21d_贷款_审核通知_pct\\":0,\\"sms_30d_30d_营销_distinct_pct\\":0,\\"sms_30d_phone_cnt\\":0,\\"sms_30d_贷款_all_pct\\":0,\\"sms_7d_7d_贷款_pct\\":0,\\"sms_7d_贷款_phone_pct\\":0,\\"sms_7d_贷款_审核通知_pct\\":0,\\"sms_7d_金融类_cnt\\":0,\\"sms_9999d_营销_cnt\\":0,\\"sms_9999d_营销_distinct_cnt\\":0,\\"sms_9999d_贷款_营销_cnt\\":0,\\"sms_赌场_pct\\":0}","user_contract_amount_sum":"0","first_borrow_whitelist_in":"0","personal_province":"788","wl_uid":"None","age_revise":"29","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"0","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD202503011145535AKnFhU2","age":"29","last_credit_amt":"0","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1740804353634","ph_diffzalo_num":"1","h5_hardwareConcurrency":"","last_settlement_order_difference":"-1","last_7d_all_same_face_phone_cnt":"0","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.26225928568358","ip":"49.48.242.152","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"80698eb4b95945ef945fb96234a62b5c","other_contact_phone":"0800699457","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"0","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"0","borrow_count_9":"0","ios_rooted":"false","if_dk_event":"1","face_similarity":"86.0","borrow_count_1":"0","const_id":"FCC2FF75-46FC-4582-9ACD-7F717DBC823A","liveness_livenessScore":"99.92","last_contract_amount_10":"0","diffphone_face_num":"0","borrow_create_time":"1733158861001","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"0","last_contract_amount_1":"0","h5_product":"","last_7d_self_same_face_apply_cnt":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"0","last_7d_all_same_face_id_cnt":"0","platform_check_baiduai_black":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"14.576946861968658","if_live_detection":"1","automatic_face_pass":"0","ios_app_version":"1.0.1","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"0","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"FCC2FF75-46FC-4582-9ACD-7F717DBC823A","h5_language":"th-TH","h5_productSub":"","h5_appName":"Kale Lending","personal_education":"7","ios_app_package":"com.zed.finance.cash","ios_battery_percentage":"0.7","borrowing_loan_id":""}',
        "response_data": '{"uuid":"aab7ff31-f8ab-4ffb-8a05-7d543cc3b302","status":"SUCCESS","ctuEntireResult":{"riskLevel":"REJECT","riskType":"concatnum","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197498446007,"leftValue":"异常校验"},{"id":382197500543014,"leftValue":"异常校验"},{"id":382197502640796,"leftValue":"异常校验"},{"id":382197500543135,"leftValue":"泰国_IOS_首贷_纯新户_当前逾期人脸_Meta_20240620"},{"id":382197500543148,"leftValue":"泰国_IOS_首贷_纯新户_其他平台逾期数量_Meta_20240620"},{"id":382197500543161,"leftValue":"泰国_IOS_首贷_纯新户_活体检测分_Meta_20240620"},{"id":382197500543194,"leftValue":"泰国_IOS_首贷_纯新户_人脸相似度_Meta_20240620"},{"id":382197500543204,"leftValue":"泰国_IOS_首贷_纯新户_有效联系人数量_Meta_20240620"},{"id":382197500543217,"leftValue":"泰国_IOS_首贷_纯新户_年龄_Meta_20240620"},{"id":382197500543234,"leftValue":"泰国_IOS_首贷_纯新户_人脸黑名单_Meta_20240620"},{"id":382197500543247,"leftValue":"泰国_IOS_首贷_纯新户_外部黑名单_Meta_20240620"},{"id":382197500543260,"leftValue":"泰国_IOS_首贷_纯新户_云服务黑名单_Meta_20240620"},{"id":393600954204183,"leftValue":"ocr年龄无法识别"},{"id":405893862588439,"leftValue":"客服人员准入"},{"id":382197502640260,"leftValue":"泰国_IOS_首贷_非正常安装程序_Meta_20240620"},{"id":382197502640273,"leftValue":"泰国_IOS_首贷_越狱_Meta_20240620"},{"id":382197502640288,"leftValue":"泰国_IOS_首贷_纯新户_历史拒绝次数_Meta_20240621"},{"id":382197502640303,"leftValue":"泰国_IOS_首贷_纯新户_onelink_Meta_20240621"},{"id":382197502640318,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同姓名"},{"id":382197502640344,"leftValue":"泰国_IOS_首贷_纯新户_同银行卡不同姓名"},{"id":382197502640357,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同姓名"},{"id":382197502640370,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同line"},{"id":382197502640383,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同银行卡"},{"id":382197502640396,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同银行卡"},{"id":382197502640411,"leftValue":"泰国_IOS_首贷_纯新户_通讯录相似度_Meta_20240621"},{"id":382197502640424,"leftValue":"泰国_IOS_首贷_纯新户_APP相似度_Meta_20240621"},{"id":382197502640439,"leftValue":"泰国_IOS_首贷_纯新户_同人脸不同手机号_Meta_20240621"},{"id":405023737446534,"leftValue":"电销欺诈"},{"id":382197500543604,"leftValue":"泰国_IOS_首贷_纯新户_当前在途订单_Meta_20240620"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197500543313,"leftValue":"应用1额度赋值"},{"id":382197498446054,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25030111om1f72MR","extraInfo":{"_observe_mode":[false],"_cost_time":103,"ruleIdCodeNamePair":"[{\\"code\\":\\"rule2\\",\\"id\\":406482220679187,\\"name\\":\\"购物券包\\"},{\\"code\\":\\"NAdmission05\\",\\"id\\":382197500543204,\\"name\\":\\"泰国_IOS_首贷_纯新户_有效联系人数量_Meta_20240620\\"},{\\"code\\":\\"ASSIGNCREDIT\\",\\"id\\":382197500543313,\\"name\\":\\"应用1额度赋值\\"}]","riskId":"aab7ff31-f8ab-4ffb-8a05-7d543cc3b302,1740804467692,9a9c492faf481221b93a614405d95d85"},"hitPolicies":[{"id":382197500543324,"name":"泰国_IOS贷超_前处理_公共变量_赋值"},{"id":382197500543117,"name":"泰国_IOS贷超_硬规则_首贷_纯新户_准入"},{"id":382197500543300,"name":"泰国_IOS贷超_后处理_变量返回_基础变量"}],"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":"","th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":"","his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":562,"th_app_last_1d_fin_update_cnt_b2407_meta_20240715":"","th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"","th_ab_model_version_meta_20240628":"2403","reject_days":"3","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":"","th_app_last_7d_food_update_cnt_b2407_meta_20240715":"","is_error":"0","credit_amt_1":"3500","th_beh_reject_pctb2407_meta_20240717":"","Expected_loan_time_1":"1740804353634","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":1,"th_app_last_21d_social_update_cnt_meta_20240715":"","lastoverdue_day":"0","coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"","th_app_last_1d_biz_update_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_meta_20240619":0,"financial_product_id_1":"000003","th_model_bscore_meta_20240719":"","th_app_3m_lowrisk_update_cnt_meta_20240619":4,"th_app_last_21d_highrisk_update_cnt_meta_20240715":"","th_model_version_2_meta_20240619":"2405","th_app_loan_earlist_install_days_b2407_meta_20240715":"","risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":"","th_app_last_3m_highrisk_install_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_a2405_meta_20240619":0,"credit_amt":"3500","is_advertising":"1","th_app_last_30d_install_cnt_meta_20240715":"","th_app_loan_latest_install_days_a2405_meta_20240619":8,"th_app_last_7d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_per_repay_b2407_meta_20240717":"","appliacation_limit":"1","model_type":"2403","th_model_bversion_meta_20240719":"","th_app_last_3d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_pct_meta_20240717":"","th_model_score_2_meta_20240619":612.46,"th_device_ios_model_meta_20240619":4,"overdue_detail":"0","model_score":620.78,"th_app_last_30d_pe_update_cnt_meta_20240715":""}}}',
        "risk_level": "REJECT",
        "risk_type": "concatnum",
        "sn": "EDD202503011145535AKnFhU2",
        "user_id": "80698eb4b95945ef945fb96234a62b5c",
        "verify_time": 1740804359,
    },
    {
        "app_name": "zedwallet_ios",
        "bank_no": "00000000001892031273191284738",
        "channel_name": "facebook",
        "confirm_cnt": 0,
        "human_verify_status": None,
        "idCard_no": "00000000001892031979169771522",
        "is_dk": 1,
        "onway_cnt": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "register_time": 1740804309,
        "request_data": '{"Settlement_frequency_2":"0","Settlement_frequency_1":"0","Settlement_frequency_4":"0","Settlement_frequency_3":"0","Settlement_frequency_6":"0","Settlement_frequency_5":"0","Settlement_frequency_8":"0","ios_model":"iPhone15,3","Settlement_frequency_7":"0","Settlement_frequency_10":"0","h5_ip":"","borrowing_loan_num":"0","diffbank_face_num":"0","diffid_face_num":"0","ios_manufacturer":"apple","current_overdue_face_sn":"None","platform_black":"0","h5_appCodeName":"","app_id":"40","current_overdue_face_count":"0","liveness_liveChannel":"brainfH5","ios_system_version":"18.3.1","wl_msg":"None","diffid_phone_num":"0","Settlement_frequency_9":"0","another_contact_phone":"0614797649","user_name":"นาย จิรวัฒน์ เสือวงษ์","h5_battery":"{\\"h5_charging\\":\\"\\",\\"h5_chargingTime\\":\\"\\",\\"h5_dischargingTime\\":\\"\\",\\"h5_level\\":\\"\\"}","wl_result":"None","if_qualification_review":"1","last_7d_self_same_face_bankcard_cnt":"0","geohash_add_time":"1743122903000","ios_device_name":"iPhone","diffphone_id_num":"0","platform_communication":"0","h5_userAgent":"","email":"","user_due_amount_sum":"0","ocr_up_type":"1","if_0_rate":"","id_expiry_date":"02-08-2030","call_risk_type":"0","id_number_ocr":"00000000001892031979169771522","h5_languages":"[]","app_name":"KaleLending_ios","last_overdue_days_8":"0","last_overdue_days_9":"0","last_7d_self_same_contact_name_phone_cnt":"0","last_overdue_days_2":"0","last_overdue_days_3":"0","diffbank_id_num":"0","last_overdue_days_1":"0","h5_appVersion":"","last_overdue_days_6":"0","last_overdue_days_7":"0","last_overdue_days_4":"0","geo_hash":"w4zr9j8uss0","last_overdue_days_5":"0","last_credit_amt_2":"0","bankcard_number":"00000000001892031273191284738","last_credit_amt_3":"0","last_7d_all_same_appname_diff_user_cnt":"0","last_credit_amt_1":"0","h5_browserName":"","last_7d_self_same_face_id_cnt":"0","personal_city":"792","client_ip":"49.237.37.145","h5_platform":"","face_similarity_channel":"brainf","geohash_type":"riskengine","last_financial_product_id":"None","bank_diffname_num":"1","wl_score2":"None","wl_score1":"None","diffbank_phone_num":"0","phone_number":"00000000001865571048323723445","fea_list":"{\\"app_3d_fi_pct\\":0,\\"app_60d_fi_pct\\":0.10526,\\"app_9999d_fi_效率_cnt\\":3}","user_contract_amount_sum":"0","first_borrow_whitelist_in":"0","personal_province":"788","wl_uid":"None","age_revise":"29","id_diffname_num":"1","wl_pron_no":"None","zalo":"00000000001892031235892965378","front_user_label":"None","h5_screen":"{\\"h5_available_height\\":\\"\\",\\"h5_available_left\\":\\"\\",\\"h5_available_top\\":\\"\\",\\"h5_available_width\\":\\"\\",\\"h5_colorDepth\\":\\"\\",\\"h5_devicePixelRatio\\":\\"\\",\\"h5_height\\":\\"\\",\\"h5_innerHeight\\":\\"\\",\\"h5_innerWidth\\":\\"\\",\\"h5_keepAwake\\":\\"\\",\\"h5_maxTouchPoints\\":\\"\\",\\"h5_orientation_angle\\":\\"\\",\\"h5_orientation_type\\":\\"\\",\\"h5_pixelDepth\\":\\"\\",\\"h5_width\\":\\"\\"}","loan_id":"0","ios_charging_status":"1","whitelist_type":"0","h5_vendorSub":"","last_credit_amt_6":"0","last_credit_amt_7":"0","last_credit_amt_4":"0","last_credit_amt_5":"0","last_7d_all_same_face_apply_cnt":"0","id_birthday_day":"8","last_credit_amt_8":"0","last_credit_amt_9":"0","personal_loan_purpose":"4","wl_amount":"None","request_id":"EDD20250328074825tCsHdvqY","age":"29","last_credit_amt":"0","h5_vendor":"","idnumber_device_num":"0","last_7d_same_device_name_model_version_cnt":"0","verify_time":"1743122905944","ph_diffzalo_num":"1","h5_hardwareConcurrency":"","last_settlement_order_difference":"-1","last_7d_all_same_face_phone_cnt":"0","emergency_contact_phone":"0819526416","h5_maxTouchPoints":"","lng":"100.2404818364593","ip":"49.237.37.145","last_credit_amt_10":"0","ph_diffmail_num":"0","last_7d_self_same_appname_cnt":"0","user_id":"80698eb4b95945ef945fb96234a62b5c","other_contact_phone":"0800699457","current_status":"0","wl_features":"None","last_30d_all_same_appname_cnt":"0","borrow_count_2":"0","borrow_count_3":"0","borrow_count_4":"0","borrow_count_5":"0","diffid_bank_num":"0","gender":"1","borrow_count_6":"0","borrow_count_7":"0","borrow_count_8":"0","last_7d_all_same_face_bankcard_cnt":"0","borrow_count_9":"0","ios_rooted":"false","if_dk_event":"1","face_similarity":"86.0","borrow_count_1":"0","const_id":"FCC2FF75-46FC-4582-9ACD-7F717DBC823A","liveness_livenessScore":"99.92","last_contract_amount_10":"0","diffphone_face_num":"0","borrow_create_time":"1733152080007","personal_children_nums":"1","job_monthly_income":"8","id_diffidnumber_num":"0","channel_name":"facebook","h5_deviceMemory":"","last_contract_amount_2":"0","last_contract_amount_1":"0","h5_product":"","last_7d_self_same_face_apply_cnt":"0","user_name_revise":"นาย จิรวัฒน์ เสือวงษ์","id_birthday_year":"1995","is_organic":"0","last_contract_amount_9":"0","phone_device_num":"0","last_contract_amount_8":"0","borrow_count_10":"0","last_contract_amount_7":"0","last_contract_amount_6":"0","last_contract_amount_5":"0","last_contract_amount_4":"0","last_contract_amount_3":"0","borrow_count":"0","last_7d_all_same_face_id_cnt":"0","platform_check_baiduai_black":"0","wl_order_id":"None","ph_diffname_num":"1","id_birthday_month":"8","lat":"16.818118582644043","if_live_detection":"1","automatic_face_pass":"0","ios_app_version":"1.0.1","diffphone_bank_num":"0","same_id_number_diff_phone_count_appid_30":"0","personal_area":"713","applier_is_aim_blacklist":"0","job":"1","h5_reqUserSource":"3","wl_score":"None","id_card":"00000000001892031979169771522","last_7d_self_same_face_phone_cnt":"0","platform":"Ios","id_address":"None","last_overdue_days_10":"0","ios_device_no":"FCC2FF75-46FC-4582-9ACD-7F717DBC823A","h5_language":"th-TH","h5_productSub":"","h5_appName":"Kale Lending","personal_education":"7","ios_app_package":"com.zed.finance.cash","ios_battery_percentage":"1","borrowing_loan_id":""}',
        "response_data": '{"uuid":"8d52768a-0ab3-415c-bff2-47ee08da5471","status":"SUCCESS","ctuEntireResult":{"riskLevel":"REJECT","riskType":"th_fraud_his_reject_meta_20240621","hitRules":[{"id":382197500543350,"leftValue":"异常校验"},{"id":405047798071317,"leftValue":"白名单_投放返回"},{"id":406482220679187,"leftValue":"购物券包"},{"id":382197498446007,"leftValue":"异常校验"},{"id":382197500543014,"leftValue":"异常校验"},{"id":411574860709947,"leftValue":"异常校验"},{"id":382197502640796,"leftValue":"异常校验"},{"id":382197500543135,"leftValue":"泰国_IOS_首贷_纯新户_当前逾期人脸_Meta_20240620"},{"id":382197500543148,"leftValue":"泰国_IOS_首贷_纯新户_其他平台逾期数量_Meta_20240620"},{"id":382197500543161,"leftValue":"泰国_IOS_首贷_纯新户_活体检测分_Meta_20240620"},{"id":382197500543194,"leftValue":"泰国_IOS_首贷_纯新户_人脸相似度_Meta_20240620"},{"id":382197500543217,"leftValue":"泰国_IOS_首贷_纯新户_年龄_Meta_20240620"},{"id":382197500543234,"leftValue":"泰国_IOS_首贷_纯新户_人脸黑名单_Meta_20240620"},{"id":382197500543247,"leftValue":"泰国_IOS_首贷_纯新户_外部黑名单_Meta_20240620"},{"id":382197500543260,"leftValue":"泰国_IOS_首贷_纯新户_云服务黑名单_Meta_20240620"},{"id":393600954204183,"leftValue":"ocr年龄无法识别"},{"id":405893862588439,"leftValue":"客服人员准入"},{"id":382197502640260,"leftValue":"泰国_IOS_首贷_非正常安装程序_Meta_20240620"},{"id":382197502640273,"leftValue":"泰国_IOS_首贷_越狱_Meta_20240620"},{"id":382197502640288,"leftValue":"泰国_IOS_首贷_纯新户_历史拒绝次数_Meta_20240621"},{"id":382197502640303,"leftValue":"泰国_IOS_首贷_纯新户_onelink_Meta_20240621"},{"id":382197502640318,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同姓名"},{"id":382197502640344,"leftValue":"泰国_IOS_首贷_纯新户_同银行卡不同姓名"},{"id":382197502640357,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同姓名"},{"id":382197502640370,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同line"},{"id":382197502640383,"leftValue":"泰国_IOS_首贷_纯新户_同手机不同银行卡"},{"id":382197502640396,"leftValue":"泰国_IOS_首贷_纯新户_同身份证不同银行卡"},{"id":382197502640411,"leftValue":"泰国_IOS_首贷_纯新户_通讯录相似度_Meta_20240621"},{"id":382197502640424,"leftValue":"泰国_IOS_首贷_纯新户_APP相似度_Meta_20240621"},{"id":382197502640439,"leftValue":"泰国_IOS_首贷_纯新户_同人脸不同手机号_Meta_20240621"},{"id":405023737446534,"leftValue":"电销欺诈"},{"id":382197500543604,"leftValue":"泰国_IOS_首贷_纯新户_当前在途订单_Meta_20240620"},{"id":382197500543369,"leftValue":"异常校验"},{"id":382197500543313,"leftValue":"应用1额度赋值"},{"id":382197498446054,"leftValue":"异常校验"},{"id":411574860709899,"leftValue":"异常校验"},{"id":382197500543103,"leftValue":"泰国_IOS_全量_最近一笔订单结清金额异常_Meta_20240618"}],"flag":"biz25032807CnMkiSt7","extraInfo":{"_observe_mode":[false],"_cost_time":109,"ruleIdCodeNamePair":"[{\\"code\\":\\"rule2\\",\\"id\\":406482220679187,\\"name\\":\\"购物券包\\"},{\\"code\\":\\"NBev01\\",\\"id\\":382197502640288,\\"name\\":\\"泰国_IOS_首贷_纯新户_历史拒绝次数_Meta_20240621\\"},{\\"code\\":\\"ASSIGNCREDIT\\",\\"id\\":382197500543313,\\"name\\":\\"应用1额度赋值\\"}]","riskId":"8d52768a-0ab3-415c-bff2-47ee08da5471,1743123034451,86099f8829bebc2a2438ed2109cea4f7"},"hitPolicies":[{"id":382197500543324,"name":"泰国_IOS贷超_前处理_公共变量_赋值"},{"id":382197502640241,"name":"泰国_IOS贷超_硬规则_首贷_纯新户_反欺诈"},{"id":382197500543300,"name":"泰国_IOS贷超_后处理_变量返回_基础变量"}],"resultVariable":{"th_person_age_edu_income_b2407_meta_20240717":"","th_app_last_1d_lowrisk_update_cnt_b2407_meta_20240715":"","his_nooverdue_1day_cnt_all":"","th_app_bank_earlist_install_days_a2405_meta_20240619":589,"th_app_last_1d_fin_update_cnt_b2407_meta_20240715":"","th_beh_last_1m_weekday_apply_cnt_b2407_meta_20240717":"","th_ab_model_version_meta_20240628":"2405","reject_days":"30","th_ab_again_tag_meta_20240822":"","th_app_last_7d_shop_update_cnt_b2407_meta_20240715":"","th_app_last_7d_food_update_cnt_b2407_meta_20240715":"","is_error":"0","credit_amt_1":"1500","th_beh_reject_pctb2407_meta_20240717":"","Expected_loan_time_1":"1743122905944","th_other_traffic_tag_meta_20240821":27,"th_app_1m_loan_update_cnt_meta_20240619":1,"th_app_last_21d_social_update_cnt_meta_20240715":"","lastoverdue_day":"0","th_ios_app_efficiency_ins_cnt_jixia_20250321":3.0,"coupon_type":"0","th_beh_last_15d_ontime_repay_pct_b2407_meta_20240717":"","th_beh_last_90d_ontime_repay_pct_b2407_meta_20240717":"","th_app_last_1d_biz_update_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_meta_20240619":0,"financial_product_id_1":"000003","th_model_bscore_meta_20240719":"","th_ios_frist_model_jixia_20250321":"2503","th_app_3m_lowrisk_update_cnt_meta_20240619":4,"th_app_last_21d_highrisk_update_cnt_meta_20240715":"","th_model_version_2_meta_20240619":"2405","th_ios_app_3d_ins_pct_jixia_20250321":0.0,"th_app_loan_earlist_install_days_b2407_meta_20240715":"","risk_validity_period":"7200","th_app_last_30d_life_update_cnt_meta_20240715":"","th_app_last_3m_highrisk_install_cnt_meta_20240715":"","th_app_3m_highrisk_install_cnt_a2405_meta_20240619":0,"job_monthly_income":"8","credit_amt":"1500","is_advertising":"1","personal_education":"7","th_app_last_30d_install_cnt_meta_20240715":"","th_app_loan_latest_install_days_a2405_meta_20240619":35,"th_app_last_7d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_per_repay_b2407_meta_20240717":"","appliacation_limit":"1","model_type":"2403","th_ios_device_model_jixia_20250321":36,"th_model_bversion_meta_20240719":"","th_app_last_3d_highrisk_update_cnt_meta_20240715":"","th_beh_ontime_repay_pct_meta_20240717":"","th_model_score_2_meta_20240619":612.46,"th_device_ios_model_meta_20240619":4,"th_ios_app_60d_ins_pct_jixia_20250321":0.10526,"th_ios_first_model_score_2503_jixia_20250321":596.79,"overdue_detail":"0","model_score":620.78,"th_app_last_30d_pe_update_cnt_meta_20240715":""}}}',
        "risk_level": "REJECT",
        "risk_type": "th_fraud_his_reject_meta_20240621",
        "sn": "EDD20250328074825tCsHdvqY",
        "user_id": "80698eb4b95945ef945fb96234a62b5c",
        "verify_time": 1743122912,
    },
]

In [170]:
post_loan_data_lst = [
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 3500,
        "borrow_count": 1,
        "borrow_id": "BOR202509171455013OM8aS",
        "contract_id": "C13202509171455011522J4yCE",
        "contract_status": "一般结清",
        "expect_repay_time": 1758600583,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 2100,
        "is_force": "1",
        "loan_time": 1758082183,
        "order_create_time": 1758095702,
        "order_flag": "展期后补齐",
        "order_id": "956774",
        "order_type": "复贷展期",
        "overdue_days": -6,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 2050,
        "repay_cnt": 1,
        "repay_time": 1758121501,
        "root_borrow_id": "BOR202509051107278I7jTZ",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 500,
        "borrow_count": 1,
        "borrow_id": "BOR20251030164621F66T1L",
        "contract_id": "C13202510301646221522u97Ur",
        "contract_status": "一般结清",
        "expect_repay_time": 1762335988,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 300,
        "is_force": "0",
        "loan_time": 1761817588,
        "order_create_time": 1761817582,
        "order_flag": "正常订单",
        "order_id": "1176024",
        "order_type": "复贷",
        "overdue_days": -6,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 500,
        "repay_cnt": 1,
        "repay_time": 1761822186,
        "root_borrow_id": "BOR20251030164621F66T1L",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 3200,
        "borrow_count": 1,
        "borrow_id": "BOR20250928002932ctkk29",
        "contract_id": "C13202509280029331522q02ji",
        "contract_status": "一般结清",
        "expect_repay_time": 1759512580,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 1920,
        "is_force": "0",
        "loan_time": 1758994180,
        "order_create_time": 1758994173,
        "order_flag": "正常订单",
        "order_id": "1007690",
        "order_type": "首贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 3200,
        "repay_cnt": 1,
        "repay_time": 1759564621,
        "root_borrow_id": "BOR20250928002932ctkk29",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 4900,
        "borrow_count": 4,
        "borrow_id": "BOR20251126133454lG2siU",
        "contract_id": "C13202511261334541522275f7",
        "contract_status": "一般结清",
        "expect_repay_time": 1764657299,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 2940,
        "is_force": "0",
        "loan_time": 1764138899,
        "order_create_time": 1764138895,
        "order_flag": "正常订单",
        "order_id": "1321646",
        "order_type": "复贷",
        "overdue_days": -6,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 4900,
        "repay_cnt": 1,
        "repay_time": 1764170228,
        "root_borrow_id": "BOR20251126133454lG2siU",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 2000,
        "borrow_count": 1,
        "borrow_id": "BOR20250905110727kXE52d",
        "contract_id": "C13202509051107281522734jC",
        "contract_status": "展期结清",
        "expect_repay_time": 1757563839,
        "extend_cnt": 1,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 1200,
        "is_force": "1",
        "loan_time": 1757045439,
        "order_create_time": 1757045248,
        "order_flag": "正常订单",
        "order_id": "892226",
        "order_type": "首贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 840,
        "repay_cnt": 1,
        "repay_time": 1757575581,
        "root_borrow_id": "BOR20250905110727kXE52d",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 7500,
        "borrow_count": 6,
        "borrow_id": "BOR202512041818483EC7LS",
        "contract_id": "C13202512041818481522HRTEa",
        "contract_status": "还款中",
        "expect_repay_time": 1765365533,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 4500,
        "is_force": "0",
        "loan_time": 1764847133,
        "order_create_time": 1764847129,
        "order_flag": "正常订单",
        "order_id": "1373547",
        "order_type": "复贷",
        "overdue_days": None,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 0,
        "repay_cnt": 0,
        "repay_time": None,
        "root_borrow_id": "BOR202512041818483EC7LS",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 12200,
        "borrow_count": 8,
        "borrow_id": "BOR2025120816002343yzq3",
        "contract_id": "C13202512081600231522nD8I8",
        "contract_status": "还款中",
        "expect_repay_time": 1765702829,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 7320,
        "is_force": "0",
        "loan_time": 1765184429,
        "order_create_time": 1765184424,
        "order_flag": "正常订单",
        "order_id": "1399971",
        "order_type": "复贷",
        "overdue_days": None,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 0,
        "repay_cnt": 0,
        "repay_time": None,
        "root_borrow_id": "BOR2025120816002343yzq3",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 3900,
        "borrow_count": 2,
        "borrow_id": "BOR20251005130544JlCg03",
        "contract_id": "C13202510051305441522kj6c8",
        "contract_status": "一般结清",
        "expect_repay_time": 1760162759,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 2340,
        "is_force": "0",
        "loan_time": 1759644359,
        "order_create_time": 1759644344,
        "order_flag": "正常订单",
        "order_id": "1043912",
        "order_type": "复贷",
        "overdue_days": -4,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 3900,
        "repay_cnt": 1,
        "repay_time": 1759844329,
        "root_borrow_id": "BOR20251005130544JlCg03",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 6500,
        "borrow_count": 5,
        "borrow_id": "BOR202511280839521ls9NO",
        "contract_id": "C132025112808395315229FklB",
        "contract_status": "一般结清",
        "expect_repay_time": 1764812397,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 3900,
        "is_force": "0",
        "loan_time": 1764293997,
        "order_create_time": 1764293993,
        "order_flag": "正常订单",
        "order_id": "1331493",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 6500,
        "repay_cnt": 1,
        "repay_time": 1764805102,
        "root_borrow_id": "BOR202511280839521ls9NO",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 4500,
        "borrow_count": 5,
        "borrow_id": "BOR2025100715270955g57F",
        "contract_id": "C132025100715270915220sSy2",
        "contract_status": "一般结清",
        "expect_repay_time": 1760344036,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 2700,
        "is_force": "0",
        "loan_time": 1759825636,
        "order_create_time": 1759825630,
        "order_flag": "正常订单",
        "order_id": "1054620",
        "order_type": "复贷",
        "overdue_days": -6,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 4500,
        "repay_cnt": 1,
        "repay_time": 1759844427,
        "root_borrow_id": "BOR2025100715270955g57F",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 4800,
        "borrow_count": 3,
        "borrow_id": "BOR20250924163520jBz6v7",
        "contract_id": "C13202509241635201522i1yxO",
        "contract_status": "一般结清",
        "expect_repay_time": 1759225997,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 2880,
        "is_force": "1",
        "loan_time": 1758707597,
        "order_create_time": 1758706521,
        "order_flag": "自动悔贷",
        "order_id": "991623",
        "order_type": "复贷",
        "overdue_days": -6,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 2880,
        "repay_cnt": 1,
        "repay_time": 1758708361,
        "root_borrow_id": "BOR20250924163520jBz6v7",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 3300,
        "borrow_count": 3,
        "borrow_id": "BOR20251107180612YKqQvt",
        "contract_id": "C13202511071806121522C30x9",
        "contract_status": "一般结清",
        "expect_repay_time": 1763031979,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 1980,
        "is_force": "0",
        "loan_time": 1762513579,
        "order_create_time": 1762513573,
        "order_flag": "正常订单",
        "order_id": "1216510",
        "order_type": "复贷",
        "overdue_days": -1,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 3300,
        "repay_cnt": 1,
        "repay_time": 1762937822,
        "root_borrow_id": "BOR20251107180612YKqQvt",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 3500,
        "borrow_count": 2,
        "borrow_id": "BOR20250918014534C58kNS",
        "contract_id": "C13202509180145341522P6yhL",
        "contract_status": "一般结清",
        "expect_repay_time": 1758667507,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 2100,
        "is_force": "1",
        "loan_time": 1758149107,
        "order_create_time": 1758134735,
        "order_flag": "正常订单",
        "order_id": "959009",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 3500,
        "repay_cnt": 1,
        "repay_time": 1758681919,
        "root_borrow_id": "BOR20250918014534C58kNS",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 2000,
        "borrow_count": 1,
        "borrow_id": "BOR20250911142620OPqVZi",
        "contract_id": "C13202509111426211522O96aE",
        "contract_status": "一般结清",
        "expect_repay_time": 1758082239,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 1200,
        "is_force": "1",
        "loan_time": 1757563839,
        "order_create_time": 1757575581,
        "order_flag": "正常订单",
        "order_id": "924614",
        "order_type": "首贷展期",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 2000,
        "repay_cnt": 1,
        "repay_time": 1758049321,
        "root_borrow_id": "BOR20250905110727kXE52d",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 8600,
        "borrow_count": 6,
        "borrow_id": "BOR20251202094421J9lD1m",
        "contract_id": "C13202512020944211522KNfo3",
        "contract_status": "一般结清",
        "expect_repay_time": 1765161866,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 5160,
        "is_force": "0",
        "loan_time": 1764643466,
        "order_create_time": 1764643462,
        "order_flag": "正常订单",
        "order_id": "1356624",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 8600,
        "repay_cnt": 1,
        "repay_time": 1765163096,
        "root_borrow_id": "BOR20251202094421J9lD1m",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 9900,
        "borrow_count": 5,
        "borrow_id": "BOR20251202214605802he7",
        "contract_id": "C132025120221460515228QwwH",
        "contract_status": "一般结清",
        "expect_repay_time": 1765205171,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 5940,
        "is_force": "0",
        "loan_time": 1764686771,
        "order_create_time": 1764686766,
        "order_flag": "正常订单",
        "order_id": "1361413",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 9900,
        "repay_cnt": 1,
        "repay_time": 1765163212,
        "root_borrow_id": "BOR20251202214605802he7",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 3300,
        "borrow_count": 2,
        "borrow_id": "BOR20251024131033sa9ad9",
        "contract_id": "C13202510241310331522k6fq8",
        "contract_status": "一般结清",
        "expect_repay_time": 1761804640,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 1980,
        "is_force": "0",
        "loan_time": 1761286240,
        "order_create_time": 1761286234,
        "order_flag": "正常订单",
        "order_id": "1141912",
        "order_type": "复贷",
        "overdue_days": -5,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 3300,
        "repay_cnt": 1,
        "repay_time": 1761377519,
        "root_borrow_id": "BOR20251024131033sa9ad9",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 6000,
        "borrow_count": 4,
        "borrow_id": "BOR20251104171738bsme8B",
        "contract_id": "C1320251104171738152271QzR",
        "contract_status": "逾期结清",
        "expect_repay_time": 1762769866,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 3600,
        "is_force": "0",
        "loan_time": 1762251466,
        "order_create_time": 1762251458,
        "order_flag": "正常订单",
        "order_id": "1201075",
        "order_type": "复贷",
        "overdue_days": 1,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 6480,
        "repay_cnt": 1,
        "repay_time": 1762855141,
        "root_borrow_id": "BOR20251104171738bsme8B",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 3700,
        "borrow_count": 4,
        "borrow_id": "BOR202510010345039A2fuq",
        "contract_id": "C13202510010345031522j22V8",
        "contract_status": "一般结清",
        "expect_repay_time": 1759783511,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 2220,
        "is_force": "0",
        "loan_time": 1759265111,
        "order_create_time": 1759265103,
        "order_flag": "正常订单",
        "order_id": "1022891",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 3700,
        "repay_cnt": 1,
        "repay_time": 1759811101,
        "root_borrow_id": "BOR202510010345039A2fuq",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 3500,
        "borrow_count": 1,
        "borrow_id": "BOR202509051107278I7jTZ",
        "contract_id": "C132025090511072715227p9Eu",
        "contract_status": "展期结清",
        "expect_repay_time": 1757563783,
        "extend_cnt": 2,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 2100,
        "is_force": "1",
        "loan_time": 1757045383,
        "order_create_time": 1757045247,
        "order_flag": "正常订单",
        "order_id": "892225",
        "order_type": "首贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 1470,
        "repay_cnt": 1,
        "repay_time": 1757570341,
        "root_borrow_id": "BOR202509051107278I7jTZ",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 7100,
        "borrow_count": 3,
        "borrow_id": "BOR20251105160916689a5d",
        "contract_id": "C13202511051609161522eh3ic",
        "contract_status": "一般结清",
        "expect_repay_time": 1762852163,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 4260,
        "is_force": "0",
        "loan_time": 1762333763,
        "order_create_time": 1762333757,
        "order_flag": "正常订单",
        "order_id": "1205936",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 7100,
        "repay_cnt": 1,
        "repay_time": 1762846924,
        "root_borrow_id": "BOR20251105160916689a5d",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 700,
        "borrow_count": 1,
        "borrow_id": "BOR202510051855464V9Kg0",
        "contract_id": "C13202510051855471522yopiH",
        "contract_status": "一般结清",
        "expect_repay_time": 1760183905,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 420,
        "is_force": "0",
        "loan_time": 1759665505,
        "order_create_time": 1759665347,
        "order_flag": "正常订单",
        "order_id": "1045788",
        "order_type": "复贷",
        "overdue_days": -4,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 700,
        "repay_cnt": 1,
        "repay_time": 1759844330,
        "root_borrow_id": "BOR202510051855464V9Kg0",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 10200,
        "borrow_count": 6,
        "borrow_id": "BOR20251126200548qF2Wm6",
        "contract_id": "C132025112620054815228K5Ww",
        "contract_status": "一般结清",
        "expect_repay_time": 1764680759,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 6120,
        "is_force": "0",
        "loan_time": 1764162359,
        "order_create_time": 1764162348,
        "order_flag": "正常订单",
        "order_id": "1323992",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 10200,
        "repay_cnt": 1,
        "repay_time": 1764643897,
        "root_borrow_id": "BOR20251126200548qF2Wm6",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 11200,
        "borrow_count": 7,
        "borrow_id": "BOR20251202214629wpsp34",
        "contract_id": "C132025120221463015226GE9T",
        "contract_status": "一般结清",
        "expect_repay_time": 1765205194,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 6720,
        "is_force": "0",
        "loan_time": 1764686794,
        "order_create_time": 1764686790,
        "order_flag": "正常订单",
        "order_id": "1361415",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 11200,
        "repay_cnt": 1,
        "repay_time": 1765162135,
        "root_borrow_id": "BOR20251202214629wpsp34",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 9200,
        "borrow_count": 5,
        "borrow_id": "BOR202511061741415ZB4b1",
        "contract_id": "C1320251106174141152247YzY",
        "contract_status": "一般结清",
        "expect_repay_time": 1762944108,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 5520,
        "is_force": "0",
        "loan_time": 1762425708,
        "order_create_time": 1762425701,
        "order_flag": "正常订单",
        "order_id": "1211553",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 9200,
        "repay_cnt": 1,
        "repay_time": 1762915742,
        "root_borrow_id": "BOR202511061741415ZB4b1",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 5000,
        "borrow_count": 3,
        "borrow_id": "BOR20251026075639pJYQ3U",
        "contract_id": "C1320251026075639152278zl2",
        "contract_status": "一般结清",
        "expect_repay_time": 1761958606,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 3000,
        "is_force": "0",
        "loan_time": 1761440206,
        "order_create_time": 1761440199,
        "order_flag": "正常订单",
        "order_id": "1151817",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 5000,
        "repay_cnt": 1,
        "repay_time": 1761948890,
        "root_borrow_id": "BOR20251026075639pJYQ3U",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 5000,
        "borrow_count": 4,
        "borrow_id": "BOR20250927142524J7DEU8",
        "contract_id": "C1320250927142524152223G6B",
        "contract_status": "展期结清",
        "expect_repay_time": 1759476332,
        "extend_cnt": 1,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 3000,
        "is_force": "0",
        "loan_time": 1758957932,
        "order_create_time": 1758957924,
        "order_flag": "正常订单",
        "order_id": "1005532",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 2100,
        "repay_cnt": 1,
        "repay_time": 1759477262,
        "root_borrow_id": "BOR20250927142524J7DEU8",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 5000,
        "borrow_count": 4,
        "borrow_id": "BOR20251003144101dTAfw3",
        "contract_id": "C13202510031441021522O5R26",
        "contract_status": "一般结清",
        "expect_repay_time": 1759994732,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 3000,
        "is_force": "0",
        "loan_time": 1759476332,
        "order_create_time": 1759477263,
        "order_flag": "展期后补齐",
        "order_id": "1034722",
        "order_type": "复贷展期",
        "overdue_days": -6,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 2900,
        "repay_cnt": 1,
        "repay_time": 1759502700,
        "root_borrow_id": "BOR20250927142524J7DEU8",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 500,
        "borrow_count": 2,
        "borrow_id": "BOR20250923192817Qq55Iv",
        "contract_id": "C13202509231928171522s2O3r",
        "contract_status": "一般结清",
        "expect_repay_time": 1759148904,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 300,
        "is_force": "1",
        "loan_time": 1758630504,
        "order_create_time": 1758630498,
        "order_flag": "正常订单",
        "order_id": "987530",
        "order_type": "复贷",
        "overdue_days": -3,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 500,
        "repay_cnt": 1,
        "repay_time": 1758850969,
        "root_borrow_id": "BOR20250923192817Qq55Iv",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 3500,
        "borrow_count": 1,
        "borrow_id": "BOR202509111259019ts6fI",
        "contract_id": "C13202509111259011522uvMVc",
        "contract_status": "展期结清",
        "expect_repay_time": 1758082183,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 2100,
        "is_force": "1",
        "loan_time": 1757563783,
        "order_create_time": 1757570342,
        "order_flag": "正常订单",
        "order_id": "924098",
        "order_type": "首贷展期",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 1470,
        "repay_cnt": 1,
        "repay_time": 1758095701,
        "root_borrow_id": "BOR202509051107278I7jTZ",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 6700,
        "borrow_count": 3,
        "borrow_id": "BOR20251024123141QOJK1U",
        "contract_id": "C13202510241231411522GHG5O",
        "contract_status": "一般结清",
        "expect_repay_time": 1761802309,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 4020,
        "is_force": "0",
        "loan_time": 1761283909,
        "order_create_time": 1761283902,
        "order_flag": "正常订单",
        "order_id": "1141676",
        "order_type": "复贷",
        "overdue_days": -2,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 6700,
        "repay_cnt": 1,
        "repay_time": 1761628742,
        "root_borrow_id": "BOR20251024123141QOJK1U",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 3800,
        "borrow_count": 3,
        "borrow_id": "BOR20251105071944vC55Y8",
        "contract_id": "C13202511050719441522PV9jd",
        "contract_status": "一般结清",
        "expect_repay_time": 1762820390,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 2280,
        "is_force": "0",
        "loan_time": 1762301990,
        "order_create_time": 1762301984,
        "order_flag": "正常订单",
        "order_id": "1202934",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 3800,
        "repay_cnt": 1,
        "repay_time": 1762846981,
        "root_borrow_id": "BOR20251105071944vC55Y8",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 2700,
        "borrow_count": 3,
        "borrow_id": "BOR20250923192815ddDBo5",
        "contract_id": "C13202509231928151522YUUbl",
        "contract_status": "一般结清",
        "expect_repay_time": 1759148908,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 1620,
        "is_force": "1",
        "loan_time": 1758630508,
        "order_create_time": 1758630495,
        "order_flag": "正常订单",
        "order_id": "987529",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 2700,
        "repay_cnt": 1,
        "repay_time": 1759084138,
        "root_borrow_id": "BOR20250923192815ddDBo5",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 7000,
        "borrow_count": 5,
        "borrow_id": "BOR202511262005519s8J68",
        "contract_id": "C13202511262005511522U62Kg",
        "contract_status": "一般结清",
        "expect_repay_time": 1764680790,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 4200,
        "is_force": "0",
        "loan_time": 1764162390,
        "order_create_time": 1764162351,
        "order_flag": "正常订单",
        "order_id": "1323993",
        "order_type": "复贷",
        "overdue_days": -1,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 7000,
        "repay_cnt": 1,
        "repay_time": 1764564798,
        "root_borrow_id": "BOR202511262005519s8J68",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 800,
        "borrow_count": 1,
        "borrow_id": "BOR20251030164619xWgtVf",
        "contract_id": "C13202510301646191522y08j4",
        "contract_status": "一般结清",
        "expect_repay_time": 1762335986,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 480,
        "is_force": "0",
        "loan_time": 1761817586,
        "order_create_time": 1761817579,
        "order_flag": "正常订单",
        "order_id": "1176023",
        "order_type": "复贷",
        "overdue_days": -6,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 800,
        "repay_cnt": 1,
        "repay_time": 1761822184,
        "root_borrow_id": "BOR20251030164619xWgtVf",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 500,
        "borrow_count": 1,
        "borrow_id": "BOR20250917185638rAE144",
        "contract_id": "C13202509171856381522kSj4T",
        "contract_status": "一般结清",
        "expect_repay_time": 1758628617,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 300,
        "is_force": "1",
        "loan_time": 1758110217,
        "order_create_time": 1758110199,
        "order_flag": "正常订单",
        "order_id": "957993",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 500,
        "repay_cnt": 1,
        "repay_time": 1758601384,
        "root_borrow_id": "BOR20250917185638rAE144",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 6000,
        "borrow_count": 5,
        "borrow_id": "BOR20251004012443UA65mF",
        "contract_id": "C13202510040124431522JlNS1",
        "contract_status": "一般结清",
        "expect_repay_time": 1760034290,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 3600,
        "is_force": "0",
        "loan_time": 1759515890,
        "order_create_time": 1759515883,
        "order_flag": "正常订单",
        "order_id": "1036878",
        "order_type": "复贷",
        "overdue_days": -3,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 6000,
        "repay_cnt": 1,
        "repay_time": 1759844426,
        "root_borrow_id": "BOR20251004012443UA65mF",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 8200,
        "borrow_count": 4,
        "borrow_id": "BOR20251030104548kidcLZ",
        "contract_id": "C13202510301045481522UddXp",
        "contract_status": "一般结清",
        "expect_repay_time": 1762314445,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 4920,
        "is_force": "0",
        "loan_time": 1761796045,
        "order_create_time": 1761795948,
        "order_flag": "正常订单",
        "order_id": "1173956",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 8200,
        "repay_cnt": 1,
        "repay_time": 1762297905,
        "root_borrow_id": "BOR20251030104548kidcLZ",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 2000,
        "borrow_count": 3,
        "borrow_id": "BOR2025092714014963upSk",
        "contract_id": "C13202509271401491522uiifi",
        "contract_status": "一般结清",
        "expect_repay_time": 1759474918,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 1200,
        "is_force": "0",
        "loan_time": 1758956518,
        "order_create_time": 1758956510,
        "order_flag": "正常订单",
        "order_id": "1005410",
        "order_type": "复贷",
        "overdue_days": -4,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 2000,
        "repay_cnt": 1,
        "repay_time": 1759084191,
        "root_borrow_id": "BOR2025092714014963upSk",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 7000,
        "borrow_count": 5,
        "borrow_id": "BOR202512021644145518es",
        "contract_id": "C13202512021644141522yPx5X",
        "contract_status": "一般结清",
        "expect_repay_time": 1765187453,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 4200,
        "is_force": "0",
        "loan_time": 1764669053,
        "order_create_time": 1764668655,
        "order_flag": "正常订单",
        "order_id": "1359817",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 7000,
        "repay_cnt": 1,
        "repay_time": 1765163213,
        "root_borrow_id": "BOR202512021644145518es",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 2700,
        "borrow_count": 2,
        "borrow_id": "BOR20250917185638etUJ78",
        "contract_id": "C132025091718563815229Arh3",
        "contract_status": "一般结清",
        "expect_repay_time": 1758633959,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 1620,
        "is_force": "1",
        "loan_time": 1758115559,
        "order_create_time": 1758110198,
        "order_flag": "正常订单",
        "order_id": "957994",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 2700,
        "repay_cnt": 1,
        "repay_time": 1758601383,
        "root_borrow_id": "BOR20250917185638etUJ78",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 7000,
        "borrow_count": 5,
        "borrow_id": "BOR20251126200554CJyA03",
        "contract_id": "C13202511262005541522iPLaB",
        "contract_status": "一般结清",
        "expect_repay_time": 1764680821,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 4200,
        "is_force": "0",
        "loan_time": 1764162421,
        "order_create_time": 1764162355,
        "order_flag": "正常订单",
        "order_id": "1323994",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 7000,
        "repay_cnt": 1,
        "repay_time": 1764635366,
        "root_borrow_id": "BOR20251126200554CJyA03",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 6000,
        "borrow_count": 4,
        "borrow_id": "BOR20251106174144OpajgU",
        "contract_id": "C13202511061741441522T6bm4",
        "contract_status": "一般结清",
        "expect_repay_time": 1762944111,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 3600,
        "is_force": "0",
        "loan_time": 1762425711,
        "order_create_time": 1762425704,
        "order_flag": "正常订单",
        "order_id": "1211554",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 6000,
        "repay_cnt": 1,
        "repay_time": 1762914781,
        "root_borrow_id": "BOR20251106174144OpajgU",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 3000,
        "borrow_count": 4,
        "borrow_id": "BOR20251007112701ZElA0J",
        "contract_id": "C13202510071127021522E5n34",
        "contract_status": "一般结清",
        "expect_repay_time": 1760321517,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 1800,
        "is_force": "0",
        "loan_time": 1759803117,
        "order_create_time": 1759811222,
        "order_flag": "展期后补齐",
        "order_id": "1053254",
        "order_type": "复贷展期",
        "overdue_days": -6,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 1760,
        "repay_cnt": 1,
        "repay_time": 1759844700,
        "root_borrow_id": "BOR20251001025857g5v0zp",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 4800,
        "borrow_count": 3,
        "borrow_id": "BOR202510260843498GLaQm",
        "contract_id": "C132025102608434915226NY8K",
        "contract_status": "一般结清",
        "expect_repay_time": 1761961436,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 2880,
        "is_force": "0",
        "loan_time": 1761443036,
        "order_create_time": 1761443030,
        "order_flag": "正常订单",
        "order_id": "1152032",
        "order_type": "复贷",
        "overdue_days": -2,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 4800,
        "repay_cnt": 1,
        "repay_time": 1761822301,
        "root_borrow_id": "BOR202510260843498GLaQm",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "happy cashing_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 3000,
        "borrow_count": 4,
        "borrow_id": "BOR20251001025857g5v0zp",
        "contract_id": "C132025100102585715220ms7S",
        "contract_status": "展期结清",
        "expect_repay_time": 1759803117,
        "extend_cnt": 1,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 1800,
        "is_force": "0",
        "loan_time": 1759284717,
        "order_create_time": 1759262337,
        "order_flag": "正常订单",
        "order_id": "1022854",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 1260,
        "repay_cnt": 1,
        "repay_time": 1759811222,
        "root_borrow_id": "BOR20251001025857g5v0zp",
        "user_id": "c9520a45f9bb4d59b3865919b6538392",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 5400,
        "borrow_count": 4,
        "borrow_id": "BOR20251126133451pPzcyt",
        "contract_id": "C13202511261334511522fN244",
        "contract_status": "一般结清",
        "expect_repay_time": 1764657296,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 3240,
        "is_force": "0",
        "loan_time": 1764138896,
        "order_create_time": 1764138892,
        "order_flag": "正常订单",
        "order_id": "1321645",
        "order_type": "复贷",
        "overdue_days": -1,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 5400,
        "repay_cnt": 1,
        "repay_time": 1764548299,
        "root_borrow_id": "BOR20251126133451pPzcyt",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 8200,
        "borrow_count": 6,
        "borrow_id": "BOR20251202093435Tgn7J7",
        "contract_id": "C13202512020934351522w173x",
        "contract_status": "一般结清",
        "expect_repay_time": 1765161280,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 4920,
        "is_force": "0",
        "loan_time": 1764642880,
        "order_create_time": 1764642876,
        "order_flag": "正常订单",
        "order_id": "1356548",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 8200,
        "repay_cnt": 1,
        "repay_time": 1765132939,
        "root_borrow_id": "BOR20251202093435Tgn7J7",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 4300,
        "borrow_count": 1,
        "borrow_id": "BOR202510230007096uAQV5",
        "contract_id": "C13202510230007091522F5Si1",
        "contract_status": "一般结清",
        "expect_repay_time": 1761671237,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 2580,
        "is_force": "0",
        "loan_time": 1761152837,
        "order_create_time": 1761152829,
        "order_flag": "正常订单",
        "order_id": "1133890",
        "order_type": "首贷",
        "overdue_days": -1,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 4300,
        "repay_cnt": 1,
        "repay_time": 1761628264,
        "root_borrow_id": "BOR202510230007096uAQV5",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 500,
        "borrow_count": 1,
        "borrow_id": "BOR20251005185549VSngih",
        "contract_id": "C132025100518554915223tx92",
        "contract_status": "一般结清",
        "expect_repay_time": 1760183907,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 300,
        "is_force": "0",
        "loan_time": 1759665507,
        "order_create_time": 1759665350,
        "order_flag": "正常订单",
        "order_id": "1045789",
        "order_type": "复贷",
        "overdue_days": -4,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 500,
        "repay_cnt": 1,
        "repay_time": 1759844331,
        "root_borrow_id": "BOR20251005185549VSngih",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 5600,
        "borrow_count": 2,
        "borrow_id": "BOR20251030085213iASg3h",
        "contract_id": "C13202510300852131522X9zsu",
        "contract_status": "一般结清",
        "expect_repay_time": 1762307540,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 3360,
        "is_force": "0",
        "loan_time": 1761789140,
        "order_create_time": 1761789134,
        "order_flag": "正常订单",
        "order_id": "1173316",
        "order_type": "复贷",
        "overdue_days": 0,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 5600,
        "repay_cnt": 1,
        "repay_time": 1762297962,
        "root_borrow_id": "BOR20251030085213iASg3h",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 2300,
        "borrow_count": 2,
        "borrow_id": "BOR20251101022234WY1Ze9",
        "contract_id": "C13202511010222351522cz7Iq",
        "contract_status": "一般结清",
        "expect_repay_time": 1762456962,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 1380,
        "is_force": "0",
        "loan_time": 1761938562,
        "order_create_time": 1761938555,
        "order_flag": "正常订单",
        "order_id": "1182942",
        "order_type": "复贷",
        "overdue_days": -6,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 2300,
        "repay_cnt": 1,
        "repay_time": 1761969181,
        "root_borrow_id": "BOR20251101022234WY1Ze9",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
    {
        "app_name": "Wing สินเชื่อสะดวกสบาย",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 3500,
        "borrow_count": 2,
        "borrow_id": "BOR20251024131030sYDiEL",
        "contract_id": "C13202510241310301522NwEVs",
        "contract_status": "一般结清",
        "expect_repay_time": 1761804637,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 2100,
        "is_force": "0",
        "loan_time": 1761286237,
        "order_create_time": 1761286231,
        "order_flag": "正常订单",
        "order_id": "1141911",
        "order_type": "复贷",
        "overdue_days": -5,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 3500,
        "repay_cnt": 1,
        "repay_time": 1761377518,
        "root_borrow_id": "BOR20251024131030sYDiEL",
        "user_id": "4d31360ad3214011ae7c4967a7e17fef",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 2000,
        "borrow_count": 2,
        "borrow_id": "BOR20251102001011RfI7si",
        "contract_id": "C13202511020010111522i6s1K",
        "contract_status": "一般结清",
        "expect_repay_time": 1762535417,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 1200,
        "is_force": "0",
        "loan_time": 1762017017,
        "order_create_time": 1762017011,
        "order_flag": "正常订单",
        "order_id": "1187869",
        "order_type": "复贷",
        "overdue_days": -1,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 2000,
        "repay_cnt": 1,
        "repay_time": 1762494964,
        "root_borrow_id": "BOR20251102001011RfI7si",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
    {
        "app_name": "APM money_ios",
        "bank": "KTB",
        "bank_card_num": "00000000001892031273191284738",
        "borrow_amt": 8700,
        "borrow_count": 4,
        "borrow_id": "BOR20251126133448pV2s8y",
        "contract_id": "C13202511261334481522090BE",
        "contract_status": "一般结清",
        "expect_repay_time": 1764657293,
        "extend_cnt": 0,
        "id_number": "00000000001892031979169771522",
        "in_hand_amt": 5220,
        "is_force": "0",
        "loan_time": 1764138893,
        "order_create_time": 1764138888,
        "order_flag": "正常订单",
        "order_id": "1321644",
        "order_type": "复贷",
        "overdue_days": -1,
        "phone": "00000000001865571048323723445",
        "platform": "ios",
        "product_id": "000002",
        "repay_amt": 8700,
        "repay_cnt": 1,
        "repay_time": 1764537471,
        "root_borrow_id": "BOR20251126133448pV2s8y",
        "user_id": "aecefd177b4844ed929d78168698cc2f",
    },
]

In [ ]:
EVENT_VOCAB = {
    "[PAD]": 0,
    "风控通过": 1,
    "风控拒绝": 2,
    "人工审核": 3,
    "展期还款": 4,
    "放款": 5,
    "悔贷": 6,
    "D-2及之前结清": 7,
    "D-1结清": 8,
    "D0结清": 9,
    "逾期结清": 10,
    "逾期D1": 11,
    "逾期D3": 12,
    "逾期D7": 13,
}

In [172]:
class Event:
    def __init__(
        self,
        event_name,
        event_time,
        app_name,
        borrow_amt,
        onway_amt_delta,
        onway_cnt_delta,
        comment = ''
    ):
        self.event_name = event_name
        self.event_time = event_time
        self.app_name = app_name
        self.borrow_amt = borrow_amt
        self.onway_amt_delta = onway_amt_delta
        self.onway_cnt_delta = onway_cnt_delta
        self.comment = comment
        self.time_delta = None
        self.is_same_pkg = None
        self.onway_amt = None
        self.onway_cnt = None

    @property
    def event_id(self):
        return EVENT_VOCAB[self.event_name]

    def __str__(self):
        return f"{self.event_name}\t{datetime.datetime.fromtimestamp(self.event_time, tz=thailand_tz)}\t{self.app_name}\t{self.borrow_amt}\t{self.onway_amt}\t{self.onway_cnt}\t{self.time_delta}\t{self.comment}"

In [173]:
verify_data = VerifyBehavior.load_from_dict(veirfy_data_lst[0])
post_loan_data = PostLoanOrder.load_from_dict(post_loan_data_lst[0])

In [174]:
def cal_overdue_expect_repay_dt(expect_repay_dt, overdue_days):
    dt = datetime.timedelta(days=overdue_days)
    target_dt = (expect_repay_dt + dt).replace(
        hour=0, minute=0, second=0, microsecond=0
    )
    return int(target_dt.timestamp())

In [175]:
from typing import List


def handle_verify_event(verify_data) -> List[Event]:
    if verify_data.risk_level == "ACCEPT":
        event_name = "风控通过"
    elif verify_data.risk_level == "REJECT":
        event_name = "风控拒绝"
    else:
        event_name = "人工审核"
    event = Event(
        event_name=event_name,
        event_time=verify_data.verify_time,
        app_name=verify_data.app_name,
        borrow_amt=0,
        onway_amt_delta=0,
        onway_cnt_delta=0,
        comment=verify_data.sn
    )
    return [event]

In [176]:
def handle_post_loan_event(post_loan_data):
    event_lst = []

    # 根订单 -> 放款事件
    if post_loan_data.is_root_order == 1:
        event_lst.append(Event(event_name='放款', event_time=post_loan_data.loan_time, app_name=post_loan_data.app_name, borrow_amt=post_loan_data.borrow_amt, onway_amt_delta=post_loan_data.borrow_amt, onway_cnt_delta=1, comment=post_loan_data.borrow_id))

    # 悔贷
    if post_loan_data.is_regret == 1:
        event_lst.append(Event(event_name='悔贷', event_time=post_loan_data.repay_time, app_name=post_loan_data.app_name, borrow_amt=post_loan_data.borrow_amt, onway_amt_delta=-post_loan_data.borrow_amt, onway_cnt_delta=-1, comment=post_loan_data.borrow_id))

    # 展期结清
    if post_loan_data.is_extend == 1:
        event_lst.append(Event(event_name='展期还款', event_time=post_loan_data.repay_time, app_name=post_loan_data.app_name, borrow_amt=post_loan_data.borrow_amt, onway_amt_delta=0, onway_cnt_delta=0, comment=post_loan_data.borrow_id))
    
    # 非展期结清订单 -> 结清事件
    if post_loan_data.is_extend == 0 and post_loan_data.is_repay == 1 and post_loan_data.is_regret == 0:
        if post_loan_data.overdue_days <= -2:
            event_lst.append(Event(event_name='D-2及之前结清', event_time=post_loan_data.repay_time, app_name=post_loan_data.app_name, borrow_amt=post_loan_data.borrow_amt, onway_amt_delta=-post_loan_data.borrow_amt, onway_cnt_delta=-1, comment=post_loan_data.borrow_id))
        elif post_loan_data.overdue_days <= -1:
            event_lst.append(Event(event_name='D-1结清', event_time=post_loan_data.repay_time, app_name=post_loan_data.app_name, borrow_amt=post_loan_data.borrow_amt, onway_amt_delta=-post_loan_data.borrow_amt, onway_cnt_delta=-1, comment=post_loan_data.borrow_id))
        elif post_loan_data.overdue_days == 0:
            event_lst.append(Event(event_name='D0结清', event_time=post_loan_data.repay_time, app_name=post_loan_data.app_name, borrow_amt=post_loan_data.borrow_amt, onway_amt_delta=-post_loan_data.borrow_amt, onway_cnt_delta=-1, comment=post_loan_data.borrow_id))
        else:
            event_lst.append(Event(event_name='逾期结清', event_time=post_loan_data.repay_time, app_name=post_loan_data.app_name, borrow_amt=post_loan_data.borrow_amt, onway_amt_delta=-post_loan_data.borrow_amt, onway_cnt_delta=-1, comment=post_loan_data.borrow_id))

    # 逾期事件
    if post_loan_data.is_extend == 0 and post_loan_data.is_repay == 1 and post_loan_data.is_regret == 0:
        if post_loan_data.overdue_days >= 7:
            event_lst.append(Event(event_name='逾期D7', event_time=cal_overdue_expect_repay_dt(post_loan_data.expect_repay_dt, 7), app_name=post_loan_data.app_name, borrow_amt=post_loan_data.borrow_amt, onway_amt_delta=0, onway_cnt_delta=0, comment=post_loan_data.borrow_id))

        if post_loan_data.overdue_days >= 3:
            event_lst.append(Event(event_name='逾期D3', event_time=cal_overdue_expect_repay_dt(post_loan_data.expect_repay_dt, 3), app_name=post_loan_data.app_name, borrow_amt=post_loan_data.borrow_amt, onway_amt_delta=0, onway_cnt_delta=0, comment=post_loan_data.borrow_id))
        
        if post_loan_data.overdue_days >= 1:
            event_lst.append(Event(event_name='逾期D1', event_time=cal_overdue_expect_repay_dt(post_loan_data.expect_repay_dt, 1), app_name=post_loan_data.app_name, borrow_amt=post_loan_data.borrow_amt, onway_amt_delta=0, onway_cnt_delta=0, comment=post_loan_data.borrow_id))

    return event_lst

In [177]:
PostLoanOrderLst = [PostLoanOrder.load_from_dict(pl) for pl in post_loan_data_lst]
VerifyBehaviorLst = [VerifyBehavior.load_from_dict(v) for v in veirfy_data_lst]

In [178]:
event_lst = []
for pl in PostLoanOrderLst:
    event_lst.extend(handle_post_loan_event(pl))
for v in VerifyBehaviorLst:
    event_lst.extend(handle_verify_event(v))

In [179]:
event_lst.sort(key=lambda x: x.event_time)

onway_amt = 0
onway_cnt = 0

for idx in range(len(event_lst)):
    event = event_lst[idx]

    if idx == 0:
        event.time_delta = 0
    else:
        event.time_delta = event.event_time - event_lst[idx - 1].event_time
    
    event.onway_amt = onway_amt
    event.onway_cnt = onway_cnt

    onway_amt += event.onway_amt_delta
    onway_cnt += event.onway_cnt_delta

    print(event)

风控拒绝	2025-02-19 09:07:34+07:00	radonmoney_ios	0	0	0	0	EDD202502190906118l05toei
风控拒绝	2025-02-19 11:02:26+07:00	finnix_ios	0	0	0	6892	SVD20250219110128lgWBH40K
风控拒绝	2025-03-01 10:28:54+07:00	finnix_ios	0	0	0	861988	SVD20250301102844dk2v478B
风控拒绝	2025-03-01 11:45:59+07:00	zedwallet_ios	0	0	0	4625	EDD202503011145535AKnFhU2
风控拒绝	2025-03-28 07:48:32+07:00	zedwallet_ios	0	0	0	2318553	EDD20250328074825tCsHdvqY
风控拒绝	2025-07-02 11:30:03+07:00	pakincash_ios	0	0	0	8307691	EDD20250702112955htBe1n71
风控拒绝	2025-07-15 08:47:25+07:00	photoPortfolio_ios	0	0	0	1113442	EDD20250715084709DJlCG5j2
风控拒绝	2025-07-17 10:00:36+07:00	radonmoney_ios	0	0	0	177191	EDD20250717100022q85f862o
风控通过	2025-09-05 11:07:24+07:00	easyloan_ios	0	0	0	4324008	SVD20250905110714ynhlt71W
放款	2025-09-05 11:09:43+07:00	happy cashing_ios	3500	0	0	139	BOR202509051107278I7jTZ
放款	2025-09-05 11:10:39+07:00	happy cashing_ios	2000	3500	1	56	BOR20250905110727kXE52d
展期还款	2025-09-11 12:59:01+07:00	happy cashing_ios	3500	5500	2	524902	BOR20250905